&nbsp;

# Heat stress-related ChIP-seq Metadata Validation

- ChIP-Atlas のエンリッチメント解析の結果から、熱ストレスなどに関連しているものを探索するため、ここで調査する
- `Input` sample は、今回はペアのメタデータを作る際は基本的に除いておくが、その際はIDの記載のみしておく
- 主に、`genome binding/occupancy profiling by high throughput sequencing` で取得されたものをこちらに記載する
- `heat shock` vs `ctrl` のペアを念頭に収集するが、後に使用する可能性もあるので、除去したプロセスを明示的に記載する

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display


pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_rows(4)
pl.Config.set_tbl_cols(100)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera"
]

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Strategy 0. これまで収集してきたメタデータに、ChIP-seq が含まれているか確認

- [../00_metadata/](../00_metadata/) にある、主に bulk RNA-seq データを収集してきたメタデータ内に、熱ストレスに関連するChIP-seq のデータがないか確認する


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. GSE140053 (ChIP-seq; Pol2)

- [Stress-induced nuclear condensation of NELF drives transcriptional downregulation](https://doi.org/10.1016/j.molcel.2021.01.016)
- [./12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)から移動
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb) から更に移動
- validate date: 2026-04-28

In [4]:
GSE140053_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE140053/GSE140053_PEP/GSE140053_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_title") != "VCaP: Input") # input sample is excluded (SRX7107233)
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("https://doi.org/10.1016/j.molcel.2021.01.016").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("30 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE140053 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE140053_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE140053_filtered = GSE140053[[s.name for s in GSE140053 if not (s.null_count() == GSE140053.height)]]

GSE140053_combined = GSE140053_PEP.join(
    GSE140053_filtered,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE140053_combined = GSE140053_combined[[s.name for s in GSE140053_combined if not (s.null_count() == GSE140053_combined.height)]]

# pl.Config.set_tbl_rows(9)
# display(GSE140053_combined)

In [5]:
defined_pairs_gse140053_chipseq = [
    ("SRX7107227", "SRX7107225"), # VCaP: Pol2_siCtrl_HS1 vs. VCaP: Pol2_siCtrl_Ctrl1
    ("SRX7107228", "SRX7107226"), # VCaP: Pol2_siCtrl_HS2 vs. VCaP: Pol2_siCtrl_Ctrl2
    ("SRX7107231", "SRX7107229"), # VCaP: Pol2_siUBC9_HS1 vs. VCaP: Pol2_siUBC9_Ctrl1
    ("SRX7107232", "SRX7107230") # VCaP: Pol2_siUBC9_HS2 vs. VCaP: Pol2_siUBC9_Ctrl2
]

paired_metadata_df_gse140053_chipseq = create_paired_metadata(
    combined_df=GSE140053_combined,
    defined_sample_pairs=defined_pairs_gse140053_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metadata_df_gse140053_chipseq is not None:
    paired_metadata_df_gse140053_chipseq.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE140053_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse140053_chipseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,chip_antibody,treatment,condition,genome_build,alternative_to,biosample,sra,sample_accession,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX7107227""","""SRX7107225""","""vcap_pol2_sictrl_hs1 vs vcap_pol2_sictrl_ctrl1""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""multi""","""VCaP: Pol2_siCtrl_HS1 vs VCaP: Pol2_siCtrl_Ctrl1""","""GSM4152129 vs GSM4152127""","""Public on Feb 05 2021""","""Nov 06 2019""","""Feb 06 2021""","""SRA""","""1""","""VCaP""","""Homo sapiens""","""9606""","""Heat shock for ChIP-seq samples was done at 43 degree C incubator for 30min. Silencing for VCaP cells was done by transfecting siRNA for UBE2I (siUBE2I samples) or non-targeting control siRNA (siNON s…","""VCaP cells were grown in DMEM medium supplemented with 10% feta bovine serum and 1% penicillin/streptomycin antibiotics at 37 degree C temperature incubator with 5% CO2.""","""genomic DNA""","""VCaP cells were grown in DMEM medium supplemented with 10% feta bovine serum and 1% penicillin/streptomycin antibiotics at 37 degree C temperature incubator with 5% CO2. Nuclei from approximately 10 m…","""resequencing of sample previously submitted to GEO as GSM1777590 vs resequencing of sample previously submitted to GEO as GSM1777588""","""ChIP-seq reads were aligned to the human genome build hg38 using Bowtie2. Duplicate and discordant reads were removed. Gene annotations and transcript start site (TSS) information for human genes wer…","""GPL18573""","""Barbara,,Hummel""","""Stübeweg 51""","""Freiburg""","""79108""","""Germany""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4152129 vs GSM4152127""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4152nnn/GSM4152129/suppl/GSM4152129_Sample3_Pol2_siCtrl_HS1.filtered.seq_depth_norm.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4152nnn/GSM4152127/suppl/G…","""GSE140053""","""0""","""SRX7107227 vs SRX7107225""","""VCaP""","""anti-Pol2, clone: 8WG16 (Covance, catalog# MMS-126R, lot# E12CF00481)""","""control siRNA transfected""","""Heat-Shock vs Non Heat-Shock""","""hg38""","""GSM1777590 vs GSM1777588""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN13226630 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN13226632""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX7107227 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX7107225""","""SAMN13226630 vs SAMN13226632""","""https://doi.org/10.1016/j.molcel.2021.01.016""","""43 C""","""30 min.""","""37 C (inferred)"""
"""SRX7107228""","""SRX7107226""","""vcap_pol2_sictrl_hs2 vs vcap_pol2_sictrl_ctrl2""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""multi""","""VCaP: Pol2_siCtrl_HS2 vs VCaP: Pol2_siCtrl_Ctrl2""","""GSM4152130 vs GSM4152128""","""Public on Feb 05 2021""","""Nov 06 2019""","""Feb 06 2021""","""SRA""","""1""","""VCaP""","""Homo sapiens""","""9606""","""Heat shock for ChIP-seq samples was done at 43 degree C incubator for 30min. Silencing for VCaP cells was done by transfecting siRNA for UBE2I (siUBE2I samples) or non-

&nbsp;

&nbsp;

&nbsp;

#### GSE259306 (NO DOWNLOAD)

- validation date: 2026-04-28
- [TurboCas: A method for locus-specific labeling of genomic regions and isolating their associated protein interactome](https://doi.org/10.1016/j.molcel.2024.11.007)
- [./12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)でフィルタリングされていると思われるデータをこちらで処理
- このエントリーには、ChIP-seqのデータはなかった

#### GSE69021 (NO DOWNLOAD)

- validate date: 2026-04-28
- [Gene target specificity of the Super Elongation Complex (SEC) family: how HIV-1 Tat employs selected SEC members to activate viral transcription](https://doi.org/10.1093/nar/gkv541)
- AFF4、AFF1に対してRNAiを行っているデータ
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb) で、bulk RNA-seq が含まれていないと判断し、収集していなかったデータ
- このエントリーではなく、同じ論文由来の別のアクセッションデータに ChIP-seq データが含まれている可能性

#### GSE182204 (NO DOWNLOAD)

- validation date: 2025-05-03
- [Acetyl-methyllysine marks chromatin at active transcription start sites](https://doi.org/10.1038/s41586-023-06565-9)
- RNA-seqに加え、TT-TL-seq (transient-transcriptome TimeLapse sequencing), start-TimeLapse sequencing (STL-seq) も含まれている
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb) で 収集しないと判断していたデータ
- このデータには、Drosophila melanogaster のデータも含まれており、こちらのデータで、熱ストレス曝露のサンプルが含まれていた

#### GSE112608 (NO DOWNLOAD) 

- validate date 2026-04-28
- [Targeting Processive Transcription Elongation via SEC Disruption for MYC-Induced Cancer Therapy](https://doi.org/10.1016/j.cell.2018.09.027)
- 論文内に以下のような記述を発見
- `Heat Shock Induction: Heat shock of mammalian cells was performed using ∼70%–80% confluent HCT-116 cells by adding pre-heated (42°C) conditioned media collected from identically growing cells (Mahat et al., 2016b). The heat shock cells were incubated at 42°C for 1 hour. After washing with PBS, the heat shock and non-heat shock HCT-116 cells were fixed with 1% formaldehyde in PBS for downstream ChIP-seq analysis. For heat shock induction of Drosophila S2 cells, the S2 cells were mixed with pre-heated medium to instantly increase the medium temperature from 24°C to 37°C and maintained in a water bath at 37°C for 10 min before fixation for ChIP-seq.`
- 探索したものの、明示的に熱ストレスと記載があるアクセッションがなかったので、今回は収集しない

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. GSE145525 (ChIP-seq; Pol II)

- validation date: 2025-05-06
- [Acute perturbation strategies in interrogating RNA polymerase II elongation factor function in gene expression](https://doi.org/10.1101/gad.346106.120)
- mRNA-seqとChIP-seqのデータが含まれている、RNA polymerase IIに関連したデータ
- [./12_4_metadata_modified_false_no_control.ipynb](./12_4_metadata_modified_false_no_control.ipynb) で、フィルタリングされていたデータを再確認
- BRD4に対してAIDシステム (タンパク質ノックダウンシステム) を行っている + 42℃ 60min.を行っているデータなどが含まれている

In [6]:
GSE145525_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE145525/GSE145525_PEP/GSE145525_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("organism") == "Homo sapiens") &
    (pl.col("sample_library_strategy") == "ChIP-Seq") &

    # Because of antigen class is unclassified in ChIP-atlas, so these entry is removed
    (pl.col("sample_name") != "0hdmso_aid660_brd4aid_dld_rep1") & # SRX7748770
    (pl.col("sample_name") != "2haux_2hdmso_aid660_brd4aid_dld_rep1") & # SRX7748772

    # treatment : Auxin
    (pl.col("sample_name") != "2haux_brd2cst_brd2aid_dld_rep1") & # SRX7748767
    (pl.col("sample_name") != "2haux_brd3bethyl_brd3aid_dld_rep1") & # SRX7748769
    (pl.col("sample_name") != "pol2_0h_h6_brd2aid_hct116_rep1") & # SRX9390303
    (pl.col("sample_name") != "pol2_1h_h6_brd2aid_hct116_rep1") & # SRX9390304
    (pl.col("sample_name") != "pol2_4h_h6_brd2aid_hct116_rep1") & # SRX9390305
    (pl.col("sample_name") != "pol2_24h_h6_brd2aid_hct116_rep1") & # SRX9390306

    # treatment: dBET6
    (pl.col("sample_name") != "2hdbet6_37c1h_pol2cst_brd4aid_dld_rep1") & # SRX7748701
    (pl.col("sample_name") != "2hdbet6_37c1h_pol2cst_wt_dld_rep1") & # SRX7748693
    (pl.col("sample_name") != "2hdbet6_aid660_brd4aid_dld_rep1") &  # SRX7748771

    # genotype: BRD2-AID OsTIR1
    (pl.col("sample_name") != "2hh2o_brd2cst_brd2aid_dld_rep1") & # SRX7748766

    # genotype: BRD3-AID OsTIR1
    (pl.col("sample_name") != "2hh2o_brd3bethyl_brd3aid_dld_rep1") & # SRX7748768

    # Input sample
    (pl.col("sample_name") != "input_2hdmso_37c1h_brd4aid_dld_rep1") & # SRX7748805
    (pl.col("sample_name") != "input_2hh2o_37c1h_wt_dld_rep1")  # SRX7748806   
).sort(
    by="sample_name"
).with_columns(
    pl.lit("https://doi.org/10.1101/gad.346106.120").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)


GSE145525 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE145525_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE145525_filtered = GSE145525[[s.name for s in GSE145525 if not (s.null_count() == GSE145525.height)]]

GSE145525_combined = GSE145525_PEP.join(
    GSE145525_filtered,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE145525_combined = GSE145525_combined[[s.name for s in GSE145525_combined if not (s.null_count() == GSE145525_combined.height)]]

pl.Config.set_tbl_rows(2)
display(GSE145525_combined)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,genotype,chip_antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""2haux_37c1h_aff1bethyl_aff1aid_dld_rep1""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR11111766""","""SRX7748786""","""2hAux_37C1h_AFF1Bethyl_AFF1AID_DLD_rep1""","""GSM4320291""","""Public on Jan 15 2021""","""Feb 19 2020""","""Jan 16 2021""","""SRA""",1,"""DLD-1""","""Homo sapiens""",9606,"""Cells were treated with 500 µM Auxin (3-indole-acetic acid sodium salt, abcam), 250 nM dBET6 (AOBIOUS), or 20 nM KL-1 (Liang et al. 2018). Heat shock treatment at 42 ˚C for 1 h was carried out as desc…","""DLD-1 cells were cultured in DMEM (Corning) supplemented with 10% FBS (Corning), 1X GlutaMAX (Gibco), 100 U/ml Penicillin and 100 µg/ml Streptomycin (Gibco).""","""genomic DNA""","""For ChIP-seq, 20-50 million cells were crosslinked with 1% paraformaldehyde (Electron Microscopy Sciences) in PBS for 10 min at r.t. Chromatin was sonicated with the Covaris E220 for 4 min with 10% du…","""AFF1 ChIP-seq""","""For ChIP-seq, single-end raw reads were trimmed by Trimmomatic 0.33 (Bolger et al., 2014) and aligned to hg19 using Bowtie version 1.1.2, allowing only uniquely mapping reads with up to two mismatches…","""GPL18573""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""",60611,"""USA""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4320291""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4320nnn/GSM4320291/suppl/GSM4320291_2hAux_37C1h_AFF1Bethyl_AFF1AID_DLD_rep1.bw""","""GSE145525""",0,"""SRX7748786""","""Colorectal adenocarcinoma DLD-1""","""AFF1-AID OsTIR1""","""AF4 Bethyl A302-345A""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN14135441""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX7748786""","""https://doi.org/10.1101/gad.346106.120""","""42 C""","""60 min.""","""NextSeq 500 sequencing: GSM4320291: 2hAux_37C1h_AFF1Bethyl_AFF1AID_DLD_rep1 Homo sapiens ChIP-Seq""","""SAMN14135441""","""Acute perturbation strategies in interrogating RNA Polymerase II elongation factor function in gene expression""","""SRA1044572""","""ChIP""","""2021-

In [7]:
defined_pairs_gse145525_chipseq = [
    ("SRX7748788", "SRX7748786"), # 2haux_42c1h_aff1bethyl_aff1aid_dld_rep1 vs. 2haux_37c1h_aff1bethyl_aff1aid_dld_rep1
    ("SRX7748792", "SRX7748790"), # 2haux_42c1h_aff4protec_aff4aid_dld_rep1 vs. 2haux_37c1h_aff4protec_aff4aid_dld_rep1
    ("SRX7748800", "SRX7748798"), # 2haux_42c1h_aid660_aff1aid_dld_rep1 vs. 2haux_37c1h_aid660_aff1aid_dld_rep1
    ("SRX7748804", "SRX7748802"), # 2haux_42c1h_aid660_aff4aid_dld_rep1 vs. 2haux_37c1h_aid660_aff4aid_dld_rep1
    ("SRX7748776", "SRX7748774"), # 2haux_42c1h_aid660_brd2aid_dld_rep1 vs. 2haux_37c1h_aid660_brd2aid_dld_rep1
    ("SRX7748784", "SRX7748782"), # 2haux_42c1h_aid660_brd3aid_dld_rep1 vs. 2haux_37c1h_aid660_brd3aid_dld_rep1
    ("SRX7748780", "SRX7748778"), # 2haux_42c1h_aid660_brd4aid_dld_rep1 vs. 2haux_37c1h_aid660_brd4aid_dld_rep1
    ("SRX7748796", "SRX7748794"), # 2haux_42c1h_aid660_doubleaid_dld_rep1 vs. 2haux_37c1h_aid660_doubleaid_dld_rep1

    # pair3_GSE145525_pol2_hs_42C_60min_2haux_pol2cst_aff1aid_dld.txt 
    # pair3_GSE145525_pol2_ctrl_2haux_aff1aid_dld.txt
    ("SRX7748741", "SRX7748739"), # 2haux_42c1h_pol2cst_aff1aid_dld_rep1 vs. 2haux_37c1h_pol2cst_aff1aid_dld_rep1
    ("SRX7748753", "SRX7748751"), # 2haux_42c1h_pol2cst_aff1aid_dld_rep2 vs. 2haux_37c1h_pol2cst_aff1aid_dld_rep2
    ("SRX7748761", "SRX7748759"), # 2haux_42c1h_pol2cst_aff1aid_dld_rep3 vs. 2haux_37c1h_pol2cst_aff1aid_dld_rep3

    
    # pair4_GSE145525_pol2_hs_42C_60min_2haux_pol2cst_aff4aid_dld.txt
    # pair4_GSE145525_pol2_ctrl_2haux_aff4aid_dld.txt
    ("SRX7748745", "SRX7748743"), # 2haux_42c1h_pol2cst_aff4aid_dld_rep1 vs. 2haux_37c1h_pol2cst_aff4aid_dld_rep1
    ("SRX7748757", "SRX7748755"), # 2haux_42c1h_pol2cst_aff4aid_dld_rep2 vs. 2haux_37c1h_pol2cst_aff4aid_dld_rep2
    ("SRX7748765", "SRX7748763"), # 2haux_42c1h_pol2cst_aff4aid_dld_rep3 vs. 2haux_37c1h_pol2cst_aff4aid_dld_rep3


    # pair5_GSE145525_pol2_hs_42C_60min_2haux_pol2cst_brd2aid_dld.txt
    # pair5_GSE145525_pol2_ctrl_2haux_brd2aid_dld.txt
    ("SRX7748715", "SRX7748695"), # 2haux_42c1h_pol2cst_brd2aid_dld_rep1 vs. 2haux_37c1h_pol2cst_brd2aid_dld_rep1
    ("SRX7748722", "SRX7748703"), # 2haux_42c1h_pol2cst_brd2aid_dld_rep2 vs. 2haux_37c1h_pol2cst_brd2aid_dld_rep2
 
    # replicate が 2つ以上ない?
    # ありました！
    # pair6_GSE145525_pol2_hs_42C_60min_2haux_pol2cst_brd3aid_dld.txt
    # pair6_GSE145525_pol2_ctrl_2haux_brd3aid_dld.txt
    ("SRX7748717", "SRX7748697"), # 2haux_42c1h_pol2cst_brd3aid_dld_rep1 vs. 2haux_37c1h_pol2cst_brd3aid_dld_rep1
    ("SRX7748724", "SRX7748705"), # 2haux_42c1h_pol2cst_brd3aid_dld_rep2 vs. 2haux_37c1h_pol2cst_brd3aid_dld_rep2


    # pair7_GSE145525_pol2_hs_42C_60min_2haux_pol2cst_brd4aid_dld.txt
    # pair7_GSE145525_pol2_ctrl_2haux_brd4aid_dld.txt
    ("SRX7748719", "SRX7748699"), # 2haux_42c1h_pol2cst_brd4aid_dld_rep1 vs. 2haux_37c1h_pol2cst_brd4aid_dld_rep1
    ("SRX7748726", "SRX7748707"), # 2haux_42c1h_pol2cst_brd4aid_dld_rep2 vs. 2haux_37c1h_pol2cst_brd4aid_dld_rep2
    ("SRX7748729", "SRX7748709"), # 2haux_42c1h_pol2cst_brd4aid_dld_rep3 vs. 2haux_37c1h_pol2cst_brd4aid_dld_rep3

    # pair8_GSE145525_pol2_hs_42C_60min_2haux_pol2cst_doubleaid_dld.txt
    # pair8_GSE145525_pol2_ctrl_2haux_doubleaid_dld.txt
    ("SRX7748733", "SRX7748732"), # 2haux_42c1h_pol2cst_doubleaid_dld_rep1 vs. 2haux_37c1h_pol2cst_doubleaid_dld_rep1
    ("SRX7748737", "SRX7748736"), # 2haux_42c1h_pol2cst_doubleaid_dld_rep2 vs. 2haux_37c1h_pol2cst_doubleaid_dld_rep2

    # replicate が 2つ以上ない?
    ("SRX7748727", "SRX7748700"), # 2hdmso_42c1h_pol2cst_brd4aid_dld_rep2 vs. 2hdmso_37c1h_pol2cst_brd4aid_dld_rep2
    ("SRX7748720", "SRX7748692"), # 2hdmso_42c1h_pol2cst_wt_dld_rep1 vs. 2hdmso_37c1h_pol2cst_wt_dld_rep1
    
    ("SRX7748787", "SRX7748785"), # 2hh2o_42c1h_aff1bethyl_aff1aid_dld_rep1 vs. 2hh2o_37c1h_aff1bethyl_aff1aid_dld_rep1
    ("SRX7748791", "SRX7748789"), # 2hh2o_42c1h_aff4protec_aff4aid_dld_rep1 vs. 2hh2o_37c1h_aff4protec_aff4aid_dld_rep1
    ("SRX7748799", "SRX7748797"), # 2hh2o_42c1h_aid660_aff1aid_dld_rep1 vs. 2hh2o_37c1h_aid660_aff1aid_dld_rep1
    ("SRX7748803", "SRX7748801"), # 2hh2o_42c1h_aid660_aff4aid_dld_rep1 vs. 2hh2o_37c1h_aid660_aff4aid_dld_rep1
    ("SRX7748775", "SRX7748773"), # 2hh2o_42c1h_aid660_brd2aid_dld_rep1 vs. 2hh2o_37c1h_aid660_brd2aid_dld_rep1
    ("SRX7748783", "SRX7748781"), # 2hh2o_42c1h_aid660_brd3aid_dld_rep1 vs. 2hh2o_37c1h_aid660_brd3aid_dld_rep1
    ("SRX7748779", "SRX7748777"), # 2hh2o_42c1h_aid660_brd4aid_dld_rep1 vs. 2hh2o_37c1h_aid660_brd4aid_dld_rep1
    ("SRX7748795", "SRX7748793"), # 2hh2o_42c1h_aid660_doubleaid_dld_rep1 vs. 2hh2o_37c1h_aid660_doubleaid_dld_rep1

    # pair9_GSE145525_pol2_hs_42C_60min_2hh2o_pol2cst_aff1aid_dld.txt
    # pair9_GSE145525_pol2_ctrl_2hh2o_aff1aid_dld.txt
    ("SRX7748740", "SRX7748738"), # 2hh2o_42c1h_pol2cst_aff1aid_dld_rep1 vs. 2hh2o_37c1h_pol2cst_aff1aid_dld_rep1
    ("SRX7748752", "SRX7748750"), # 2hh2o_42c1h_pol2cst_aff1aid_dld_rep2 vs. 2hh2o_37c1h_pol2cst_aff1aid_dld_rep2
    ("SRX7748760", "SRX7748758"), # 2hh2o_42c1h_pol2cst_aff1aid_dld_rep3 vs. 2hh2o_37c1h_pol2cst_aff1aid_dld_rep3

    # pair10_GSE145525_pol2_hs_42C_60min_2hh2o_pol2cst_aff4aid_dld.txt
    # pair10_GSE145525_pol2_ctrl_2hh2o_aff4aid_dld.txt
    ("SRX7748744", "SRX7748742"), # 2hh2o_42c1h_pol2cst_aff4aid_dld_rep1 vs. 2hh2o_37c1h_pol2cst_aff4aid_dld_rep1
    ("SRX7748756", "SRX7748754"), # 2hh2o_42c1h_pol2cst_aff4aid_dld_rep2 vs. 2hh2o_37c1h_pol2cst_aff4aid_dld_rep2
    ("SRX7748764", "SRX7748762"), # 2hh2o_42c1h_pol2cst_aff4aid_dld_rep3 vs. 2hh2o_37c1h_pol2cst_aff4aid_dld_rep3

    # pair11_GSE145525_pol2_hs_42C_60min_2hh2o_pol2cst_brd2aid_dld.txt
    # pair11_GSE145525_pol2_ctrl_2hh2o_brd2aid_dld.txt
    ("SRX7748714", "SRX7748694"), # 2hh2o_42c1h_pol2cst_brd2aid_dld_rep1 vs. 2hh2o_37c1h_pol2cst_brd2aid_dld_rep1
    ("SRX7748721", "SRX7748702"), # 2hh2o_42c1h_pol2cst_brd2aid_dld_rep2 vs. 2hh2o_37c1h_pol2cst_brd2aid_dld_rep2

    # pair12_GSE145525_pol2_hs_42C_60min_2hh2o_pol2cst_brd3aid_dld.txt
    # pair12_GSE145525_pol2_ctrl_2hh2o_brd3aid_dld.txt
    ("SRX7748716", "SRX7748696"), # 2hh2o_42c1h_pol2cst_brd3aid_dld_rep1 vs. 2hh2o_37c1h_pol2cst_brd3aid_dld_rep1
    ("SRX7748723", "SRX7748704"), # 2hh2o_42c1h_pol2cst_brd3aid_dld_rep2 vs. 2hh2o_37c1h_pol2cst_brd3aid_dld_rep2

    # pair13_GSE145525_pol2_hs_42C_60min_2hh2o_pol2cst_brd4aid_dld.txt
    # pair13_GSE145525_pol2_ctrl_2hh2o_brd4aid_dld.txt
    ("SRX7748718", "SRX7748698"), # 2hh2o_42c1h_pol2cst_brd4aid_dld_rep1 vs. 2hh2o_37c1h_pol2cst_brd4aid_dld_rep1
    ("SRX7748725", "SRX7748706"), # 2hh2o_42c1h_pol2cst_brd4aid_dld_rep2 vs. 2hh2o_37c1h_pol2cst_brd4aid_dld_rep2
    ("SRX7748728", "SRX7748708"), # 2hh2o_42c1h_pol2cst_brd4aid_dld_rep3 vs. 2hh2o_37c1h_pol2cst_brd4aid_dld_rep3

    # pair14_GSE145525_pol2_hs_42C_60min_2hh2o_pol2cst_doubleaid_dld.txt
    # pair14_GSE145525_pol2_ctrl_2hh2o_doubleaid_dld.txt
    ("SRX7748731", "SRX7748730"), # 2hh2o_42c1h_pol2cst_doubleaid_dld_rep1 vs. 2hh2o_37c1h_pol2cst_doubleaid_dld_rep1
    ("SRX7748735", "SRX7748734"), # 2hh2o_42c1h_pol2cst_doubleaid_dld_rep2 vs. 2hh2o_37c1h_pol2cst_doubleaid_dld_rep2

    # pair15_GSE145525_pol2_hs_42C_60min_2hh2o_pol2cst_wt_dld.txt
    # pair15_GSE145525_pol2_ctrl_2hh2o_wt_dld.txt
    ("SRX7748711", "SRX7748710"), # 2hh2o_42c1h_pol2cst_wt_dld_rep1 vs. 2hh2o_37c1h_pol2cst_wt_dld_rep1
    ("SRX7748713", "SRX7748712"), # 2hh2o_42c1h_pol2cst_wt_dld_rep3 vs. 2hh2o_37c1h_pol2cst_wt_dld_rep3


    # replicate が 2つ以上ない?
    ("SRX7748748", "SRX7748746"), # 6hdmso_42c1h_pol2cst_aff4aid_dld_rep1 vs. 6hdmso_37c1h_pol2cst_aff4aid_dld_rep1

    # replicate が 2つ以上ない?
    ("SRX7748749", "SRX7748747"), # 6hkl1_20um_42c1h_pol2cst_aff4aid_dld_rep1 vs. 6hkl1_20um_37c1h_pol2cst_aff4aid_dld_rep1
]

paired_metada_df_gse145525 = create_paired_metadata(
    combined_df=GSE145525_combined,
    defined_sample_pairs=defined_pairs_gse145525_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse145525 is not None:
    paired_metada_df_gse145525.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE145525_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metada_df_gse145525)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,genotype,chip_antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX7748788""","""SRX7748786""","""2haux_42c1h_aff1bethyl_aff1aid_dld_rep1 vs 2haux_37c1h_aff1bethyl_aff1aid_dld_rep1""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR11111768 vs SRR11111766""","""2hAux_42C1h_AFF1Bethyl_AFF1AID_DLD_rep1 vs 2hAux_37C1h_AFF1Bethyl_AFF1AID_DLD_rep1""","""GSM4320293 vs GSM4320291""","""Public on Jan 15 2021""","""Feb 19 2020""","""Jan 16 2021""","""SRA""","""1""","""DLD-1""","""Homo sapiens""","""9606""","""Cells were treated with 500 µM Auxin (3-indole-acetic acid sodium salt, abcam), 250 nM dBET6 (AOBIOUS), or 20 nM KL-1 (Liang et al. 2018). Heat shock treatment at 42 ˚C for 1 h was carried out as desc…","""DLD-1 cells were cultured in DMEM (Corning) supplemented with 10% FBS (Corning), 1X GlutaMAX (Gibco), 100 U/ml Penicillin and 100 µg/ml Streptomycin (Gibco).""","""genomic DNA""","""For ChIP-seq, 20-50 million cells were crosslinked with 1% paraformaldehyde (Electron Microscopy Sciences) in PBS for 10 min at r.t. Chromatin was sonicated with the Covaris E220 for 4 min with 10% du…","""AFF1 ChIP-seq""","""For ChIP-seq, single-end raw reads were trimmed by Trimmomatic 0.33 (Bolger et al., 2014) and aligned to hg19 using Bowtie version 1.1.2, allowing only uniquely mapping reads with up to two mismatches…","""GPL18573""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4320293 vs GSM4320291""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4320nnn/GSM4320293/suppl/GSM4320293_2hAux_42C1h_AFF1Bethyl_AFF1AID_DLD_rep1.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4320nnn/GSM4320291/suppl/GSM432029…","""GSE145525""","""0""","""SRX7748788 vs SRX7748786""","""Colorectal adenocarcinoma DLD-1""","""AFF1-AID OsTIR1""","""AF4 Bethyl A302-345A""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN14135439 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN14135441""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX7748788 vs https://www.

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Strategy 1. Gene Expression Omnibus (GEO) から 一旦リストを集める

- 全てのリストを調べるのは不可能なので、以下のクエリでヒットしたアクセッションを調べる (2026-04-23)
- [../../Data/curation/GEO_ChIPseq_result/gds_result_summary_260423_chipseq_modified.md](../../Data/curation/GEO_ChIPseq_result/gds_result_summary_260423_chipseq_modified.md) のリストから取得
- `genome binding/occupancy profiling by high throughput sequencing` であることに注意

```text
Search (((("homo sapiens"[Organism]) AND "genome binding/occupancy profiling by high throughput sequen"[Filter]) \
AND "gds sra"[Filter])) AND \
(((((("heat shock response"[MeSH Terms]) OR \
"heat shock"[Description]) OR \
"heat stress"[Description]) OR \
"high temperature stress"[Description]) OR \
"hyperthermia"[Description]) OR \
"heat stroke"[Description])
```

&nbsp;

&nbsp;

## 3. GSE38901 (ChIP-seq; HSF1)

&nbsp;

- [HSF1 Drives a Transcriptional Program Distinct from Heat Shock to Support Highly Malignant Human Cancers](https://doi.org/10.1016/j.cell.2012.06.031)
- 42 C & 1h.
- replicate が2つ以上あることを満たす MCF10A のデータセットを ChIP-Atlas Diff Analysis に供する

In [8]:
GSE38901_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE38901/GSE38901_PEP/GSE38901_PEP_raw.csv",
    separator=','
).filter(
    (
        (pl.col("sample_name").str.contains("hme_")) |
        (pl.col("sample_name").str.contains("bpe_")) |
        (pl.col("sample_name").str.contains("mcf10a_"))
    ) &
    (~pl.col("sample_name").str.contains("_igg_")) & # remove IgG control
    (pl.col("organism") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1016/j.cell.2012.06.031").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE38901 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE38901.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE38901_filtered = GSE38901[[s.name for s in GSE38901 if not (s.null_count() == GSE38901.height)]]

GSE38901_combined = GSE38901_PEP.join(
    GSE38901_filtered,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE38901_combined = GSE38901_combined[[s.name for s in GSE38901_combined if not (s.null_count() == GSE38901_combined.height)]]

# pl.Config.set_tbl_rows(11)
# display(GSE38901_combined)

In [9]:
defined_pairs_gse38901_chipseq = [
    ("SRX155719", "SRX155715"), # HME_HS_HSF1_ChIPSeq_1 vs. HME_HSF1_ChIPSeq
    ("SRX155764", "SRX155715"), # HME_HS_HSF1_ChIPSeq_2 vs. HME_HSF1_ChIPSeq
    ("SRX155721", "SRX155717"), # BPE_HS_HSF1_ChIPSeq_1 vs. BPE_HSF1_ChIPSeq
    ("SRX155763", "SRX155717"), # BPE_HS_HSF1_ChIPSeq_2 vs. BPE_HSF1_ChIPSeq
    ("SRX155765", "SRX155717"), # BPE_HS_HSF1_ChIPSeq_3 vs. BPE_HSF1_ChIPSeq

    # pair1_GSE38901_hsf1_mcf10a_hs_42C_60min.txt
    # pair1_GSE38901_hsf1_mcf10a_ctrl.txt
    ("SRX155730", "SRX155727"), # MCF10A_HS_HSF1_ChIPSeq_1 vs. MCF10A_HSF1_ChIPSeq_1
    ("SRX155731", "SRX155728")  # MCF10A_HS_HSF1_ChIPSeq_2 vs. MCF10A_HSF1_ChIPSeq_2
]

paired_metada_df_gse38901 = create_paired_metadata(
    combined_df=GSE38901_combined,
    defined_sample_pairs=defined_pairs_gse38901_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse38901 is not None:
    paired_metada_df_gse38901.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE38901_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_cols(100)
    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse38901)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_type,antibody,tumorigenic_or_nontumorigenic,42˚_c_1h_heat-shock_,genome_build,sra,biosample,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,tax_lineage,sample_alias,run_alias,experiment_alias,center_name,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX155719""","""SRX155715""","""hme_hs_hsf1_chipseq_1 vs hme_hsf1_chipseq""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR515394 vs SRR515390""","""HME_HS_HSF1_ChIPSeq_1 vs HME_HSF1_ChIPSeq""","""GSM951853 vs GSM951849""","""Public on Aug 07 2012""","""Jun 24 2012""","""May 15 2019""","""SRA""","""1""","""HME cells""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL9052""","""Marc,,Mendillo""","""Whitehead Institute for Biomedical Research""","""9 Cambridge Center""","""Cambridge""","""MA""","""2142""","""USA""","""Illumina Genome Analyzer""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM951853 vs GSM951849""","""NONE""","""GSE38901, GSE38912""","""0""","""SRX155719 vs SRX155715""","""Immortalized mammary epithelial cells""","""HSF1 (H-311, Santa Cruz, sc-9144, Lot# J0510)""","""Non-tumorigenic""","""YES vs No""","""HG18""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX155719 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX155715""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN01057454 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN01057450""","""https://doi.org/10.1016/j.cell.2012.06.031""","""42 C""","""60 min.""","""Illumina Genome Analyzer sequencing: GSM951853: HME_HS_HSF1_ChIPSeq_1 Homo sapiens ChIP-Seq vs Illumina Genome Analyzer sequencing: GSM951849: HME_HSF1_ChIPSeq Homo sapiens ChIP-Seq""","""SAMN01057454 vs SAMN01057450""","""Homo sapiens""","""SRA054929""","""ChIP""","""2019-10-18 vs 2021-01-10""","""395492634 vs 308979986""","""ILLUMINA""","""Generic""","""HME_HS_HSF1_ChIPSeq_1 vs HME_HSF1_ChIPSeq""","""8fb57236cebfafb706054318b1b51cc9 vs 453c77b1f50fdd2a8a07fbcde1b076d6""","""SRP013883""","""18271201 vs 14234092""","""HSF1 drives a transcriptional program distinct from heat shock to support highly malignant human cancers [ChIP-Seq]""","""GENERATED_FILE""","""HME_HS_HSF1_ChIPSeq_1 vs HME_HSF1_ChIPSeq""","""Illumina Genome Analyzer""","""Illumina Genome Analyzer sequencing: GSM951853: HME_HS_HSF1_ChIPSeq_1 Homo sapiens ChIP-Seq vs Illumina Genome Analyzer sequencing: GSM951849: HME_HSF1_ChIPSeq Homo sapiens ChIP-Seq""","""Immortalized mammary epithelial cells""","""639492035 vs 498193220""","""ChIP-Seq""","""GENOMIC""","""SINGLE""","""public""","""1;131567;2759;33154;33208;6072;33213;33511

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. GSE43579 (ChIP-seq; HSF1)

- [Transcriptional response to stress in the dynamic chromatin environment of cycling and mitotic cells](https://doi.org/10.1073/pnas.1305275110)
- IgG control も含まれているが、熱ストレス条件下 vs コントロール条件下のデータが揃っているため、偽陽性のピーク (diff analysis で検出されたピーク？)を 取り除ける可能性があるため、収集しておく

In [10]:
GSE43579_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE43579/GSE43579_PEP/GSE43579_PEP_raw.csv",
    separator=','
).filter(
    ~pl.col("sample_name").str.contains("input_sample_")
).with_columns(
    pl.lit("https://doi.org/10.1073/pnas.1305275110").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"), # from treatment column
    pl.lit("30 min.").alias("stress treatment time") # from treatment column
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE43579 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE43579.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE43579_filtered = GSE43579[[s.name for s in GSE43579 if not (s.null_count() == GSE43579.height)]]

GSE43579_combined = GSE43579_PEP.join(
    GSE43579_filtered,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE43579_combined = GSE43579_combined[[s.name for s in GSE43579_combined if not (s.null_count() == GSE43579_combined.height)]]

# pl.Config.set_tbl_rows(14)
# display(GSE43579_combined)

In [11]:
defined_pairs_gse43579_chipseq = [

    # pair2_GSE43579_hsf1_k562_hs_42C_30min.txt
    # pair2_GSE43579_hsf1_k562_ctrl.txt
    ("SRX218220", "SRX218217"), # HSF1_heat-treated cycling K562 vs. HSF1_non-treated cycling K562
    ("SRX218221", "SRX218218"), # HSF2_heat-treated cycling K562 vs. HSF2_non-treated cycling K562

    ("SRX218222", "SRX218219"), # IgG_heat-treated cycling K562 vs. IgG_non-treated cycling K562

    # pair3_GSE43579_hsf1_k562_mitotic_hs_42C_30min.txt
    # pair3_GSE43579_hsf1_k562_mitotic_ctrl.txt
    ("SRX218227", "SRX218224"), # HSF1_heat-treated mitotic K562 vs. HSF1_non-treated mitotic K562
    ("SRX218228", "SRX218225"), # HSF2_heat-treated mitotic K562 vs. HSF2_non-treated mitotic K562
    
    ("SRX218229", "SRX218226"), # IgG_heat-treated mitotic K562 vs. IgG_non-treated mitotic K562
]

paired_metada_df_gse43579 = create_paired_metadata(
    combined_df=GSE43579_combined,
    defined_sample_pairs=defined_pairs_gse43579_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse43579 is not None:
    paired_metada_df_gse43579.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE43579_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_cols(100)
    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse43579)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,sample_series_id,sample_data_row_count,big_key,cell_line,cell_cycle_phase,treatment,chip_antibody,genome_build,sra,biosample,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,tax_lineage,sample_alias,run_alias,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX218220""","""SRX218217""","""hsf1_heat-treated_cycling_k562 vs hsf1_non-treated_cycling_k562""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR650334 vs SRR650331""","""HSF1_heat-treated cycling K562 vs HSF1_non-treated cycling K562""","""GSM1065714 vs GSM1065711""","""Public on Aug 04 2013""","""Jan 17 2013""","""May 15 2019""","""SRA""","""1""","""K562 cells""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL10999""","""Anniina,,Vihervaara""","""mavih@kth.se""","""+358 50 5463694""","""Vihervaara""","""Gene Technology""","""KTH Royal Institute of Technology""","""Tomtebodavägen 23A""","""Stockholm""","""17165""","""Sweden""","""Illumina Genome Analyzer IIx""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM1065714 vs GSM1065711""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1065nnn/GSM1065714/suppl/GSM1065714_HSF1_heat-treated_cycling_K562_against_Input_peaks.txt.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1065nnn/GSM1065711/…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1065nnn/GSM1065714/suppl/GSM1065714_c_s_4_HSF1_heat-treated_cycling_K562_all.wig.gz.tdf vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1065nnn/GSM1065711/suppl/…","""GSE43579""","""0""","""SRX218220 vs SRX218217""","""K562 cells""","""freely cycling""","""30 min at 42C vs no treatment""","""HSF1 (SPA-901, Enzo, Lot#09011025)""","""hg19""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX218220 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX218217""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN01889090 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN01889087""","""https://doi.org/10.1073/pnas.1305275110""","""42 C""","""30 min.""","""Illumina Genome Analyzer IIx sequencing: GSM1065714: HSF1_heat-treated cycling K562 Homo sapiens ChIP-Seq vs Illumina Genome Analyzer IIx sequencing: GSM1065711: HSF1_non-treated cycling K562 Homo sap…","""SAMN01889090 vs SAMN01889087""","""Homo sapiens""","""SRA065119""","""ChIP""","""2021-01-10 vs 2019-10-28""","""1078361307 vs 783043494""","""ILLUMINA""","""Generic""","""HSF1_heat-treated cycling K562 vs HSF1_non-treated cycling K562""","""0925695bba6122f3e7949ef1f3db54bf vs 18f1b0f043d2c0f09791d37c8583585d""","""SRP01801

&nbsp;

&nbsp;

&nbsp;

## 5. GSE62309 (ChIP-seq; p-KDM3A)

- [Specific Phosphorylation of Histone Demethylase KDM3A Determines Target Gene Expression in Response to Heat Shock](https://doi.org/10.1371/journal.pbio.1002026)
- Histone Demethylase KDM3A を標的にした ChIP-seq のデータ
- 抗体で使用されている、`KDM3A`と、`p-KDM3A`の違いは、リン酸化されているものを標的にしているかどうか
- `KDM3A` を標的としている ChIP-seq のデータは、ペアとなるデータがなかったため、除外しておく

In [12]:
GSE62309_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE62309/GSE62309_PEP/GSE62309_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_name") != "input_pool") & # SRX732512
    (pl.col("sample_name") != "kdm3a_hs_-_") # SRX732509
).with_columns(
    pl.lit("https://doi.org/10.1371/journal.pbio.1002026").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("30 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE62309 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE62309.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE62309_combined = GSE62309_PEP.join(
    GSE62309,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE62309_combined = GSE62309_combined[[s.name for s in GSE62309_combined if not (s.null_count() == GSE62309_combined.height)]]

# pl.Config.set_tbl_rows(14)
# display(GSE62309_combined)

In [13]:
defined_pairs_gse62309_chipseq = [
    ("SRX732511", "SRX732510")  # p-KDM3A_HS(+) vs. p-KDM3A_HS(-)
]

paired_metada_df_gse62309 = create_paired_metadata(
    combined_df=GSE62309_combined,
    defined_sample_pairs=defined_pairs_gse62309_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse62309 is not None:
    paired_metada_df_gse62309.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE62309_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metada_df_gse62309)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,sample_series_id,sample_data_row_count,big_key,cell_line,cell_type,treatment,chip_antibody,antibody_manufacturer,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX732511""","""SRX732510""","""p-kdm3a_hs_+_ vs p-kdm3a_hs_-_""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1611952 vs SRR1611951""","""p-KDM3A_HS(+) vs p-KDM3A_HS(-)""","""GSM1525105 vs GSM1525104""","""Public on Oct 15 2014""","""Oct 14 2014""","""May 15 2019""","""SRA""","""1""","""Jurkat cell line""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL10999""","""Ye,,Zhang""","""yezhang@pumc.edu.cn""","""861069155939""","""5 Dongdan Santiao""","""Beijing""","""Not applicable""","""100005""","""China""","""Illumina Genome Analyzer IIx""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM1525105 vs GSM1525104""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1525nnn/GSM1525105/suppl/GSM1525105_6_p-KDM3A_heat_shock_clean_uniqnorm_hits-W200-G200-FDR1E-10-island.bed.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM152…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1525nnn/GSM1525105/suppl/GSM1525105_6_p-KDM3A_heat_shock_clean_uniqnorm_signal.wig.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1525nnn/GSM1525104/suppl/GS…","""GSE62309""","""0""","""SRX732511 vs SRX732510""","""Jurkat""","""Acute T Cell Leukemia T lymphocyte""","""42℃ 1h vs 37℃ 1h""","""p-KDM3A""","""customized from Beijing B&M Biotech (Beijing, China)""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN03107086 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN03107085""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX732511 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX732510""","""https://doi.org/10.1371/journal.pbio.1002026""","""42 C""","""30 min.""","""Illumina Genome Analyzer IIx sequencing: GSM1525105: p-KDM3A_HS(+) Homo sapiens ChIP-Seq vs Illumina Genome Analyzer IIx sequencing: GSM1525104: p-KDM3A_HS(-) Homo sapiens ChIP-Seq""","""SAMN03107086 vs SAMN03107085""","""Homo sapiens""","""SRA190777""","""ChIP""","""2020-05-14""","""893711604 vs 782761489""","""ILLUMINA""","""Generic""","""p-KDM3A_HS(+) vs p-KDM3A_HS(-)""","""462150e21fd008c2caa2b91dfcf7cfd2 vs 74a20c10bb4a7a1e0f9c435272b03357""","""SRP048900""","""28399220 vs 24970499""","""Specific phosphorylation of histone demethylase KDM3A determines target gene expression in response to heat 

&nbsp;

&nbsp;

&nbsp;

## 6. GSE57398 (ChIP-seq; HSF1)

- [FBXW7 modulates cellular stress response and metastatic potential through HSF1 post-translational modification.](https://doi.org/10.1038/ncb3121)
- Input control data が、熱ストレス条件下 vs. コントロール条件下のデータがあるので、念のため収集しておく
- control の replicate が2つ以上ないため、ChIP-Atlas Diff Analysis は実行しない

In [14]:
GSE57398_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE57398/GSE57398_PEP/GSE57398_PEP_raw.csv",
    separator=','
).filter(
    pl.col("sample_library_strategy") == "ChIP-Seq"
).with_columns(
    pl.lit("https://doi.org/10.1038/ncb3121").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE57398 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE57398.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE57398_combined = GSE57398_PEP.join(
    GSE57398,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE57398_combined = GSE57398_combined[[s.name for s in GSE57398_combined if not (s.null_count() == GSE57398_combined.height)]]

# pl.Config.set_tbl_rows(14)
# display(GSE57398_combined)

In [15]:
defined_pairs_gse57398_chipseq = [

    # FBXW7 KO
    ("SRX535173", "SRX535176"), # ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO cells upon heat shock treatment replicate1 vs. ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO untreated cells replicate1
    ("SRX535174", "SRX535176"), # ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO cells upon heat shock treatment replicate2 vs. ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO untreated cells replicate1
    ("SRX535175", "SRX535177"), # Sequencing analysis of input in HCT116 FBXW7 KO cells upon heat shock treatment vs. Sequencing analysis of input in HCT116 FBXW7 KO untreated cells

    # FBXW7 WT
    ("SRX535178", "SRX535181"), # ChIP-Seq analysis of HSF1 in HCT116 WT cells upon heat shock treatment replicate1 vs. ChIP-Seq analysis of HSF1 in HCT116 WT untreated cells replicate1
    ("SRX535179", "SRX535181"), # ChIP-Seq analysis of HSF1 in HCT116 WT cells upon heat shock treatment replicate2 vs. ChIP-Seq analysis of HSF1 in HCT116 WT untreated cells replicate1
    ("SRX535180", "SRX535182")  # Sequencing analysis of input in HCT116 WT cells upon heat shock treatment vs. Sequencing analysis of input in HCT116 WT untreated cells
]

paired_metada_df_gse57398 = create_paired_metadata(
    combined_df=GSE57398_combined,
    defined_sample_pairs=defined_pairs_gse57398_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse57398 is not None:
    paired_metada_df_gse57398.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE57398_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse57398)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_type,cell_line,chip_antibody,genome_build,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX535173""","""SRX535176""","""chip-seq_analysis_of_hsf1_in_hct116_fbxw7_ko_cells_upon_heat_shock_treatment_replicate1 vs chip-seq_analysis_of_hsf1_in_hct116_fbxw7_ko_untreated_cells_replicate1""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1274639 vs SRR1274642""","""ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO cells upon heat shock treatment replicate1 vs ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO untreated cells replicate1""","""GSM1382049 vs GSM1382052""","""Public on Feb 16 2015""","""May 07 2014""","""May 15 2019""","""SRA""","""1""","""HCT116 FBXW7 KO""","""Homo sapiens""","""9606""","""attached cells, growing in McCoy's 5A""","""genomic DNA""","""GPL11154""","""Nikos,,Kourtis""","""nkourtis@gmail.com""","""NYU School of Medicine""","""550 First Avenue""","""New York""","""10016""","""USA""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM1382049 vs GSM1382052""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1382nnn/GSM1382049/suppl/GSM1382049_macs.KO_HS_HSF1_REP1.vs.KO_HS_INPUT.peaks.bed.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1382nnn/GSM1382052/suppl/GSM…","""GSE57398, GSE57399""","""0""","""SRX535173 vs SRX535176""","""human colon cancer cells""","""HCT116""","""HSF1: Cell Signaling, 4356""","""hg19""","""bed format containg relative enrichments""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN02767270 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN02767272""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX535173 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX535176""","""https://doi.org/10.1038/ncb3121""","""42 C""","""60 min.""","""Illumina HiSeq 2000 sequencing: GSM1382049: ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO cells upon heat shock treatment replicate1 Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM1382052:…","""SAMN02767270 vs SAMN02767272""","""Homo sapiens""","""SRA162008""","""ChIP""","""2019-10-11 vs 2020-05-20""","""1075787849 vs 1166113437""","""ILLUMINA""","""Generic""","""ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO cells upon heat shock treatment replicate1 vs ChIP-Seq analysis of HSF1 in HCT116 FBXW7 KO untreated cells replicate1""","""e20633ffdeb9747aea0cace3b

&nbsp;

&nbsp;

&nbsp;

## 7. GSE66448 (ChIP-seq; SUMO2 / Pol II)

- [Global SUMOylation on active chromatin is an acute heat stress response restricting transcription](https://doi.org/10.1186/s13059-015-0717-y)
- Input control data が、熱ストレス条件下 vs. コントロール条件下のデータがあるので、念のため収集しておく
- 

In [16]:
GSE66448_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE66448/GSE66448_PEP/GSE66448_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (~pl.col("sample_name").str.contains("vcap-chipseq-pias1")) & # remove vcap-chipseq-pias1 because it is not included ctrl "-c-" ChIP-seq data
    (~pl.col("sample_name").str.contains("vcap-chipseq-sumo2-")) & # remove vcap-chipseq-sumo2- because it is not included ctrl "-c-" ChIP-seq data
    (~pl.col("sample_name").str.contains("vcap-drb-chipseq-sumo2")) & # remove vcap-drb-chipseq-sumo2 because it is not included heat-shocked "-hs-" ChIP-seq data
    (~pl.col("sample_name").str.contains("vcap-trp-chipseq-sumo2"))   # remove vcap-trp-chipseq-sumo2 because it is not included heat-shocked "-hs-" ChIP-seq data
).with_columns(
    pl.lit("https://doi.org/10.1186/s13059-015-0717-y").alias("reference_doi"),
    pl.lit("43 C").alias("stress treatment temperature"),
    pl.lit("30 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE66448 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE66448.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE66448_combined = GSE66448_PEP.join(
    GSE66448,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE66448_combined = GSE66448_combined[[s.name for s in GSE66448_combined if not (s.null_count() == GSE66448_combined.height)]]

pl.Config.set_tbl_rows(2)
display(GSE66448_PEP)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,treatment1,chip_antibody,processed_data_file,genome_build,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str
"""k562-chipseq-sumo2-c-rep1""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1823895""","""SRX895417""","""K562-ChIPseq-SUMO2-C-rep1""","""GSM1622606""","""Public on Jul 02 2015""","""Mar 03 2015""","""May 15 2019""","""SRA""",1,"""K562 cells""","""Homo sapiens""",9606,"""Heat shock for ChIP-seq samples was done at 43 degree C incubator for 30min.""","""K562 cells were grown in RPMI1640 medium supplemented with 10% fetal bovine serum, 1% penicillin/streptomycin antibiotics and 2mM l-glutamine at 37 degree C temperature incubator with 5% CO2.""","""genomic DNA""","""Nuclei from approximately 10 million formaldehyde crosslinked (1%; 10min at room temperature for VCaP cells, and 8min. at room temperature for K562 cells) cells were isolated and chromatin was fragme…","""ChIP-seq""","""GPL11154""","""Einari,A,Niskanen""","""einari.niskanen@uef.fi""","""Biomedicine""","""School of Medicine""","""University of Eastern Finland""","""Yliopistonranta 8""","""Kuopio""",70210,"""Finland""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM1622606""","""NONE""","""GSE66448""",0,"""SRX895417""","""K562""","""control (37 degree C)""","""anti-SUMO-2/3 (MBL, catalog# M114-3, lot# 024)""","""K562-SUMO2-C.bedGraph.gz""","""hg19""","""ChIP-seq and GRO-seq signal intensities as bedGraph files.""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN03383019""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX895417""","""https://doi.org/10.1186/s13059-015-0717-y""","""43 C""","""30 min."""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""vcap-sipias1-chipseq-sumo2-hs-rep2""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR2085154""","""SRX1079389""","""VCaP-siPIAS1-ChIPseq-SUMO2-HS-rep2""","""GSM1777605""","""Public on Jul 02 2015""","""Jul 02 2015""","""May 15 2019""","""SRA""",1,"""VCaP cells""","""Homo sapiens""",9606,"""Heat shock for ChIP-seq samples was done at 43 degree C incubator for 30min., Silencing for VCaP cells was done by transfecting siRNA for UBE2I (siUBE2I samples), PIAS1 (siPIAS1 samples) or non-target…","""VCaP cells were grown in DMEM medium supplemented with 10% feta bovine serum and 1% penicillin/streptomycin antibiotics at 37 degree C temperature incubator with 5% CO2.""","""genomic DNA""","""Nuclei from approximately 10 million formaldehyde crosslinked (1%; 10min at room temperature for VCaP cells, and 8min. at room temperature for K562 cells) cells were isolated and chromatin was fragme…","""ChIP-seq""","""GPL11154""","""Einari,A,Niskanen""","""einari.niskanen@uef.fi""","""Biomedicine""","""School of Medicine""","""University of Eastern Finland""","""Yliopistonranta 8""","""Kuopio""",70210,"""Finland""","""Illumina HiSeq 2000""","""ChIP""","""genomic

In [17]:
defined_pairs_gse66448_chipseq = [
    ("SRX895422", "SRX895421"), # k562-chipseq-input-hs vs. k562-chipseq-input
    ("SRX895419", "SRX895417"), # k562-chipseq-sumo2-hs-rep1 vs. k562-chipseq-sumo2-c-rep1
    ("SRX895420", "SRX895418"), # k562-chipseq-sumo2-hs-rep2 vs. k562-chipseq-sumo2-c-rep2
    ("SRX895429", "SRX895427"), # k562-shhsf1-chipseq-sumo2-hs-rep1 vs. k562-shhsf1-chipseq-sumo2-c-rep1
    ("SRX895430", "SRX895428"), # k562-shhsf1-chipseq-sumo2-hs-rep2 vs. k562-shhsf1-chipseq-sumo2-c-rep2
    ("SRX895433", "SRX895431"), # k562-shscr-chipseq-sumo2-hs-rep1 vs. k562-shscr-chipseq-sumo2-c-rep1
    ("SRX895434", "SRX895432"), # k562-shscr-chipseq-sumo2-hs-rep2 vs. k562-shscr-chipseq-sumo2-c-rep2

    # pair15_GSE66448_pol2_hs_43C_30min_vcap.txt
    # pair15_GSE66448_pol2_ctrl_vcap.txt
    ("SRX895445", "SRX895443"), # vcap-chipseq-pol2-hs-rep1 vs. vcap-chipseq-pol2-c-rep1
    ("SRX895446", "SRX895444"), # vcap-chipseq-pol2-hs-rep2 vs. vcap-chipseq-pol2-c-rep2

    # "-re-" means heat shock and recovery (30min at 43 degree C followed by 1h at 37 degree C)

    # ("SRX895447", "SRX895443"), # vcap-chipseq-pol2-re-rep1 vs. vcap-chipseq-pol2-c-rep1
    # ("SRX895448", "SRX895444"), # vcap-chipseq-pol2-re-rep2 vs. vcap-chipseq-pol2-c-rep2
    

    # pair16_GSE66448_pol2_hs_43C_30min_vcap_re.txt
    # pair16_GSE66448_pol2_ctrl_vcap_re.txt
    ("SRX1079376", "SRX1079374"), # vcap-sinon-chipseq-pol2-hs-rep1 vs. vcap-sinon-chipseq-pol2-c-rep1
    ("SRX1079377", "SRX1079375"), # vcap-sinon-chipseq-pol2-hs-rep2 vs. vcap-sinon-chipseq-pol2-c-rep2

    
    ("SRX1079384", "SRX1079382"), # vcap-sinon-chipseq-sumo2-hs-rep1 vs. vcap-sinon-chipseq-sumo2-c-rep1
    ("SRX1079385", "SRX1079383"), # vcap-sinon-chipseq-sumo2-hs-rep2 vs. vcap-sinon-chipseq-sumo2-c-rep2
    ("SRX1079388", "SRX1079386"), # vcap-sipias1-chipseq-sumo2-hs-rep1 vs. vcap-sipias1-chipseq-sumo2-c-rep1
    ("SRX1079389", "SRX1079387"), # vcap-sipias1-chipseq-sumo2-hs-rep2 vs. vcap-sipias1-chipseq-sumo2-c-rep2

    # pair17_GSE66448_pol2_hs_43C_30min_vcap_siube2i.txt
    # pair17_GSE66448_pol2_ctrl_vcap_siube2i.txt
    ("SRX1079380", "SRX1079378"), # vcap-siube2i-chipseq-pol2-hs-rep1 vs. vcap-siube2i-chipseq-pol2-c-rep1
    ("SRX1079381", "SRX1079379")  # vcap-siube2i-chipseq-pol2-hs-rep2 vs. vcap-siube2i-chipseq-pol2-c-rep2
]

paired_metada_df_gse66448 = create_paired_metadata(
    combined_df=GSE66448_combined,
    defined_sample_pairs=defined_pairs_gse66448_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse66448 is not None:
    paired_metada_df_gse66448.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE66448_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse66448)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,treatment1,chip_antibody,processed_data_file,genome_build,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX895422""","""SRX895421""","""k562-chipseq-input-hs vs k562-chipseq-input""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1823900 vs SRR1823899""","""K562-ChIPseq-input-HS vs K562-ChIPseq-input""","""GSM1622611 vs GSM1622610""","""Public on Jul 02 2015""","""Mar 03 2015""","""May 15 2019""","""SRA""","""1""","""K562 cells""","""Homo sapiens""","""9606""","""Heat shock for ChIP-seq samples was done at 43 degree C incubator for 30min.""","""K562 cells were grown in RPMI1640 medium supplemented with 10% fetal bovine serum, 1% penicillin/streptomycin antibiotics and 2mM l-glutamine at 37 degree C temperature incubator with 5% CO2.""","""genomic DNA""","""Nuclei from approximately 10 million formaldehyde crosslinked (1%; 10min at room temperature for VCaP cells, and 8min. at room temperature for K562 cells) cells were isolated and chromatin was fragme…","""sequencing of fragmented chromatin""","""GPL11154""","""Einari,A,Niskanen""","""einari.niskanen@uef.fi""","""Biomedicine""","""School of Medicine""","""University of Eastern Finland""","""Yliopistonranta 8""","""Kuopio""","""70210""","""Finland""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM1622611 vs GSM1622610""","""NONE""","""GSE66448""","""0""","""SRX895422 vs SRX895421""","""K562""","""input control for HS samples (30min at 43 degree C) vs input control (37 degree C)""","""none (input)""","""K562-Input-HS.bedGraph.gz vs K562-Input-C.bedGraph.gz""","""hg19""","""ChIP-seq and GRO-seq signal intensities as bedGraph files.""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN03383037 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN03383041""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX895422 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX895421""","""https://doi.org/10.1186/s13059-015-0717-y""","""43 C""","""30 min.""","""Illumina HiSeq 2000 sequencing: GSM1622611: K562-ChIPseq-input-HS Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM1622610: K562-ChIPseq-input Homo sapiens ChIP-Seq""","""SAMN0

&nbsp;

&nbsp;

&nbsp;

## 8. GSE60984 (ChIP-seq; HSF1)

- [Cross talk between cytokine and hyperthermia-induced pathways: identification of different subsets of NF-κB-dependent genes regulated by TNFα and heat shock](https://doi.org/10.1007/s00438-015-1055-1)
- control の replicate が 2つ以上ないため、ChIP-Atlas Diff Analysis は実行しない

In [18]:
GSE60984_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE60984/GSE60984_PEP/GSE60984_PEP_raw.csv",
    separator=','
).filter(
    pl.col("sample_library_strategy") == "ChIP-Seq"
).with_columns(
    pl.lit("https://doi.org/10.1007/s00438-015-1055-1").alias("reference_doi"),
    pl.lit("43 C").alias("stress treatment temperature"),
    pl.when(pl.col("sample_name") == "hs_10").then(pl.lit("10 min."))
      .when(pl.col("sample_name") == "hs_20").then(pl.lit("20 min."))
      .when(pl.col("sample_name") == "control").then(pl.lit("ctrl"))
      .otherwise(pl.lit(None))
      .alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE60984 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE60984.metadata.tsv",
    separator='\t'
)

GSE60984_combined = GSE60984_PEP.join(
    GSE60984,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE60984_combined = GSE60984_combined[[s.name for s in GSE60984_combined if not (s.null_count() == GSE60984_combined.height)]]

pl.Config.set_tbl_rows(3)
display(GSE60984_combined)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,sample_series_id,sample_data_row_count,big_key,cell_line,antibody,sex,tissue_cells,age,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""hs_10""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1562556""","""SRX690823""","""HS_10""","""GSM1495181""","""Public on Aug 01 2015""","""Sep 02 2014""","""May 15 2019""","""SRA""",1,"""HSF1 ChIP DNA from cells heat shocked at 43°C for 10 minutes""","""Homo sapiens""",9606,"""A growth medium from experimental dish was replaced with an equal volume of CO2 saturated, pre-heated to 43°C medium. Dish was then placed in a growth incubator at 43°C and cultured for 10 minutes., D…","""genomic DNA""","""GPL10999""","""Tomasz,,Stokowy""","""tomasz.stokowy@uib.no""","""Scientific Computing""","""IT Division""","""University of Bergen""","""Nygardsgaten 5""","""Bergen""",5020,"""Norway""","""Illumina Genome Analyzer IIx""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM1495181""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1495nnn/GSM1495181/suppl/GSM1495181_annotation_biblP10_PK_131227.txt.gz""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1495nnn/GSM1495181/suppl/GSM1495181_biblP10_PK_comparative_peaks.bed.gz""","""GSE60984""",0,"""SRX690823""","""U-2 OS""","""anti-HSF1""","""female""","""bone/osteosarcoma""","""15 years""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN03015209""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX690823""","""https://doi.org/10.1007/s00438-015-1055-1""","""43 C""","""10 min.""","""Illumina Genome Analyzer IIx sequencing: GSM1495181: HS_10 Homo sapiens ChIP-Seq""","""SAMN03015209""","""Homo sapiens""","""SRA180957""","""ChIP""","""2019-10-07""",2978986626,"""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR156/006/SRR1562556/SRR1562556.fastq.gz""","""Generic""","""HS_10""","""978ccbabf65b35b2dae922bc46b0461a""","""SRP045982""",41622402,"""Subsets of NFkappaB-dependent genes are differentially regulated after combined treatment with hyperthermia and cytokine""","""GENERATED_FILE""","""HS_10""","""Illumina Genome Analyzer IIx""","""Illumina Genome Analyzer IIx sequencing: GSM1495181: HS_10 Homo sapiens ChIP-Seq""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR156/006/SRR1562556/SRR1562556.fastq.gz""",3329792160,"""ChIP-Seq""","""GENOMIC"""

In [19]:
defined_pairs_gse60984_chipseq = [
    ("SRX690823", "SRX690825"), # HS_10 vs. control
    ("SRX690824", "SRX690825")  # HS_20 vs. control
]

paired_metada_df_gse60984 = create_paired_metadata(
    combined_df=GSE60984_combined,
    defined_sample_pairs=defined_pairs_gse60984_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse60984 is not None:
    paired_metada_df_gse60984 = paired_metada_df_gse60984.with_columns(
        pl.col("stress treatment time").str.strip_suffix(" vs ctrl")
    )
    paired_metada_df_gse60984.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE60984_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse60984)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,sample_series_id,sample_data_row_count,big_key,cell_line,antibody,sex,tissue_cells,age,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX690823""","""SRX690825""","""hs_10 vs control""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1562556 vs SRR1562558""","""HS_10 vs control""","""GSM1495181 vs GSM1495183""","""Public on Aug 01 2015""","""Sep 02 2014""","""May 15 2019""","""SRA""","""1""","""HSF1 ChIP DNA from cells heat shocked at 43°C for 10 minutes vs HSF1 ChIP DNA from untreated cells cultured at 37°C""","""Homo sapiens""","""9606""","""A growth medium from experimental dish was replaced with an equal volume of CO2 saturated, pre-heated to 43°C medium. Dish was then placed in a growth incubator at 43°C and cultured for 10 minutes., D…","""genomic DNA""","""GPL10999""","""Tomasz,,Stokowy""","""tomasz.stokowy@uib.no""","""Scientific Computing""","""IT Division""","""University of Bergen""","""Nygardsgaten 5""","""Bergen""","""5020""","""Norway""","""Illumina Genome Analyzer IIx""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM1495181 vs GSM1495183""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1495nnn/GSM1495181/suppl/GSM1495181_annotation_biblP10_PK_131227.txt.gz vs NONE""",null,"""GSE60984""","""0""","""SRX690823 vs SRX690825""","""U-2 OS""","""anti-HSF1 vs none""","""female""","""bone/osteosarcoma""","""15 years""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN03015209 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN03015211""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX690823 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX690825""","""https://doi.org/10.1007/s00438-015-1055-1""","""43 C""","""10 min.""","""Illumina Genome Analyzer IIx sequencing: GSM1495181: HS_10 Homo sapiens ChIP-Seq vs Illumina Genome Analyzer IIx sequencing: GSM1495183: control Homo sapiens ChIP-Seq""","""SAMN03015209 vs SAMN03015211""","""Homo sapiens""","""SRA180957""","""ChIP""","""2019-10-07 vs 2020-05-16""","""2978986626 vs 3082094839""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR156/006/SRR1562556/SRR1562556.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR156/008/SRR1562558/SRR1562558.fastq.gz""","""Generic""","""HS_10 vs control""","""978ccbabf65b35b2dae922bc46b0461a vs 79a14f1

&nbsp;

&nbsp;

&nbsp;

## 9. GSE105028 (ChIP-seq; RAD21/CTCF/ZNF143/PolII/NANOG/KLF4/OCT4/CFOS/BRG1/HSF1/WAPL/NIPBL)

- [Architectural Proteins and Pluripotency Factors Cooperate to Orchestrate the Transcriptional Response of hESCs to Temperature Stress](https://doi.org/10.1016/j.molcel.2018.07.012)
- `Cell culture and heat shock treatment ... For heat shock treatment, 80% confluent H9 cells were changed into 42°C prewarmed conditioned SFM medium and transferred into a 42°C incubator for 1 h.`

In [20]:
GSE105028_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE105028/GSE105028_PEP/GSE105028_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &

    # removed histone modification ChIP-seq data
    (~pl.col("sample_name").str.contains("h3k27ac")) &
    (~pl.col("sample_name").str.contains("h3k27me3")) &
    (~pl.col("sample_name").str.contains("h3k4me1")) &
    (~pl.col("sample_name").str.contains("h3k4me3")) 
).with_columns(
    pl.lit("https://doi.org/10.1016/j.molcel.2018.07.012").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE105028 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE105028.metadata.tsv",
    separator='\t'
)

GSE105028_combined = GSE105028_PEP.join(
    GSE105028,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE105028_combined = GSE105028_combined[[s.name for s in GSE105028_combined if not (s.null_count() == GSE105028_combined.height)]]

pl.Config.set_tbl_rows(35)
display(GSE105028_combined)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,condition,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy_right,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""igg_chip-seq""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6177932""","""SRX3288554""","""IgG ChIP-Seq""","""GSM2816613""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""",1,"""H9 hESCs""","""Homo sapiens""",9606,"""genomic DNA""","""ChIP-Seq under normal conditions""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""",361102,"""China""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM2816613""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2816nnn/GSM2816613/suppl/GSM2816613_IgG_NT.bw""","""GSE105028""",0,"""SRX3288554""","""H9 human Embryonic Stem Cell Line""","""under normal conditions""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07790039""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3288554""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""42 C""","""60 min.""","""Illumina HiSeq 2000 sequencing: GSM2816613: IgG ChIP-Seq Homo sapiens ChIP-Seq""","""SAMN07790039""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""SRA619794""","""ChIP""","""2018-08-04""","""421179770;435277223""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/002/SRR6177932/SRR6177932_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/002/SRR6177932/SRR6177932_2.fastq.gz""","""Generic""","""IgG ChIP-Seq""","""dcdc26b53280c08f2f6f6f41551c0674;4a1b27c121effc3fb74b48890f147c18""","""SRP120027""",14145363,"""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""GENERATED_FILE;GENERATED_FILE""","""IgG ChIP-Seq""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM2816613: IgG ChIP-Seq Homo sapiens ChIP-Seq""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/002/SRR6177932/SRR6177932_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/002/SRR6177932/SRR6177932_2.fastq.gz""",1414536300,"""ChIP-Seq""","""GENOMIC""","""PAIRED""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR617/002/SR

In [21]:
defined_pairs_gse105028_chipseq = [
    ("SRX3288555", "SRX3288554"), # hs_igg_chip-seq vs. igg_chip-seq
    ("SRX3288558", "SRX3288556"), # hs_rad21_chip-seq_rep1 vs. rad21_chip-seq_rep1
    ("SRX3288559", "SRX3288557"), # hs_rad21_chip-seq_rep2 vs. rad21_chip-seq_rep2
    ("SRX3288561", "SRX3288560"), # hs_ctcf_chip-seq vs. ctcf_chip-seq
    ("SRX3288563", "SRX3288562"), # hs_znf143_chip-seq vs. znf143_chip-seq
    ("SRX3288565", "SRX3288564"), # hs_rnapol_ii_chip-seq vs. rnapol_ii_chip-seq (replicate が 2つ以上ないので、diff analysis では除外)
    ("SRX3288567", "SRX3288566"), # hs_nanog_chip-seq vs. nanog_chip-seq
    ("SRX3288569", "SRX3288568"), # hs_klf4_chip-seq vs. klf4_chip-seq
    ("SRX3288571", "SRX3288570"), # hs_oct4_chip-seq vs. oct4_chip-seq
    ("SRX3288573", "SRX3288572"), # hs_ring1b_chip-seq vs. ring1b_chip-seq
    ("SRX3288575", "SRX3288574"), # hs_cfos_chip-seq vs. cfos_chip-seq
    ("SRX3288577", "SRX3288576"), # hs_brg1_chip-seq vs. brg1_chip-seq
    ("SRX3288579", "SRX3288578"), # hs_hsf1_chip-seq vs. hsf1_chip-seq
    ("SRX3288580", "SRX3288578"), # hs_ki_hsf1_chip-seq vs. hsf1_chip-seq
    ("SRX3288582", "SRX3288581"), # hs_wapl_chip-seq vs. wapl_chip-seq
    ("SRX3288584", "SRX3288583"), # hs_nipbl_chip-seq vs. nipbl_chip-seq
    ("SRX3288586", "SRX3288585"), # hs_ctcf_chip-nexus vs. ctcf_chip-nexus
    ("SRX3288588", "SRX3288587")  # hs_rad21_chip-nexus vs. rad21_chip-nexus
]

paired_metada_df_gse105028 = create_paired_metadata(
    combined_df=GSE105028_combined,
    defined_sample_pairs=defined_pairs_gse105028_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse105028 is not None:
    paired_metada_df_gse105028.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE105028_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse105028)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,condition,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy_right,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX3288555""","""SRX3288554""","""hs_igg_chip-seq vs igg_chip-seq""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6177933 vs SRR6177932""","""HS IgG ChIP-Seq vs IgG ChIP-Seq""","""GSM2816614 vs GSM2816613""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""","""1""","""H9 hESCs""","""Homo sapiens""","""9606""","""genomic DNA""","""ChIP-Seq under heat shock conditions vs ChIP-Seq under normal conditions""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""","""361102""","""China""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM2816614 vs GSM2816613""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2816nnn/GSM2816614/suppl/GSM2816614_IgG_HS.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2816nnn/GSM2816613/suppl/GSM2816613_IgG_NT.bw""","""GSE105028""","""0""","""SRX3288555 vs SRX3288554""","""H9 human Embryonic Stem Cell Line""","""under heat shock conditions vs under normal conditions""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07790038 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN07790039""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3288555 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX3288554""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""42 C""","""60 min.""","""Illumina HiSeq 2000 sequencing: GSM2816614: HS IgG ChIP-Seq Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM2816613: IgG ChIP-Seq Homo sapiens ChIP-Seq""","""SAMN07790038 vs SAMN07790039""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""SRA619794""","""ChIP""","""2018-08-04""","""412727700;426400759 vs 421179770;435277223""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/003/SRR6177933/SRR6177933_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/003/SRR6177933/SRR6177933_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/002/SRR6177932/…","""Generic""","""HS IgG ChIP-Seq vs IgG ChIP-Seq""","""023c3759b3448af5a64a4c38e9651e07;a94c834bcdaa07

&nbsp;

&nbsp;

&nbsp;

## 10. GSE112379 (ChIP-seq; Pol II; NELFE)

- [Nascent-protein ubiquitination is required for heat shock–induced gene downregulation in human cells](https://doi.org/10.1038/s41594-018-0182-x)
- Method に以下のような記載がある
- `Heat shock treatment: Unless otherwise stated, heat shock treatment was done by moving tissue-culture plates with growing cells from an incubator at normal growth temperature (37 °C) into an incubator at heat-shock temperatures (43 °C) for 1.5 h. Heat-shock treatment of K562 cells was done for 0.5 h. Heat-shock treatment of HeLa cells was done at 42 °C for the indicated times for activation and attenuation phase experiments29. To explore whether different heat-shock paradigms could influence the results obtained, we also applied acute heat shock, as described51. Briefly, two plates were grown up to 80% confluence. Medium from one of the plates was removed, preheated to 43 °C and added to the other plate from which the medium was withdrawn. The plate was then placed at 43 °C for 1.5 h.`

&nbsp;

### Cycloheximide (chx)

- Translation inhibitor

In [22]:
is_hek  = pl.col("sample_name").str.starts_with("hek_")
is_k562 = pl.col("sample_name").str.starts_with("k562_")
is_hs   = pl.col("sample_name").str.contains("_hs_")
is_nhs  = pl.col("sample_name").str.contains("_nhs_")


GSE112379_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE112379/GSE112379_PEP/GSE112379_PEP_raw.csv",
    separator=','
).filter(
    pl.col("sample_library_strategy") == "ChIP-Seq"
).with_columns(
    pl.lit("https://doi.org/10.1038/s41594-018-0182-x").alias("reference_doi"),
    # HS temperature for heat-shocked samples; NHS controls are marked as ctrl (ATAC-seq GSE101731 pattern)
    pl.when(is_hs).then(pl.lit("43 C"))
      .when(is_nhs).then(pl.lit("ctrl"))
      .otherwise(pl.lit(None))
      .alias("stress treatment temperature"),
    # Duration: HEK 1.5 h (90 min.) vs K562 0.5 h (30 min.); NHS controls are marked as ctrl
    pl.when(is_hek & is_hs).then(pl.lit("90 min."))
      .when(is_k562 & is_hs).then(pl.lit("30 min."))
      .when(is_nhs).then(pl.lit("ctrl"))
      .otherwise(pl.lit(None))
      .alias("stress treatment time"),
    # Normal growth temperature for NHS controls (Methods: 37 C incubator)
    pl.lit("37 C").alias("control treatment temperature"),
    pl.when(is_hek).then(pl.lit("HEK"))
      .when(is_k562).then(pl.lit("K562"))
      .otherwise(pl.lit(None))
      .alias("cell_line"),
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE112379 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE112379.metadata.tsv",
    separator='\t'
)

GSE112379_combined = GSE112379_PEP.join(
    GSE112379,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE112379_combined = GSE112379_combined[[s.name for s in GSE112379_combined if not (s.null_count() == GSE112379_combined.height)]]

display(GSE112379_combined)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,treatment,condition,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,cell_line,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,sra_aspera,fastq_galaxy,ncbi_reporting_standard,sample_description,sra_galaxy,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,sra_md5,fastq_ftp,base_count,library_strategy_right,library_source,sra_ftp,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""hek_chx_hs_rpb1_chip_rep1""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6904133""","""SRX3852865""","""HEK_CHX_HS_RPB1_ChIP_Rep1""","""GSM3068958""","""Public on Feb 07 2019""","""Mar 27 2018""","""Feb 07 2019""","""SRA""",1,"""Flp-In T-REx 293 Cell Line""","""Homo sapiens""",9606,"""Flp-In T-REx 293 were cultured in DMEM High Glucose supplemented with 10% FBS. K562 cells were cultured in IMDM supplemented with 10% FBS.""","""genomic DNA""","""Cells were fixed using 1% methanol-free formaldehyde (CAT no) in DMEM (HEK293) or IMDM (K562) at room temperature for 10 minutes, followed by 5 minutes blocking in 0.125M glycine. Cells were washed tw…","""ChIP-seq reads were aligned to the human genome build hg38 using Bowtie2. Duplicate and discordant reads were removed. Peak calling was done with MACS2 using ""--keep-dup all"", ""--nomodel"", ""--extsize""…","""GPL21290""","""Barbara,,Hummel""","""Stübeweg 51""","""Freiburg""",79108,"""Germany""","""Illumina HiSeq 3000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM3068958""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3068nnn/GSM3068958/suppl/GSM3068958_HEK_CHX_HS_RPB1_ChIP_Rep1.filtered.seq_depth_norm.bw""","""GSE112379""",0,"""SRX3852865""","""Cycloheximide""","""Heat-Shock""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN08800999""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3852865""","""https://doi.org/10.1038/s41594-018-0182-x""","""43 C""","""90 min.""","""37 C""","""HEK""","""Illumina HiSeq 3000 sequencing: GSM3068958: HEK_CHX_HS_RPB1_ChIP_Rep1 Homo sapiens ChIP-Seq""","""SAMN08800999""","""Nascent-protein ubiquitination is required for heat shock-induced gene downregulation in human cells.""","""SRA675427""","""ChIP""","""2019-02-14""",5124166039,"""2958529780;3766752113""","""ILLUMINA""","""ARCHIVAL_FILE""","""fasp.sra.ebi.ac.uk:/vol1/srr/SRR690/003/SRR6904133""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR690/003/SRR6904133/SRR6904133_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR690/003/SRR6904133/SRR6904133_2.fastq.gz""","""Generic""","""HEK_CHX_HS_RPB1_ChIP_Rep1""","""ftp.sra.ebi.ac.uk/vol1/srr/S

In [23]:
defined_pairs_gse112379_chipseq = [

    # pair18_GSE112379_pol2_hs_43C_90min_hek_rpb1.txt
    # pair18_GSE112379_pol2_ctrl_hek_rpb1_chx.txt
    ("SRX3852865", "SRX3852867"), # hek_chx_hs_rpb1_chip_rep1 vs. hek_chx_nhs_rpb1_chip_rep1
    ("SRX3852866", "SRX3852868"), # hek_chx_hs_rpb1_chip_rep2 vs. hek_chx_nhs_rpb1_chip_rep2

    ("SRX3852869", "SRX3852870"), # hek_hs_input vs. hek_nhs_input

    # pair19_GSE112379_pol2_hs_43C_90min_hek_rpb1_scramble.txt
    # pair19_GSE112379_pol2_ctrl_hek_rpb1_scramble.txt
    ("SRX3852871", "SRX3852874"), # hek_scramble_hs_rpb1_chip_rep1 vs. hek_scramble_nhs_rpb1_chip_rep1
    ("SRX3852872", "SRX3852875"), # hek_scramble_hs_rpb1_chip_rep2 vs. hek_scramble_nhs_rpb1_chip_rep2
    ("SRX3852873", "SRX3852876"), # hek_scramble_hs_rpb1_chip_rep3 vs. hek_scramble_nhs_rpb1_chip_rep3

    # sinelfe: silencing NELFE
    # pair20_GSE112379_pol2_hs_43C_90min_hek_rpb1_sinelfe.txt
    # pair20_GSE112379_pol2_ctrl_hek_rpb1_sinelfe.txt
    ("SRX3852877", "SRX3852879"), # hek_sinelfe_hs_rpb1_chip_rep1 vs. hek_sinelfe_nhs_rpb1_chip_rep1
    ("SRX3852878", "SRX3852880"), # hek_sinelfe_hs_rpb1_chip_rep2 vs. hek_sinelfe_nhs_rpb1_chip_rep2


    # ub-e1: Ubiquitin-activating enzyme E1
    # pair21_GSE112379_pol2_hs_43C_90min_hek_rpb1_ub-e1.txt
    # pair21_GSE112379_pol2_ctrl_hek_rpb1_ub-e1.txt
    ("SRX3852881", "SRX3852883"), # hek_ub-e1_hs_rpb1_chip_rep1 vs. hek_ub-e1_nhs_rpb1_chip_rep1
    ("SRX3852882", "SRX3852884"), # hek_ub-e1_hs_rpb1_chip_rep2 vs. hek_ub-e1_nhs_rpb1_chip_rep2


    # vehicle: vehicle control?
    # pair22_GSE112379_pol2_hs_43C_90min_hek_rpb1_vehicle.txt
    # pair22_GSE112379_pol2_ctrl_hek_rpb1_vehicle.txt
    ("SRX3852885", "SRX3852887"), # hek_vehicle_hs_rpb1_chip_rep1 vs. hek_vehicle_nhs_rpb1_chip_rep1
    ("SRX3852886", "SRX3852888"), # hek_vehicle_hs_rpb1_chip_rep2 vs. hek_vehicle_nhs_rpb1_chip_rep2


    ("SRX3852889", "SRX3852896"), # k562_hs_input vs. k562_nhs_input
    ("SRX4643714", "SRX4643712"), # k562_hs_mapk14_chip_rep1 vs. k562_nhs_mapk14_chip_rep1
    ("SRX4643715", "SRX4643713"), # k562_hs_mapk14_chip_rep2 vs. k562_nhs_mapk14_chip_rep2
    ("SRX3852895", "SRX3852897"), # k562_nelfa_hs_chip_rep1 vs. k562_nhs_nelfa_chip_rep1
    ("SRX3852890", "SRX3852898"), # k562_hs_nelfa_chip_rep2 vs. k562_nhs_nelfa_chip_rep2
    ("SRX3852891", "SRX3852899"), # k562_hs_nelfe_chip_rep1 vs. k562_nhs_nelfe_chip_rep1
    ("SRX3852892", "SRX3852900"), # k562_hs_nelfe_chip_rep2 vs. k562_nhs_nelfe_chip_rep2


    # pair23_GSE112379_pol2_hs_43C_30min_k562_rpb1.txt
    # pair23_GSE112379_pol2_ctrl_k562_rpb1.txt
    ("SRX3852893", "SRX3852901"), # k562_hs_rpb1_chip_rep1 vs. k562_nhs_rpb1_chip_rep1
    ("SRX3852894", "SRX3852902")  # k562_hs_rpb1_chip_rep2 vs. k562_nhs_rpb1_chip_rep2
]


paired_metada_df_gse112379 = create_paired_metadata(
    combined_df=GSE112379_combined,
    defined_sample_pairs=defined_pairs_gse112379_chipseq,
    sample_id_column_name="experiment_accession"
)


if paired_metada_df_gse112379 is not None:
    # Keep only heat-side values after pairing (e.g. "90 min. vs ctrl" -> "90 min.")
    paired_metada_df_gse112379 = paired_metada_df_gse112379.with_columns(
        pl.col(["stress treatment temperature", "stress treatment time"])
        .str.strip_suffix(" vs ctrl") # https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.strip_suffix.html
    )
    paired_metada_df_gse112379.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE112379_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse112379)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,treatment,condition,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,cell_line,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_file_role,sra_aspera,fastq_galaxy,ncbi_reporting_standard,sample_description,sra_galaxy,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,sra_md5,fastq_ftp,base_count,library_strategy_right,library_source,sra_ftp,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX3852865""","""SRX3852867""","""hek_chx_hs_rpb1_chip_rep1 vs hek_chx_nhs_rpb1_chip_rep1""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6904133 vs SRR6904135""","""HEK_CHX_HS_RPB1_ChIP_Rep1 vs HEK_CHX_NHS_RPB1_ChIP_Rep1""","""GSM3068958 vs GSM3068960""","""Public on Feb 07 2019""","""Mar 27 2018""","""Feb 07 2019""","""SRA""","""1""","""Flp-In T-REx 293 Cell Line""","""Homo sapiens""","""9606""","""Flp-In T-REx 293 were cultured in DMEM High Glucose supplemented with 10% FBS. K562 cells were cultured in IMDM supplemented with 10% FBS.""","""genomic DNA""","""Cells were fixed using 1% methanol-free formaldehyde (CAT no) in DMEM (HEK293) or IMDM (K562) at room temperature for 10 minutes, followed by 5 minutes blocking in 0.125M glycine. Cells were washed tw…","""ChIP-seq reads were aligned to the human genome build hg38 using Bowtie2. Duplicate and discordant reads were removed. Peak calling was done with MACS2 using ""--keep-dup all"", ""--nomodel"", ""--extsize""…","""GPL21290""","""Barbara,,Hummel""","""Stübeweg 51""","""Freiburg""","""79108""","""Germany""","""Illumina HiSeq 3000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM3068958 vs GSM3068960""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3068nnn/GSM3068958/suppl/GSM3068958_HEK_CHX_HS_RPB1_ChIP_Rep1.filtered.seq_depth_norm.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3068nnn/GSM3068960/suppl…","""GSE112379""","""0""","""SRX3852865 vs SRX3852867""","""Cycloheximide""","""Heat-Shock vs Non Heat-Shock""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN08800999 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN08800996""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3852865 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX3852867""","""https://doi.org/10.1038/s41594-018-0182-x""","""43 C""","""90 min.""","""37 C""","""HEK""","""Illumina HiSeq 3000 sequencing: GSM3068958: HEK_CHX_HS_RPB1_ChIP_Rep1 Homo sapiens ChIP-Seq vs Illumina HiSeq 3000 sequencing: GSM3068960: HEK_CHX_NHS_RPB1_ChIP_Rep1 Homo sapiens ChIP-Seq""","""SAMN08800999 vs SAMN08800996""","""Nascent-protein ubiquitination

&nbsp;

&nbsp;

&nbsp;

## 11. GSE121182 (ChIP-seq; CBX8/CBX2)

- [Protein quality control in the nucleolus safeguards recovery of epigenetic regulators after heat shock](https://doi.org/10.7554/eLife.45205)
- 

In [24]:
GSE121182_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE121182/GSE121182_PEP/GSE121182_PEP_raw.csv",
    separator=','
).filter(
    pl.col("sample_library_strategy") == "ChIP-Seq"
).with_columns(
    pl.lit("https://doi.org/10.7554/eLife.45205").alias("reference_doi"),
    # GEO treatment protocol: heat shocked for 1 hour at 44 C
    pl.when(pl.col("sample_name").str.ends_with("-hs"))
      .then(pl.lit("44 C"))
      .otherwise(pl.lit("ctrl"))
      .alias("stress treatment temperature"),
    pl.when(pl.col("sample_name").str.ends_with("-hs"))
      .then(pl.lit("60 min."))
      .otherwise(pl.lit("ctrl"))
      .alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE121182 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE121182.metadata.tsv",
    separator='\t'
)

GSE121182_combined = GSE121182_PEP.join(
    GSE121182,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE121182_combined = GSE121182_combined[[s.name for s in GSE121182_combined if not (s.null_count() == GSE121182_combined.height)]]

pl.Config.set_tbl_rows(4)
display(GSE121182_combined)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_characteristics_ch1,sample_treatment_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,antibody,treatment,cell_line,genome_build,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""cbx8-untreated""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR8042967""","""SRX4872742""","""CBX8-untreated""","""GSM3427438""","""Public on May 31 2019""","""Oct 12 2018""","""Nov 11 2021""","""SRA""",1,"""K562""","""Homo sapiens""",9606,"""sample collection (day): 1""","""Cells were heat shocked for 1 hour at 44C""","""genomic DNA""","""GPL18573""","""Joost,,Martens""","""joost.martens@ru.nl""",243780645,"""Molecular Biology""","""RIMLS""","""Radboud University""","""Geert Grooteplein 28""","""Nijmegen""","""Nederland""","""6525GA""","""Netherlands""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM3427438""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3427nnn/GSM3427438/suppl/GSM3427438_CBX8-untreated-ChIP-13690.bam.filtered.bam.bw""","""GSE121182""",0,"""SRX4872742""","""anti-CBX8 (C15410333, Diagenode)""","""untreated""","""K562""","""hg19""","""bigwig (bw) files""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN10236330""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX4872742""","""https://doi.org/10.7554/eLife.45205""","""ctrl""","""ctrl""","""37 C (inferred)""","""NextSeq 500 sequencing: GSM3427438: CBX8-untreated Homo sapiens ChIP-Seq""","""SAMN10236330""","""Dynamic behavior of the epigenetic machinery upon thermal stress""","""SRA793600""","""ChIP""","""2019-06-01""","""127287103;132749883""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR804/007/SRR8042967/SRR8042967_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR804/007/SRR8042967/SRR8042967_2.fastq.gz""","""Generic""","""CBX8-untreated""","""81e036dc6c9907c99a5f61849e4617c4;005a9c742ae5039440e7a0f1a8af3728""","""SRP165293""",4910444,"""Dynamic behavior of the epigenetic machinery upon thermal stress""","""GENERATED_FILE;GENERATED_FILE""","""CBX8-untreated""","""NextSeq 500""","""NextSeq 500 sequencing: GSM3427438: CBX8-untreated Homo sapiens ChIP-Seq""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR804/007/SRR8042967/SRR8042967_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR804/0

In [25]:
defined_pairs_gse121182_chipseq = [
    ("SRX4872743", "SRX4872742"), # cbx8-hs vs. cbx8-untreated
    ("SRX4872745", "SRX4872744")  # gfp-cbx2-hs vs. gfp-cbx2-untreated
]

paired_metada_df_gse121182 = create_paired_metadata(
    combined_df=GSE121182_combined,
    defined_sample_pairs=defined_pairs_gse121182_chipseq,
    sample_id_column_name="experiment_accession"
)


if paired_metada_df_gse121182 is not None:
    paired_metada_df_gse121182 = paired_metada_df_gse121182.with_columns(
        pl.col(["stress treatment temperature", "stress treatment time"])
        .str.strip_suffix(" vs ctrl")
    )
    paired_metada_df_gse121182.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE121182_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse121182)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_characteristics_ch1,sample_treatment_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,antibody,treatment,cell_line,genome_build,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX4872743""","""SRX4872742""","""cbx8-hs vs cbx8-untreated""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR8042968 vs SRR8042967""","""CBX8-HS vs CBX8-untreated""","""GSM3427439 vs GSM3427438""","""Public on May 31 2019""","""Oct 12 2018""","""Nov 11 2021""","""SRA""","""1""","""K562""","""Homo sapiens""","""9606""","""sample collection (day): 1""","""Cells were heat shocked for 1 hour at 44C""","""genomic DNA""","""GPL18573""","""Joost,,Martens""","""joost.martens@ru.nl""","""243780645""","""Molecular Biology""","""RIMLS""","""Radboud University""","""Geert Grooteplein 28""","""Nijmegen""","""Nederland""","""6525GA""","""Netherlands""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM3427439 vs GSM3427438""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3427nnn/GSM3427439/suppl/GSM3427439_CBX8-HS-ChIP-13691.bam.filtered.bam.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3427nnn/GSM3427438/suppl/GSM3427438_CB…","""GSE121182""","""0""","""SRX4872743 vs SRX4872742""","""anti-CBX8 (C15410333, Diagenode)""","""1h, 44C vs untreated""","""K562""","""hg19""","""bigwig (bw) files""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN10236329 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN10236330""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX4872743 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX4872742""","""https://doi.org/10.7554/eLife.45205""","""44 C""","""60 min.""","""37 C (inferred)""","""NextSeq 500 sequencing: GSM3427439: CBX8-HS Homo sapiens ChIP-Seq vs NextSeq 500 sequencing: GSM3427438: CBX8-untreated Homo sapiens ChIP-Seq""","""SAMN10236329 vs SAMN10236330""","""Dynamic behavior of the epigenetic machinery upon thermal stress""","""SRA793600""","""ChIP""","""2019-06-01""","""113250766;119529604 vs 127287103;132749883""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR804/008/SRR8042968/SRR8042968_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR804/008/SRR8042968/SRR8042968_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/

&nbsp;

&nbsp;

&nbsp;

## 12. GSE137558 (ChIP-seq; HSF1)

- [17β-Estradiol Activates HSF1 via MAPK Signaling in ERα-Positive Breast Cancer Cells](https://doi.org/10.3390/cancers11101533)
- 

In [26]:
GSE137558_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE137558/GSE137558_PEP/GSE137558_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (~pl.col("sample_name").str.contains("estrogen")) &
    (~pl.col("sample_name").str.contains("_input_"))
).with_columns(
    pl.lit("https://doi.org/10.3390/cancers11101533").alias("reference_doi"),
    pl.lit("43 C").alias("stress treatment temperature"),
    pl.lit("15 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE137558 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE137558.metadata.tsv",
    separator='\t'
)

GSE137558_combined = GSE137558_PEP.join(
    GSE137558,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE137558_combined = GSE137558_combined[[s.name for s in GSE137558_combined if not (s.null_count() == GSE137558_combined.height)]]

pl.Config.set_tbl_rows(5)
display(GSE137558_combined)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,tumor,sex,age,tissue_cells,chip_antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""chip-seq_control""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR10129834""","""SRX6858247""","""ChIP-seq_control""","""GSM4081759""","""Public on Oct 05 2019""","""Sep 17 2019""","""Oct 05 2019""","""SRA""",1,"""untreated control""","""Homo sapiens""",9606,"""genomic DNA""","""untreated control""","""GPL18460""","""Tomasz,,Stokowy""","""tomasz.stokowy@uib.no""","""Scientific Computing""","""IT Division""","""University of Bergen""","""Nygardsgaten 5""","""Bergen""",5020,"""Norway""","""Illumina HiSeq 1500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4081759""","""NONE""","""GSE137558, GSE137560""",0,"""SRX6858247""","""MCF7""","""adenocarcinoma""","""female""","""69 years""","""breast""","""HSF1 polyclonal antibody (Enzo, ADI-SPA-901)""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN12771835""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX6858247""","""https://doi.org/10.3390/cancers11101533""","""43 C""","""15 min.""","""Illumina HiSeq 1500 sequencing: GSM4081759: ChIP-seq_control Homo sapiens ChIP-Seq""","""SAMN12771835""","""Heat Shock Factor 1 (HSF1) acquires transcriptional competence under 17b-estradiol in ERa-positive breast cancer cells [ChIP-seq]""","""SRA963112""","""ChIP""","""2019-10-07""",1551550759,"""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR101/034/SRR10129834/SRR10129834.fastq.gz""","""Generic""","""ChIP-seq_control""","""bad77ec99df74ad60239e89ae9045325""","""SRP221988""",40358792,"""Heat Shock Factor 1 (HSF1) acquires transcriptional competence under 17b-estradiol in ERa-positive breast cancer cells [ChIP-seq]""","""GENERATED_FILE""","""ChIP-seq_control""","""Illumina HiSeq 1500""","""Illumina HiSeq 1500 sequencing: GSM4081759: ChIP-seq_control Homo sapiens ChIP-Seq""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR101/034/SRR10129834/SRR10129834.fastq.gz""",2603418815,"""ChIP-Seq""","""GENOMIC""","""SINGLE""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR101/034/SRR10129834/SRR10129834.fastq.gz""","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;7776;117570;117571;8287;1338369;32523;32524;40674;32525;9347;1437010;314146;9443;376913;3

In [27]:
defined_pairs_gse137558_chipseq = [
    ("SRX6858250", "SRX6858247")  # chip-seq_hyperthermia_15min vs. chip-seq_control
]

paired_metada_df_gse137558 = create_paired_metadata(
    combined_df=GSE137558_combined,
    defined_sample_pairs=defined_pairs_gse137558_chipseq,
    sample_id_column_name="experiment_accession"
)


if paired_metada_df_gse137558 is not None:
    paired_metada_df_gse137558.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE137558_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse137558)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,tumor,sex,age,tissue_cells,chip_antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX6858250""","""SRX6858247""","""chip-seq_hyperthermia_15min vs chip-seq_control""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR10129837 vs SRR10129834""","""ChIP-seq_hyperthermia_15min vs ChIP-seq_control""","""GSM4081762 vs GSM4081759""","""Public on Oct 05 2019""","""Sep 17 2019""","""Oct 05 2019""","""SRA""","""1""","""subjected to hyperthermia (43°C) for 15minutes vs untreated control""","""Homo sapiens""","""9606""","""genomic DNA""","""subjected to hyperthermia (43°C) for 15minutes vs untreated control""","""GPL18460""","""Tomasz,,Stokowy""","""tomasz.stokowy@uib.no""","""Scientific Computing""","""IT Division""","""University of Bergen""","""Nygardsgaten 5""","""Bergen""","""5020""","""Norway""","""Illumina HiSeq 1500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4081762 vs GSM4081759""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4081nnn/GSM4081762/suppl/GSM4081762_HSF1_HS_K_annotatedPeaks.txt.gz vs NONE""","""GSE137558, GSE137560""","""0""","""SRX6858250 vs SRX6858247""","""MCF7""","""adenocarcinoma""","""female""","""69 years""","""breast""","""HSF1 polyclonal antibody (Enzo, ADI-SPA-901)""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN12771833 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN12771835""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX6858250 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX6858247""","""https://doi.org/10.3390/cancers11101533""","""43 C""","""15 min.""","""Illumina HiSeq 1500 sequencing: GSM4081762: ChIP-seq_hyperthermia_15min Homo sapiens ChIP-Seq vs Illumina HiSeq 1500 sequencing: GSM4081759: ChIP-seq_control Homo sapiens ChIP-Seq""","""SAMN12771833 vs SAMN12771835""","""Heat Shock Factor 1 (HSF1) acquires transcriptional competence under 17b-estradiol in ERa-positive breast cancer cells [ChIP-seq]""","""SRA963112""","""ChIP""","""2019-10-07""","""1760645526 vs 1551550759""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR101/037/SRR10129837/SRR10129837.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR101/034/SRR10129834/SRR10129834.fastq.gz""","""Generic""","""ChIP-seq_hyperthermia_15min vs ChIP-seq_control""",

&nbsp;

&nbsp;

&nbsp;

## 13. GSE152144 (ChIP-seq; HSF1/HSF2)

- [A ChIP-exo screen of 887 Protein Capture Reagents Program transcription factor antibodies in human cells](https://doi.org/10.1101/gr.275472.121)
- HSF1のデータのみ、熱ストレス条件下のサンプルを使用していると考えられるため、このデータを選抜、利用
- 以下のような記述から判断
- `ChIP assessment in multiple cell states and types: Any number of targets may be sequestered in a state that prevents their interaction with chromatin (and thus detection by ChIP) unless activated to do so through a change in cell state. We examined this with HSF1, which is rapidly induced to bind in the nucleus upon heat shock to activate heat shock–response genes (Baler et al. 1993). HSF1 was bound to cognate motifs at relatively low levels under nonstressed conditions but increased in binding upon treatment of cells with hydrogen peroxide (0.3 mM) for 3 min and 30 min (assayed by ChIP-exo) (Supplemental Fig. 5A) or upon heat shock (shift from 37°C to 42°C for 1 h; assayed by ChIP-seq) (Supplemental Fig. 5B). This illustrates a potential problem with using mRNA expression levels as a basis for expecting a factor to be actively bound to chromatin. Many TFs are sequestered and only bind chromatin when released by signaling events.`
- 一方で、HSF2のデータも含まれており、`treatment: One hour heat shock`とあるので、こちらも収集する
- Input control データにおいても、熱ストレス条件下 vs. コントロールのデータ が存在しているため、偽陽性排除に使える可能性があると判断し、収集
- 抗体を作成する ハイブリドーマ細胞のクローン 「1A10」、「1A8」など、クローンごとに実行されている
- replicate が2つ以上ないため、ChIP-Atlas Diff analysis には使用しない

In [28]:
GSE152144_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE152144/GSE152144_PEP/GSE152144_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("organism") == "Homo sapiens") &
    (
        (pl.col("sample_name").str.contains("hsf1_")) |
        (pl.col("sample_name").str.contains("hsf2_"))
    )
).with_columns(
    pl.lit("https://doi.org/10.1101/gr.275472.121").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"), # may be 42 C because of the paper description
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE152144 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE152144.metadata.tsv",
    separator='\t'
)

GSE152144_combined = GSE152144_PEP.join(
    GSE152144,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE152144_combined = GSE152144_combined[[s.name for s in GSE152144_combined if not (s.null_count() == GSE152144_combined.height)]]

pl.Config.set_tbl_rows(2)
display(GSE152144_combined)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,cell_type,antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""hsf1_1a10_37""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR11968993""","""SRX8512773""","""HSF1_1A10_37""","""GSM4604715""","""Public on Sep 01 2020""","""Jun 09 2020""","""Sep 02 2020""","""SRA""",1,"""HCT116""","""Homo sapiens""",9606,"""genomic DNA""","""GPL18573""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""",60611,"""USA""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4604715""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4604nnn/GSM4604715/suppl/GSM4604715_HSF1_1A10_37.bw""","""GSE152144""",0,"""SRX8512773""","""HCT116""","""colorectal carcinoma""","""HSF1_1A10""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN15192250""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX8512773""","""https://doi.org/10.1101/gr.275472.121""","""42 C""","""60 min.""","""NextSeq 500 sequencing: GSM4604715: HSF1_1A10_37 Homo sapiens ChIP-Seq""","""SAMN15192250""","""Screening of PCRP transcription 1 factor antibodies in biochemical assays""","""SRA1085187""","""ChIP""","""2020-09-02""",1400025739,"""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR119/093/SRR11968993/SRR11968993.fastq.gz""","""Generic""","""HSF1_1A10_37""","""5891e06100a2b9939ccc175e59495770""","""SRP266663""",43356617,"""Screening of PCRP transcription 1 factor antibodies in biochemical assays""","""GENERATED_FILE""","""HSF1_1A10_37""","""NextSeq 500""","""NextSeq 500 sequencing: GSM4604715: HSF1_1A10_37 Homo sapiens ChIP-Seq""","""colorectal carcinoma""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR119/093/SRR11968993/SRR11968993.fastq.gz""",2150002216,"""ChIP-Seq""","""GENOMIC""","""SINGLE""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR119/093/SRR11968993/SRR11968993.fastq.gz""","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;7776;117570;117571;8287;1338369;32523;32524;40674;32525;9347;1437010;314146;9443;376913;314293;9526;314295;9604;207598;9605;9606""","""GSM4604715""","""GSM4604715_r1""","""1x10^8 cells were fixed in 1% formaldehyde (Sigma, 252549) in PBS for 15- 20 min at room temperature and 

In [29]:
defined_pairs_gse152144_chipseq = [


    ("SRX8512774", "SRX8512773"), # HSF1_1A10_43 vs. HSF1_1A10_37
    ("SRX8512776", "SRX8512775"), # HSF1_1A6_43 vs. HSF1_1A6_37
    ("SRX8512778", "SRX8512777"), # HSF1_1A8_43 vs. HSF1_1A8_37
    ("SRX8512780", "SRX8512779"), # HSF1_1A9_43 vs. HSF1_1A9_37
    ("SRX8512782", "SRX8512781"), # HSF1_1C1_43 vs. HSF1_1C1_37
    ("SRX8512784", "SRX8512783"), # HSF1_1D11_43 vs. HSF1_1D11_37

    ("SRX8512786", "SRX8512785"), # HSF2_2C1_43 vs. HSF2_2C1_37
    ("SRX8512890", "SRX8512889"), # Input_HSF1_1A10_HSF1_1A8_HSF1_1A9_HSF1_1D11_HSF2_2C1_43 vs. Input_HSF1_1A10_HSF1_1A8_HSF1_1A9_HSF1_1D11_HSF2_2C1_37
    ("SRX8512724", "SRX8512891")  # Input_HSF1_1A6_HSF1_1C1_43 vs. Input_HSF1_1A6_HSF1_1C1_37
]

paired_metada_df_gse152144 = create_paired_metadata(
    combined_df=GSE152144_combined,
    defined_sample_pairs=defined_pairs_gse152144_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse152144 is not None:
    paired_metada_df_gse152144.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE152144_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse152144)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,cell_type,antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX8512774""","""SRX8512773""","""hsf1_1a10_43 vs hsf1_1a10_37""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR11968994 vs SRR11968993""","""HSF1_1A10_43 vs HSF1_1A10_37""","""GSM4604716 vs GSM4604715""","""Public on Sep 01 2020""","""Jun 09 2020""","""Sep 02 2020""","""SRA""","""1""","""HCT116""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL18573""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4604716 vs GSM4604715""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4604nnn/GSM4604716/suppl/GSM4604716_HSF1_1A10_43.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4604nnn/GSM4604715/suppl/GSM4604715_HSF1_1A10_37.bw""","""GSE152144""","""0""","""SRX8512774 vs SRX8512773""","""HCT116""","""colorectal carcinoma""","""HSF1_1A10""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN15192249 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN15192250""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX8512774 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX8512773""","""https://doi.org/10.1101/gr.275472.121""","""42 C""","""60 min.""","""NextSeq 500 sequencing: GSM4604716: HSF1_1A10_43 Homo sapiens ChIP-Seq vs NextSeq 500 sequencing: GSM4604715: HSF1_1A10_37 Homo sapiens ChIP-Seq""","""SAMN15192249 vs SAMN15192250""","""Screening of PCRP transcription 1 factor antibodies in biochemical assays""","""SRA1085187""","""ChIP""","""2020-09-02""","""830654417 vs 1400025739""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR119/094/SRR11968994/SRR11968994.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR119/093/SRR11968993/SRR11968993.fastq.gz""","""Generic""","""HSF1_1A10_43 vs HSF1_1A10_37""","""7c3b477320a09515a92cb4d3c28990dd vs 5891e06100a2b9939ccc175e59495770""","""SRP266663""","""28136781 vs 43356617""","""Screening of PCRP transcription 1 factor antibodies in biochemical assays""","""GENERATED_FILE""","""HSF1_1A10_43 vs HSF1_1A10_37""","""NextSeq 500""","""NextSeq 500 sequencing: GSM4604716: HSF1_1A10

&nbsp;

&nbsp;

&nbsp;

## 14. GSE143591 (ChIP-seq; PolII/SETD1A/MRE11A)

- [Chromatin-associated MRN complex protects highly transcribing genes from genomic instability](https://doi.org/10.1126/sciadv.abb2947)
- 以下のような記述を発見
- `MATERIALS AND METHODS:Cell culture, treatment, and lentiviral infection: ... All cells were grown in a humidified incubator at 37°C with 5% CO2. Where indicated, cells were subjected to heat shock at 42°C for 1 hour, and the medium was replaced before fixation. `

In [30]:
GSE143591_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE143591/GSE143591_PEP/GSE143591_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("organism") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1126/sciadv.abb2947").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE143591 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE143591.metadata.tsv",
    separator='\t'
)

GSE143591_combined = GSE143591_PEP.join(
    GSE143591,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE143591_combined = GSE143591_combined[[s.name for s in GSE143591_combined if not (s.null_count() == GSE143591_combined.height)]]

# display(GSE143591_combined)

In [31]:
defined_pairs_gse143591_chipseq = [
    ("SRX7551111", "SRX7551108"), # Zd4_Heatshock_NBS1 vs. Zd1_Untreated_NBS1
    ("SRX7551112", "SRX7551109"), # Zd5_Heatshock_MRE11 vs. Zd2_Untreated_MRE11
    ("SRX7551113", "SRX7551110")  # Zd6_Heatshock_Pol2 vs. Zd3_Untreated_Pol2 (replicate が 2つ以上ないので、diff analysis では除外)
]

paired_metada_df_gse143591 = create_paired_metadata(
    combined_df=GSE143591_combined,
    defined_sample_pairs=defined_pairs_gse143591_chipseq,
    sample_id_column_name="experiment_accession"
)


if paired_metada_df_gse143591 is not None:
    paired_metada_df_gse143591.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE143591_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(3)
    display(paired_metada_df_gse143591)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,treatment,passages,chip_antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX7551111""","""SRX7551108""","""zd4_heatshock_nbs1 vs zd1_untreated_nbs1""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR10882047 vs SRR10882044""","""Zd4_Heatshock_NBS1 vs Zd1_Untreated_NBS1""","""GSM4263566 vs GSM4263563""","""Public on Jan 13 2021""","""Jan 13 2020""","""Jan 14 2021""","""SRA""","""1""","""HeLa cell""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL18573""","""Rosemary,,Kiernan""","""Gene Regulation lab""","""UMR9002 CNRS-UM""","""141 rue de la cardonille""","""Montpellier""","""34396""","""France""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM4263566 vs GSM4263563""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4263nnn/GSM4263566/suppl/GSM4263566_Zd4_HS_NBS1.RPGC.sinorm.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4263nnn/GSM4263563/suppl/GSM4263563_Zd1_Ut_NBS1.RP…","""GSE143591""","""0""","""SRX7551111 vs SRX7551108""","""heat shock vs None""","""13-15""","""Bethyl A300-289A""","""HG38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN13839657 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN13839660""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX7551111 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX7551108""","""https://doi.org/10.1126/sciadv.abb2947""","""42 C""","""60 min.""","""NextSeq 500 sequencing: GSM4263566: Zd4_Heatshock_NBS1 Homo sapiens ChIP-Seq vs NextSeq 500 sequencing: GSM4263563: Zd1_Untreated_NBS1 Homo sapiens ChIP-Seq""","""SAMN13839657 vs SAMN13839660""","""Genome-wide maps of MRE11, NBS1 total RNAPII and phospho-Ser2-RNAPII in WT cells, upon shControl, shMRE11 or shNBS1 treatment, after treatment by DRB and after heat shock in HeLa cells""","""SRA1025671""","""ChIP""","""2021-01-14""","""5640930252 vs 1970627341""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR108/047/SRR10882047/SRR10882047.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR108/044/SRR10882044/SRR10882044.fastq.gz""","""Generic""","""Zd4_Heatshock_NBS1 vs Zd1_Untreated_NBS1""","""c0f91da64d613b1697bf35115661a7e5 vs 771ec837c3d9198b89d40631d21f0599""","""SRP241821""","""116300616 vs 40506803""","""Genome-wide maps of MRE11, NBS1 total RNAPII and phospho-Ser2-RNAPII in WT cells, upon shControl, shMRE11 or shNBS1 treatment, after treatment by DRB and after

&nbsp;

&nbsp;

&nbsp;

## 15. GSE194364 (ChIP-seq; HSF1)

- [The stress response regulator HSF1 modulates natural killer cell anti-tumour immunity](https://doi.org/10.1038/s41556-024-01490-z)
- [GSM8289675](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8289675)
- replicate が2つ以上ないため、ChIP-Atlas Diff analysis には使用しない

In [32]:
GSE194364_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE194364/GSE194364_PEP/GSE194364_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1038/s41556-024-01490-z").alias("reference_doi"),
    # GSM8289675 Treatment protocol: heat shocked at 42 C for 1 hour
    pl.when(pl.col("sample_name").str.contains("(?i)_hs_")).then(pl.lit("42 C"))
      .otherwise(pl.lit("ctrl"))
      .alias("stress treatment temperature"),
    pl.when(pl.col("sample_name").str.contains("(?i)_hs_")).then(pl.lit("60 min."))
      .otherwise(pl.lit("ctrl"))
      .alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature")
).rename(
    {
        "big_key": "experiment_accession"
    }
)

GSE194364 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE194364.metadata.tsv",
    separator='\t'
)

GSE194364_combined = GSE194364_PEP.join(
    GSE194364,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE194364_combined = GSE194364_combined[[s.name for s in GSE194364_combined if not (s.null_count() == GSE194364_combined.height)]]

display(GSE194364_combined)

sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,sample_series_id,sample_data_row_count,experiment_accession,antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""nk92_hsf1_ip""","""NK92_HSF1_IP""","""GSM5834091""","""Public on Jan 26 2022""","""Jan 25 2022""","""Jan 26 2022""","""SRA""",1,"""NK-92 MI cells""","""Homo sapiens""",9606,"""genomic DNA""","""Reads were aligned to the Mus musculus genome (mm10/GRCm38) using the Bowtie2 aligner (Langmead et al., 2009) (v2.3.4.1)., Duplicate reads were removed using sambamba markdup., Peaks were determined a…","""GPL24676""","""Kathryn,Grace,Hockemeyer""","""Aifantis""","""Pathology""","""NYU Langone Health""","""550 1st Avenue""","""New York""","""New York""",10016,"""USA""","""Illumina NovaSeq 6000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM5834091""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5834nnn/GSM5834091/suppl/GSM5834091_NK92_HSF1_IP.peaks.bed.gz""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5834nnn/GSM5834091/suppl/GSM5834091_NK92_HSF1_IP.peaks.tsv.gz""","""GSE167552, GSE194364""",0,"""SRX13898238""","""HSF1""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN25243265""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX13898238""","""https://doi.org/10.1038/s41556-024-01490-z""","""ctrl""","""ctrl""","""37 C (inferred)""","""SRR17735673""","""Illumina NovaSeq 6000 sequencing: GSM5834091: NK92_HSF1_IP Homo sapiens ChIP-Seq""","""SAMN25243265""","""The NK cell stress response status modulates anti-tumor immunity [ChIP-seq]""","""SRA1361651""","""ChIP""","""2022-01-27""","""1914225964;1820935624""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR177/073/SRR17735673/SRR17735673_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR177/073/SRR17735673/SRR17735673_2.fastq.gz""",null,"""Generic""","""NK92_HSF1_IP""","""5e295be447260ec662e2283cc2c182f9;5783b88cafa4f0ef986ef0eea62c7862""","""SRP356652""",91022449,"""The NK cell stress response status modulates anti-tumor immunity [ChIP-seq]""","""GENERATED_FILE;GENERATED_FILE""","""NK92_HSF1_IP""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM5834091: NK92_HSF1_IP Homo sapiens ChIP-Seq""",null,"""ftp.sra.ebi.ac.uk/vol1/fastq/SRR177/073/SRR17735673/SRR17735673_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR177/073/SRR17735673/SRR17735673_2.fastq.gz""",9284289798,"""ChIP-Seq

In [33]:
defined_pairs_gse194364_chipseq = [
    ("SRX24692877", "SRX13898238") # nk92_hs_hsf1_ip_2 vs. nk92_hsf1_ip
]

paired_metada_df_gse194364 = create_paired_metadata(
    combined_df=GSE194364_combined,
    defined_sample_pairs=defined_pairs_gse194364_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse194364 is not None:
    paired_metada_df_gse194364 = paired_metada_df_gse194364.with_columns(
        pl.col(["stress treatment temperature", "stress treatment time"])
        .str.strip_suffix(" vs ctrl")
    )
    paired_metada_df_gse194364.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE194364_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse194364)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,sample_series_id,sample_data_row_count,antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX24692877""","""SRX13898238""","""nk92_hs_hsf1_ip_2 vs nk92_hsf1_ip""","""NK92_HS_HSF1_IP_2 vs NK92_HSF1_IP""","""GSM8289675 vs GSM5834091""","""Public on May 28 2024 vs Public on Jan 26 2022""","""May 24 2024 vs Jan 25 2022""","""May 28 2024 vs Jan 26 2022""","""SRA""","""1""","""NK-92 MI cells""","""Homo sapiens""","""9606""","""genomic DNA""","""Reads were aligned to the Homo Sapiens genome using the Bowtie2 aligner (Langmead et al., 2009) (v2.3.4.1)., Duplicate reads were removed using sambamba markdup., Peaks were determined algorithmically…","""GPL24676""","""Kathryn,Grace,Hockemeyer""","""Aifantis""","""Pathology""","""NYU Langone Health""","""550 1st Avenue""","""New York""","""New York""","""10016""","""USA""","""Illumina NovaSeq 6000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8289675 vs GSM5834091""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8289nnn/GSM8289675/suppl/GSM8289675_NK92_HS_HSF1_IP_2.peaks.bed.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5834nnn/GSM5834091/suppl/GSM5834091_NK92_HSF1_…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8289nnn/GSM8289675/suppl/GSM8289675_NK92_HS_HSF1_IP_2.peaks.tsv.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5834nnn/GSM5834091/suppl/GSM5834091_NK92_HSF1_…","""GSE194364 vs GSE167552, GSE194364""","""0""",null,null,"""https://www.ncbi.nlm.nih.gov/biosample/SAMN41527440 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN25243265""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX24692877 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX13898238""","""https://doi.org/10.1038/s41556-024-01490-z""","""42 C""","""60 min.""","""37 C (inferred)""","""SRR29165320 vs SRR17735673""","""Illumina NovaSeq 6000 sequencing: GSM8289675: NK92_HS_HSF1_IP_2 Homo sapiens ChIP-Seq vs Illumina NovaSeq 6000 sequencing: GSM5834091: NK92_HSF1_IP Homo sapiens ChIP-Seq""","""SAMN41527440 vs SAMN25243265""","""The NK cell stress response status modulates anti-tumor immunity [ChIP-seq]""","""SRA1877342 vs SRA1361651""","""ChIP""","""2024-06-30 vs 2022-01-27""","""2151901402;1948688232 vs 1914225964;1820935624""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR291/020/SRR29165320/SRR29165320

&nbsp;

&nbsp;

&nbsp;

## 16. GSE166456 (ChIP-seq; YTHDC1)

- [Chromatin-associated YTHDC1 coordinates heat-induced reprogramming of gene expression](https://doi.org/10.1016/j.celrep.2022.111784)
- YTHDC1 の ChIP-seq のデータは、シスエレメントというより、gene body で見られるかを 調査することで、効果を発揮する
- `Heat-stress experiments were performed by immersion of the culture flasks or culture plates in a warm water bath for 1h at 43°C.`

In [34]:
GSE166456_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE166456/GSE166456_PEP/GSE166456_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1016/j.celrep.2022.111784").alias("reference_doi"),
    pl.lit("43 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE166456 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE166456.metadata.tsv",
    separator='\t'
)

GSE166456_combined = GSE166456_PEP.join(
    GSE166456,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE166456_combined = GSE166456_combined[[s.name for s in GSE166456_combined if not (s.null_count() == GSE166456_combined.height)]]

# display(GSE166456_combined)

In [35]:
defined_pairs_gse166456_chipseq = [
    ("SRX10059566", "SRX10059568"), # YTHDC1_HS vs. YTHDC1_UT
    ("SRX10059567", "SRX10059569")  # YTHDC1_HS_2 vs. YTHDC1_UT_2
]

paired_metada_df_gse166456 = create_paired_metadata(
    combined_df=GSE166456_combined,
    defined_sample_pairs=defined_pairs_gse166456_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse166456 is not None:
    paired_metada_df_gse166456.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE166456_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse166456)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,treatment,passages,chip_antibody,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX10059566""","""SRX10059568""","""ythdc1_hs vs ythdc1_ut""","""ChIP""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR13668061 vs SRR13668063""","""YTHDC1_HS vs YTHDC1_UT""","""GSM5071369 vs GSM5071371""","""Public on Dec 14 2022""","""Feb 09 2021""","""Dec 14 2022""","""SRA""","""1""","""HeLa""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL18573""","""Rosemary,,Kiernan""","""Gene Regulation lab""","""UMR9002 CNRS-UM""","""141 rue de la cardonille""","""Montpellier""","""34396""","""France""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM5071369 vs GSM5071371""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5071nnn/GSM5071369/suppl/GSM5071369_YTHDC1_HS.input_normalised.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5071nnn/GSM5071371/suppl/GSM5071371_YTHDC1_UT.i…","""GSE166456""","""0""","""SRX10059566 vs SRX10059568""","""Heat shock vs None""","""13-15""","""A305-096A""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN17841818 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN17841816""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX10059566 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX10059568""","""https://doi.org/10.1016/j.celrep.2022.111784""","""43 C""","""60 min.""","""NextSeq 500 sequencing: GSM5071369: YTHDC1_HS Homo sapiens ChIP-Seq vs NextSeq 500 sequencing: GSM5071371: YTHDC1_UT Homo sapiens ChIP-Seq""","""SAMN17841818 vs SAMN17841816""","""Genome-wide maps of YTHDC1 upon Heat-shock (HS) or in untreated conditions (UT) in HeLa cells""","""SRA1194006""","""ChIP""","""2022-12-15""","""1285094563 vs 1248365989""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/061/SRR13668061/SRR13668061.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/063/SRR13668063/SRR13668063.fastq.gz""","""Generic""","""YTHDC1_HS vs YTHDC1_UT""","""1d3868b7f1e492def2d48c3950e4b763 vs 649fe25862349066a9dc9c1ebb7b6bd3""","""SRP305514""","""25540977 vs 24931667""","""Genome-wide maps of YTHDC1 upon Heat-shock (HS) or in untreated conditions (UT) in HeLa cells""","""GENERATED_FILE""","""YTHDC1_HS vs YTHDC1_UT""","""NextSeq 500""","""NextSeq 500 sequencing: GSM5071369: YTHDC1_HS Homo sapiens ChIP-Seq vs NextSeq 500 sequencing: GSM5071371: YTHDC1_UT Homo sapiens ChIP-Seq"""

&nbsp;

&nbsp;

&nbsp;

## 17. GSE209667 (ChIP-seq; HSF1/PolII)

- [Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding](https://doi.org/10.1038/s41467-023-43157-7)
- Super series: [GSE209687](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE209687)
- 下記のような記述があることから、42℃ 60 min であると考えられる
- `Results; HSF1 binding is a temperature-independent hallmark of HSR; We began by examining MCF7 breast adenocarcinoma cells during temperature-induced Heat Shock Response (HSR) (Fig. 1, Supplementary Data 1, Supplementary Fig. 1a). A 60-min incubation at 42 °C induced a control HSP gene seen as spreading of the Pol II and nascent RNA signal into the gene body (Fig. 1a).`

In [36]:
GSE209667_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE209667/GSE209667_PEP/GSE209667_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens") &
    (pl.col("treatment") != "Arsenic")
).with_columns(
    pl.lit("https://doi.org/10.1038/s41467-023-43157-7").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "big_key": "experiment_accession"
    }
)

GSE209667 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE209667.metadata.tsv",
    separator='\t'
)

GSE209667_combined = GSE209667_PEP.join(
    GSE209667,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE209667_combined = GSE209667_combined[[s.name for s in GSE209667_combined if not (s.null_count() == GSE209667_combined.height)]]

display(GSE209667_combined)

sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,experiment_accession,cell_line,cell_type,chip_antibody,treatment,assembly,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""mcf7_nhs_rna_pol_ii_rep_1""","""MCF7 NHS RNA Pol II rep 1""","""GSM6383088""","""Public on Oct 30 2023""","""Jul 25 2022""","""Oct 30 2023""","""SRA""",1,"""MCF7""","""Homo sapiens""",9606,"""genomic DNA""","""Raw files were adapter-trimmed with trimmomatic and aligned to hg19 genome using hisat2. For HSF1 ChIP-seq, raw fastq files were randomly subset to 40M read pairs prior to alignment. After removing PC…","""GPL20301""","""Sergei,,Nechaev""","""sergei.nechaev@und.edu""","""Nechaev Lab""","""Biomedical Sciences""","""University of North Dakota""","""Grand Forks""","""North Dakota""",58202,"""USA""","""Illumina HiSeq 4000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM6383088""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383088/suppl/GSM6383088_MCF7_NHS_RNA_PolII_ChIPseq_rep1.bw""","""GSE209667, GSE209687""",0,"""SRX16646091""","""MCF7""","""mammary gland, adenocarcinoma""","""Rpb1 NTD (D8L4Y), Cell Signaling""","""Untreated""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN29938377""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX16646091""","""https://doi.org/10.1038/s41467-023-43157-7""","""42 C""","""60 min.""","""SRR20622371""","""Illumina HiSeq 4000 sequencing: GSM6383088: MCF7 NHS RNA Pol II rep 1 Homo sapiens ChIP-Seq""","""SAMN29938377""","""Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding""","""SRA1461984""","""ChIP""","""2023-11-02""","""2113400210;2119441394""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR206/071/SRR20622371/SRR20622371_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR206/071/SRR20622371/SRR20622371_2.fastq.gz""","""missing""","""Generic""","""MCF7 NHS RNA Pol II rep 1""","""71e1861da6c432d7900c0a0d7bc838f8;1a5a54202061877be88b0a3d1ba3b0c1""","""SRP387953""",39363347,"""Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding""","""GENERATED_FILE;GENERATED_FILE""","""MCF7 NHS RNA Pol II rep 1""","""Illumina HiSeq 4000""","""Illumina HiSeq 4000 sequencing: GSM6383088: MCF7 NHS RNA Pol II rep 1 Homo sapiens ChIP-Seq""","""mammary gland, adenocarcinom

In [37]:
defined_pairs_gse209667_chipseq = [

    # pair4_GSE209667_hsf1_k562_hs_42C_60min.txt (ChIP-Atlas には登録されていない)
    # pair4_GSE209667_hsf1_k562_ctrl.txt (ChIP-Atlas には登録されていない)
    ("SRX16646100", "SRX16646099"), # k562_hs_hsf1_rep_1 vs. k562_nhs_hsf1_rep_1
    ("SRX16646111", "SRX16646103"), # k562_hs_hsf1_rep_2 vs. k562_nhs_hsf1_rep_2

    # pair24_GSE209667_pol2_hs_42C_60min_mcf7.txt
    # pair24_GSE209667_pol2_ctrl_mcf7.txt
    ("SRX16646101", "SRX16646091"), # mcf7_hs_rna_pol_ii_rep_1 vs. mcf7_nhs_rna_pol_ii_rep_1
    ("SRX16646106", "SRX16646105"), # mcf7_hs_rna_pol_ii_rep_2 vs. mcf7_nhs_rna_pol_ii_rep_2
    # ("SRX16646109", "SRX16646108"), # mcf7_hs_h3k4me3_rep_1 vs. mcf7_nhs_h3k4me3_rep_1
    # ("SRX16646092", "SRX16646110"), # mcf7_hs_h3k4me3_rep_2 vs. mcf7_nhs_h3k4me3_rep_2


    # pair5_GSE209667_hsf1_mcf7_hs_42C_60min.txt
    # pair5_GSE209667_hsf1_mcf7_ctrl.txt
    ("SRX16646094", "SRX16646093"), # mcf7_hs_hsf1_rep_1 vs. mcf7_nhs_hsf1_rep_1
    ("SRX16646097", "SRX16646096"), # mcf7_hs_hsf1_rep_2 vs. mcf7_nhs_hsf1_rep_2

    # pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min.txt
    # pair6_GSE209667_hsf1_polyclonal_ab_ctrl.txt
    ("SRX16646115", "SRX16646113"), # mcf7_hsf1_hs_polyclonal_ab_rep_1 vs. mcf7_hsf1_nhs_polyclonal_ab_rep_1
    ("SRX16646116", "SRX16646114")  # mcf7_hsf1_hs_polyclonal_ab_rep_2 vs. mcf7_hsf1_nhs_polyclonal_ab_rep_2
]

paired_metada_df_gse209667 = create_paired_metadata(
    combined_df=GSE209667_combined,
    defined_sample_pairs=defined_pairs_gse209667_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse209667 is not None:
    paired_metada_df_gse209667.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE209667_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse209667)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")  

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,cell_line,cell_type,chip_antibody,treatment,assembly,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX16646100""","""SRX16646099""","""k562_hs_hsf1_rep_1 vs k562_nhs_hsf1_rep_1""","""K562 HS HSF1 rep 1 vs K562 NHS HSF1 rep 1""","""GSM6383105 vs GSM6383104""","""Public on Oct 30 2023""","""Jul 25 2022""","""Oct 30 2023""","""SRA""","""1""","""K562""","""Homo sapiens""","""9606""","""genomic DNA""","""Raw files were adapter-trimmed with trimmomatic and aligned to hg19 genome using hisat2. For HSF1 ChIP-seq, raw fastq files were randomly subset to 40M read pairs prior to alignment. After removing PC…","""GPL20795""","""Sergei,,Nechaev""","""sergei.nechaev@und.edu""","""Nechaev Lab""","""Biomedical Sciences""","""University of North Dakota""","""Grand Forks""","""North Dakota""","""58202""","""USA""","""HiSeq X Ten""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM6383105 vs GSM6383104""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383105/suppl/GSM6383105_K562_HS_HSF1_ChIPseq_rep1.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383104/suppl/GSM6383104_K562_NHS_HSF…","""GSE209667, GSE209687""","""0""","""K562""","""bone marrow, myelogenous leukemia""","""HSF1 (Enzo # ADI-SPA-901-D)""","""Heat shock vs Untreated""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN29938360 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN29938361""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX16646100 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX16646099""","""https://doi.org/10.1038/s41467-023-43157-7""","""42 C""","""60 min.""","""SRR20622364 vs SRR20622365""","""HiSeq X Ten sequencing: GSM6383105: K562 HS HSF1 rep 1 Homo sapiens ChIP-Seq vs HiSeq X Ten sequencing: GSM6383104: K562 NHS HSF1 rep 1 Homo sapiens ChIP-Seq""","""SAMN29938360 vs SAMN29938361""","""Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding""","""SRA1461984""","""ChIP""","""2023-11-01""","""2749755074;2762634622 vs 2626357708;2642234234""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR206/064/SRR20622364/SRR20622364_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR206/064/SRR20622364/SRR20622364_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR206/065/SRR2062…","""missing""",

&nbsp;

&nbsp;

&nbsp;

## 18. GSE270671 (ChIP-seq; HSF1)

- [An HSF1–JMJD6–HSP feedback circuit promotes cell adaptation to proteotoxic stress](https://doi.org/10.1073/pnas.2313370121)
- Appendix 01 (PDF) に以下のような記載があるため、条件は 42℃, 60 min であると考えられる
- `Treatments. ... For the heat shock experiments, cells were heat shocked at 42 ˚C for 1 hour. ...`

In [38]:
GSE270671_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE270671/GSE270671_PEP/GSE270671_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1073/pnas.2313370121").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "sample_geo_accession": "library_name",
        "big_key": "experiment_accession"
    }
)

GSE270671 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE270671.metadata.tsv",
    separator='\t'
).unique(subset=["library_name", "experiment_accession"], keep="first")

GSE270671_combined = GSE270671_PEP.join(
    GSE270671,
    on=["library_name", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE270671_combined = GSE270671_combined[[s.name for s in GSE270671_combined if not (s.null_count() == GSE270671_combined.height)]]

display(GSE270671_combined)

sample_name,sample_title,library_name,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""pc3m_sint_basal_hsf1_chip_biol_rep_1""","""PC3M, siNT, Basal, HSF1 ChIP, biol rep 1""","""GSM8349779""","""Public on Jul 18 2024""","""Jun 24 2024""","""Jul 18 2024""","""SRA""",1,"""PC3M""","""Homo sapiens""",9606,"""Cells were grown at 37°C and 5% CO2""","""genomic DNA""","""GPL18573""","""Marc,L,Mendillo""","""mendillo@northwestern.edu""",3125034862,"""Marc Mendillo""","""Biochemistry and Molecular Genetics""","""Northwestern University""","""Chicago""","""IL""",60611,"""USA""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8349779""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8349nnn/GSM8349779/suppl/GSM8349779_MA_PC3M_siNT_Basal_H1_1.bw""","""GSE270671""",0,"""SRX25380936""","""PC3M""","""Prostate cancer cell line""","""siNT""","""Basal""","""hg38""","""bigWig, narrowPeak""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN42017728""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX25380936""","""https://doi.org/10.1073/pnas.2313370121""","""42 C""","""60 min.""","""SRR29884683""","""NextSeq 500 sequencing: GSM8349779: PC3M, siNT, Basal, HSF1 ChIP, biol rep 1 Homo sapiens ChIP-Seq""","""SAMN42017728""","""An HSF1-JMJD6-HSP feedback circuit promotes cell adaptation to proteotoxic stress [ChIP-Seq]""","""SRA1906780""","""ChIP""","""2024-07-23""",147407188,"""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR298/083/SRR29884683/SRR29884683.fastq.gz""","""missing""","""Generic""","""PC3M, siNT, Basal, HSF1 ChIP, biol rep 1""","""4ca268ad8b3db1d6a079d77e5e4564d5""","""SRP520857""",4605872,"""An HSF1-JMJD6-HSP feedback circuit promotes cell adaptation to proteotoxic stress [ChIP-Seq]""","""GENERATED_FILE""","""PC3M, siNT, Basal, HSF1 ChIP, biol rep 1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM8349779: PC3M, siNT, Basal, HSF1 ChIP, biol rep 1 Homo sapiens ChIP-Seq""","""Prostate cancer cell line""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR298/083/SRR29884683/SRR29884683.fastq.gz""",229794952,"""ChIP-Seq""","""GENOMIC""","""SINGLE""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR298/083/SRR29884683/SRR29884683.fastq.gz""","""1;131567;2759;33154;33208;6072;33213;33511

In [39]:
defined_pairs_gse270671_chipseq = [

    # pair7_GSE270671_hsf1_pc3m_sijmjd6_hs_42C_60min.txt
    # pair7_GSE270671_hsf1_pc3m_sijmjd6_ctrl.txt
    ("SRX25380948", "SRX25380940"), # pc3m_sijmjd6_hs_hsf1_chip_biol_rep_1 vs. pc3m_sijmjd6_basal_hsf1_chip_biol_rep_1
    ("SRX25380949", "SRX25380941"), # pc3m_sijmjd6_hs_hsf1_chip_biol_rep_2 vs. pc3m_sijmjd6_basal_hsf1_chip_biol_rep_2


    ("SRX25380934", "SRX25380942"), # pc3m_sijmjd6_hs_input_biol_rep_1 vs. pc3m_sijmjd6_basal_input_biol_rep_1
    ("SRX25380935", "SRX25380943"), # pc3m_sijmjd6_hs_input_biol_rep_2 vs. pc3m_sijmjd6_basal_input_biol_rep_2

    # pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min.txt
    # pair8_GSE270671_hsf1_pc3m_sint_ctrl.txt
    ("SRX25380944", "SRX25380936"), # pc3m_sint_hs_hsf1_chip_biol_rep_1 vs. pc3m_sint_basal_hsf1_chip_biol_rep_1
    ("SRX25380945", "SRX25380937"), # pc3m_sint_hs_hsf1_chip_biol_rep_2 vs. pc3m_sint_basal_hsf1_chip_biol_rep_2

    
    ("SRX25380946", "SRX25380938"), # pc3m_sint_hs_input_biol_rep_1 vs. pc3m_sint_basal_input_biol_rep_1
    ("SRX25380947", "SRX25380939")  # pc3m_sint_hs_input_biol_rep_2 vs. pc3m_sint_basal_input_biol_rep_2
]

paired_metada_df_gse270671 = create_paired_metadata(
    combined_df=GSE270671_combined,
    defined_sample_pairs=defined_pairs_gse270671_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse270671 is not None:
    paired_metada_df_gse270671.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE270671_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse270671)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,library_name,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,cell_line,cell_type,genotype,treatment,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX25380948""","""SRX25380940""","""pc3m_sijmjd6_hs_hsf1_chip_biol_rep_1 vs pc3m_sijmjd6_basal_hsf1_chip_biol_rep_1""","""PC3M, siJMJD6, HS, HSF1 ChIP, biol rep 1 vs PC3M, siJMJD6, Basal, HSF1 ChIP, biol rep 1""","""GSM8349791 vs GSM8349783""","""Public on Jul 18 2024""","""Jun 24 2024""","""Jul 18 2024""","""SRA""","""1""","""PC3M""","""Homo sapiens""","""9606""","""Cells were grown at 37°C and 5% CO2""","""genomic DNA""","""GPL18573""","""Marc,L,Mendillo""","""mendillo@northwestern.edu""","""3125034862""","""Marc Mendillo""","""Biochemistry and Molecular Genetics""","""Northwestern University""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NextSeq 500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8349791 vs GSM8349783""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8349nnn/GSM8349791/suppl/GSM8349791_MA_PC3M_siJ6_HS_H1_1.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8349nnn/GSM8349783/suppl/GSM8349783_MA_PC3M_siJ6_Basa…","""GSE270671""","""0""","""PC3M""","""Prostate cancer cell line""","""siJMJD6""","""Heat shock 42C (1h), 2h Recovery vs Basal""","""hg38""","""bigWig, narrowPeak""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN42017716 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN42017724""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX25380948 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX25380940""","""https://doi.org/10.1073/pnas.2313370121""","""42 C""","""60 min.""","""SRR29884635 vs SRR29884665""","""NextSeq 500 sequencing: GSM8349791: PC3M, siJMJD6, HS, HSF1 ChIP, biol rep 1 Homo sapiens ChIP-Seq vs NextSeq 500 sequencing: GSM8349783: PC3M, siJMJD6, Basal, HSF1 ChIP, biol rep 1 Homo sapiens ChIP-…","""SAMN42017716 vs SAMN42017724""","""An HSF1-JMJD6-HSP feedback circuit promotes cell adaptation to proteotoxic stress [ChIP-Seq]""","""SRA1906780""","""ChIP""","""2024-07-23""","""149151114 vs 175147900""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR298/035/SRR29884635/SRR29884635.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR298/065/SRR29884665/SRR29884665.fastq.gz""","""missing""","""Generic""","""PC3M, siJMJD6, HS, HSF1 ChIP, biol rep 1 vs PC3M

&nbsp;

&nbsp;

&nbsp;

## 19. GSE253873 (ChIP-seq; H3/H3.3)

- [Histone H3.3 chaperone HIRA renders stress-responsive genes poised for prospective lethal stresses in acquired tolerance](https://doi.org/10.1111/gtc.13140)
- `4.2 Heat treatment: Cells were seeded at a density of 1.1 × 104 or 1.7 × 104 cells/cm2 in 35 mm dishes for immortalized cells and primary cells, respectively, and incubated for 24 h at 37°C before heat shock treatment. For priming stress treatment, culture media was replaced with fresh media pre-heated at 40°C, and cells were cultured in an incubator at 40°C (±0.5°C) and 5% CO2 for 2 h. For lethal stress treatment, culture media was replaced with fresh media pre-heated at 47°C, and cells were cultured in an incubator at 47°C (±0.5°C) and 5% CO2 for 35 min. After heat treatment, the media was replaced by fresh media (37°C), and the cells were incubated in an incubator at 37°C (±0.5°C) and 5% CO2 for 48 h. Cell viability was quantified using a Cell Counting Kit-8 (Dojindo) and SpectraMax 340 Microplate Reader (Molecular Devices).`

In [40]:
GSE253873_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE253873/GSE253873_PEP/GSE253873_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1111/gtc.13140").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("120 min.").alias("stress treatment time")
).rename(
    {
        "big_key": "experiment_accession"
    }
)

GSE253873 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE253873.metadata.tsv",
    separator='\t'
)

GSE253873_combined = GSE253873_PEP.join(
    GSE253873,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE253873_combined = GSE253873_combined[[s.name for s in GSE253873_combined if not (s.null_count() == GSE253873_combined.height)]]

display(GSE253873_combined)

sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,genotype,treatment,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str
"""wi-38-htert_37°c_h3_3_chip""","""WI-38-hTERT, 37°C, H3.3, ChIP""","""GSM8029678""","""Public on Oct 15 2024""","""Jan 22 2024""","""Oct 15 2024""","""SRA""",1,"""lung""","""Homo sapiens""",9606,"""genomic DNA""","""WI38_37C_H3.3""","""GPL11154""","""Yoshikazu,,Nagagaki""","""Kyoto University""","""Yoshida-Konoe-cho, Sakyo-ku""","""Kyoto-shi""","""Kyoto""","""606-8501""","""Japan""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8029678""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8029nnn/GSM8029678/suppl/GSM8029678_37C-H3.3-n2-m1-hg38.100.bw""","""GSE253873""",0,"""SRX23349043""","""lung""","""WI-38-hTERT""","""immortalized human fibroblast cells""","""WT""","""control""","""hg19""","""bigWig""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN39538440""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX23349043""","""https://doi.org/10.1111/gtc.13140""","""42 C""","""120 min.""","""SRR27681803""","""Illumina HiSeq 2000 sequencing: GSM8029678: WI-38-hTERT, 37°C, H3.3, ChIP Homo sapiens ChIP-Seq""","""SAMN39538440""","""Histone H3.3 chaperone HIRA is required for acquired tolerance by rendering stress-responsive genes poised for prospective lethal stresses [ChIP-seq]""","""SRA1789713""","""ChIP""","""2024-10-16""",1201059139,"""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR276/003/SRR27681803/SRR27681803.fastq.gz""","""missing""","""Generic""","""WI-38-hTERT, 37°C, H3.3, ChIP""","""8b55684d32788c99b6460bc851dfe03a""","""SRP485224""",36990971,"""Histone H3.3 chaperone HIRA is required for acquired tolerance by rendering stress-responsive genes poised for prospective lethal stresses [ChIP-seq]""","""GENERATED_FILE""","""WI-38-hTERT, 37°C, H3.3, ChIP""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM8029678: WI-38-hTERT, 37°C, H3.3, ChIP Homo sapiens ChIP-Seq""","""immortalized human fibroblast cells""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR276/003/SRR27681803/SRR27681803.fastq.gz""",2441404086,"""ChIP-Seq""","""GENOMIC""","""SINGLE""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR276/003/SRR27681803/SRR27681803.fastq.gz""","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;

In [41]:
defined_pairs_gse253873_chipseq = [
    ("SRX23349047", "SRX23349043"), # wi-38-htert_40°c_h3_3_chip vs. wi-38-htert_37°c_h3_3_chip
    ("SRX23349048", "SRX23349044"), # wi-38-htert_40°c_h3_chip vs. wi-38-htert_37°c_h3_chip
    ("SRX23349049", "SRX23349045"), # wi-38-htert_40°c_control_igg_chip vs. wi-38-htert_37°c_control_igg_chip
    ("SRX23349050", "SRX23349046")  # wi-38-htert_40°c_input vs. wi-38-htert_37°c_input
]

paired_metada_df_gse253873 = create_paired_metadata(
    combined_df=GSE253873_combined,
    defined_sample_pairs=defined_pairs_gse253873_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse253873 is not None:
    paired_metada_df_gse253873.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE253873_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse253873)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,tissue,cell_line,cell_type,genotype,treatment,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX23349047""","""SRX23349043""","""wi-38-htert_40°c_h3_3_chip vs wi-38-htert_37°c_h3_3_chip""","""WI-38-hTERT, 40°C, H3.3, ChIP vs WI-38-hTERT, 37°C, H3.3, ChIP""","""GSM8029682 vs GSM8029678""","""Public on Oct 15 2024""","""Jan 22 2024""","""Oct 15 2024""","""SRA""","""1""","""lung""","""Homo sapiens""","""9606""","""genomic DNA""","""WI38_40C_H3.3 vs WI38_37C_H3.3""","""GPL11154""","""Yoshikazu,,Nagagaki""","""Kyoto University""","""Yoshida-Konoe-cho, Sakyo-ku""","""Kyoto-shi""","""Kyoto""","""606-8501""","""Japan""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8029682 vs GSM8029678""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8029nnn/GSM8029682/suppl/GSM8029682_40C-H3.3-n2-m1-hg38.100.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8029nnn/GSM8029678/suppl/GSM8029678_37C-H3.3-n2-m1…","""GSE253873""","""0""","""lung""","""WI-38-hTERT""","""immortalized human fibroblast cells""","""WT""","""heat stress treatment (40°C, 2 hours) vs control""","""hg19""","""bigWig""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN39538436 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN39538440""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX23349047 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX23349043""","""https://doi.org/10.1111/gtc.13140""","""42 C""","""120 min.""","""SRR27681799 vs SRR27681803""","""Illumina HiSeq 2000 sequencing: GSM8029682: WI-38-hTERT, 40°C, H3.3, ChIP Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM8029678: WI-38-hTERT, 37°C, H3.3, ChIP Homo sapiens ChIP-Seq""","""SAMN39538436 vs SAMN39538440""","""Histone H3.3 chaperone HIRA is required for acquired tolerance by rendering stress-responsive genes poised for prospective lethal stresses [ChIP-seq]""","""SRA1789713""","""ChIP""","""2024-10-16""","""1114912481 vs 1201059139""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR276/099/SRR27681799/SRR27681799.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR276/003/SRR27681803/SRR27681803.fastq.gz""","""missing""","""Generic""","""WI-38-hTERT, 40°C, H3.3, ChIP vs WI-38-hTERT, 37°C, H3.3, ChIP""","""d6e1018531b98c275e815e433dd74b20 vs 8b55684d32788c99b6460bc851dfe03a"

&nbsp;

&nbsp;

&nbsp;

## 20. GSE259305 (ChIP-seq; FUBP3/PolII(Ser2p)/dCas9) (ChIP-Atlas: False)

- [TurboCas: A method for locus-specific labeling of genomic regions and isolating their associated protein interactome](https://doi.org/10.1016/j.molcel.2024.11.007)
- dCas9 などの他に、FUBP3, Pol II などのデータも含まれている

In [42]:
GSE259305_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE259305/GSE259305_PEP/GSE259305_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1016/j.cell.2023.03.010").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("120 min.").alias("stress treatment time")
).rename(
    {
        "big_key": "experiment_accession"
    }
)

GSE259305 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE259305.metadata.tsv",
    separator='\t'
)

GSE259305_combined = GSE259305_PEP.join(
    GSE259305,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE259305_combined = GSE259305_combined[[s.name for s in GSE259305_combined if not (s.null_count() == GSE259305_combined.height)]]

display(GSE259305_combined)

sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,chip_antibody,chip,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""chipseq_293t_tc_300_hs_neg_chip_dcas9""","""ChIPseq_293T_TC_300_HS_neg_ChIP_dCas9""","""GSM8113615""","""Public on Dec 21 2024""","""Feb 26 2024""","""Dec 21 2024""","""SRA""",1,"""HEK293T""","""Homo sapiens""",9606,"""100mM biotin (Sigma-Aldrich, B4501) stock prepared in DMSO was added directly to tissue culture media at a final concentration of 100-500µM for immunostaining experiments and at 500µM for immunoblotti…","""Human HCT116 cells were cultured in DMEM (Sigma #D6429) supplemented with 10% FBS (Sigma #F2442), 1% Pen-Strep (GIBCO #15140-122) in incubators set to 37°C and 5% CO2. Human DLD-1 cells were cultured …","""genomic DNA""","""ChIP libraries were prepared using the KAPA HTP library preparation kit (KAPA Biosystems). RNA-seq libraries were prepared using the NEBNext® Ultra™ II Directional RNA Library Prep Kit for Illumina® (…","""GPL24676""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""",60611,"""USA""","""Illumina NovaSeq 6000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8113615""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8113nnn/GSM8113615/suppl/GSM8113615_ChIPseq_293T_TC_300_HS_neg_ChIP_dCas9.bw""","""GSE259305""",0,"""SRX23751446""","""HEK293T""","""human embryonic kidney cells""","""TurboCas FOS-300""","""37C 3 hours (no heat shock)""","""Cas9 (Di+M36:M67agenode, #C15310258-100)""","""Trimmed ChIP-seq reads were aligned to the human genome (University of California at Santa Cruz [UCSC] hg38 build) using bowtie 2.4.5 with --sensitive option. Where spike-in is indicated, mm10 assembl…","""hg38""","""bigwig files for ChIP-seq data""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN40144550""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX23751446""","""https://doi.org/10.1016/j.cell.2023.03.010""","""42 C""","""120 min.""","""SRR28106413""","""Illumina NovaSeq 6000 sequencing: GSM8113615: ChIPseq_293T_TC_300_HS_neg_ChIP_dCas9 Homo sapiens ChIP-Seq""","""SAMN40144550""","""TurboCas:

In [43]:
defined_pairs_gse259305_chipseq = [

    # FUBP3
    ("SRX26888307", "SRX26888305"), # chip-seq_fubp3_dld1_shfubp3_hs_ab1 vs. chip-seq_fubp3_dld1_shfubp3_nonhs_ab1
    ("SRX26888311", "SRX26888309"), # chip-seq_fubp3_dld1_shfubp3_hs_ab2 vs. chip-seq_fubp3_dld1_shfubp3_nonhs_ab2
    ("SRX26888306", "SRX26888304"), # chip-seq_fubp3_dld1_shscr_hs_ab1 vs. chip-seq_fubp3_dld1_shscr_nonhs_ab1
    ("SRX26888310", "SRX26888308"), # chip-seq_fubp3_dld1_shscr_hs_ab2 vs. chip-seq_fubp3_dld1_shscr_nonhs_ab2

    # Pol II
    # pair25_GSE259305_pol2_hs_42C_120min_ser2p_dld1_shfubp3.txt
    # pair25_GSE259305_pol2_ctrl_dld1_shfubp3.txt
    ("SRX23751483", "SRX23751481"), # chip-seq-ser2p-dld1-shfubp3-hs-rep1 vs. chip-seq-ser2p-dld1-shfubp3-nonhs-rep1
    ("SRX23751487", "SRX23751486"), # chip-seq-ser2p-dld1-shfubp3-hs-rep2 vs. chip-seq-ser2p-dld1-shfubp3-nonhs-rep2

    # pair26_GSE259305_pol2_hs_42C_120min_ser2p_dld1_shscr.txt
    # pair26_GSE259305_pol2_ctrl_dld1_shscr.txt
    ("SRX23751485", "SRX23751484"), # chip-seq-ser2p-dld1-shscr-hs-rep1 vs. chip-seq-ser2p-dld1-shscr-nonhs-rep1
    ("SRX23751489", "SRX23751488"), # chip-seq-ser2p-dld1-shscr-hs-rep2 vs. chip-seq-ser2p-dld1-shscr-nonhs-rep2

    # dCas9
    ("SRX23751448", "SRX23751446"), # chipseq_293t_tc_300_hs_pos_chip_dcas9 vs. chipseq_293t_tc_300_hs_neg_chip_dcas9
    ("SRX23751449", "SRX23751447"), # chipseq_293t_tc_300_hs_pos_input vs. chipseq_293t_tc_300_hs_neg_input
    ("SRX23751482", "SRX23751479"), # chipseq_hct116_tc_lgp_hs_pos_input vs. chipseq_hct116_tc_lgp_hs_neg_input
    ("SRX23751480", "SRX23751478"), # chipseq_hct116_tc_lgp_hs_pos_chip_dcas9 vs. chipseq_hct116_tc_lgp_hs_neg_chip_dcas9
    ("SRX23751477", "SRX23751474"), # chipseq_hct116_tc_ex1_hs_pos_input vs. chipseq_hct116_tc_ex1_hs_neg_input
    ("SRX23751476", "SRX23751475"), # chipseq_hct116_tc_ex1_hs_pos_chip_dcas9 vs. chipseq_hct116_tc_ex1_hs_neg_chip_dcas9
    ("SRX23751473", "SRX23751471"), # chipseq_hct116_tc_600_hs_pos_input vs. chipseq_hct116_tc_600_hs_neg_input
    ("SRX23751472", "SRX23751470"), # chipseq_hct116_tc_600_hs_pos_chip_dcas9 vs. chipseq_hct116_tc_600_hs_neg_chip_dcas9
    ("SRX23751469", "SRX23751467"), # chipseq_hct116_tc_300_hs_pos_input vs. chipseq_hct116_tc_300_hs_neg_input
    ("SRX23751468", "SRX23751466"), # chipseq_hct116_tc_300_hs_pos_chip_dcas9 vs. chipseq_hct116_tc_300_hs_neg_chip_dcas9
    ("SRX23751465", "SRX23751463"), # chipseq_hct116_mt_lgp_hs_pos_input vs. chipseq_hct116_mt_lgp_hs_neg_input
    ("SRX23751464", "SRX23751462"), # chipseq_hct116_mt_lgp_hs_pos_chip_dcas9 vs. chipseq_hct116_mt_lgp_hs_neg_chip_dcas9
    ("SRX23751461", "SRX23751459"), # chipseq_293t_tc_lgp_hs_pos_input vs. chipseq_293t_tc_lgp_hs_neg_input
    ("SRX23751460", "SRX23751458"), # chipseq_293t_tc_lgp_hs_pos_chip_dcas9 vs. chipseq_293t_tc_lgp_hs_neg_chip_dcas9
    ("SRX23751456", "SRX23751455"), # chipseq_293t_tc_ex1_hs_pos_input vs. chipseq_293t_tc_ex1_hs_neg_input
    ("SRX23751457", "SRX23751454"), # chipseq_293t_tc_ex1_hs_pos_chip_dcas9 vs. chipseq_293t_tc_ex1_hs_neg_chip_dcas9
    ("SRX23751453", "SRX23751451"), # chipseq_293t_tc_600_hs_pos_input vs. chipseq_293t_tc_600_hs_neg_input
    ("SRX23751452", "SRX23751450")  # chipseq_293t_tc_600_hs_pos_chip_dcas9 vs. chipseq_293t_tc_600_hs_neg_chip_dcas9
]

paired_metada_df_gse259305 = create_paired_metadata(
    combined_df=GSE259305_combined,
    defined_sample_pairs=defined_pairs_gse259305_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse259305 is not None:
    paired_metada_df_gse259305.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE259305_chipseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse259305)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,cell_line,cell_type,genotype,treatment,chip_antibody,chip,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX26888307""","""SRX26888305""","""chip-seq_fubp3_dld1_shfubp3_hs_ab1 vs chip-seq_fubp3_dld1_shfubp3_nonhs_ab1""","""ChIP-seq_FUBP3_DLD1_shFUBP3_HS_Ab1 vs ChIP-seq_FUBP3_DLD1_shFUBP3_NonHS_Ab1""","""GSM8653191 vs GSM8653189""","""Public on Dec 21 2024""","""Nov 26 2024""","""Dec 21 2024""","""SRA""","""1""","""DLD-1""","""Homo sapiens""","""9606""","""Cells were treated with heat shock at 42°C for 2 h, following shRNA knockdown.""","""Human DLD-1 cells were cultured at 37°C in DMEM supplemented with 10% FBS and 1% Pen-Strep in CO2 incubators.""","""genomic DNA""","""Cells were crosslinked with 1% formaldehyde (Thermo Scientific, Cat# 28908) in PBS for 10 min at r.t. To shear chromatin, sonication was performed using the E220 ultrasonicator (Covaris). The mouse em…","""GPL30173""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""","""60611""","""USA""","""NextSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8653191 vs GSM8653189""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8653nnn/GSM8653191/suppl/GSM8653191_ChIP-seq_FUBP3_DLD1_shFUBP3_HS_Ab1.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8653nnn/GSM8653189/suppl/GSM8653189_ChI…","""GSE259305""","""0""","""DLD-1""","""Colorectal adenocarcinoma""","""FUBP3 knockdown""","""null""","""FUBP3""","""The poor-quality bases were trimmed using Trimmomatic 0.33 (PMID: 24695404).Trimmed ChIP-seq reads were aligned to the human genome (University of California at Santa Cruz [UCSC] hg38 build) using bow…","""hg38""","""bigWig""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN45068049 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN45068051""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX26888307 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX26888305""","""https://doi.org/10.1016/j.cell.2023.03.010""","""42 C""","""120 min.""","""SRR31520788 vs SRR31520790""","""NextSeq 2000 sequencing: GSM8653191: ChIP-seq_FUBP3_DLD1_shFUBP3_HS_Ab1 Homo sapiens

&nbsp;

&nbsp;

&nbsp;

## 21. GSE276302 (ChIP-seq; HSF1) (ChIP-Atlas: False)

- [HSF1–DBC1 axis drives prostate cancer progression by activating a metastatic transcriptional program](https://doi.org/10.1038/s12276-025-01545-7)
- `Detection of phosphorylated HSF1 and kinase assays; 22RV1 and SM1 cells were treated with HS at 42 °C or GSK-626616 (DYRK inhibitor, Tocris Bioscience) and lysed in FLAG lysis buffer24 containing phosphatase and protease inhibitor cocktails (Roche Diagnostics). `

In [44]:
GSE276302_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE276302/GSE276302_PEP/GSE276302_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1038/s12276-025-01545-7").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "big_key": "experiment_accession"
    }
)

GSE276302 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE276302.metadata.tsv",
    separator='\t'
)

GSE276302_combined = GSE276302_PEP.join(
    GSE276302,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE276302_combined = GSE276302_combined[[s.name for s in GSE276302_combined if not (s.null_count() == GSE276302_combined.height)]]

display(GSE276302_combined)

sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,experiment_accession,chip_antibody,cell_line,cell_type,treatment,basecalling,adapter_trimming,align,deduplication,peak_calling,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""hsf1_nhs_22rv1""","""HSF1_NHS_22RV1""","""GSM8496166""","""Public on Sep 02 2025""","""Sep 04 2024""","""Sep 02 2025""","""SRA""",1,"""22RV1""","""Homo sapiens""",9606,"""genomic DNA""","""GPL24676""","""Jeong,Hoon,Kim""","""Sungkyunkwan University""","""Irwon-dong Gangnam-gu""","""Seoul""","""135-710""","""South Korea""","""Illumina NovaSeq 6000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8496166""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8496nnn/GSM8496166/suppl/GSM8496166_HSF1_NHS_22RV1.bw""","""GSE276302""",0,"""SRX25966363""","""HSF1 (Enzo, ADI-SPA-901-F)""","""22RV1""","""Castration-resistant prostate cancer cells""","""Non-heat shock (NHS)""","""bcl2fastq v2.20.0""","""Trimmomatic v0.38""","""Bowtie 1.1.2""","""Picard 1.118 MarkDuplicates""","""MACS2 2.1.1.20160309""","""hg19""","""bigwig""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN43495088""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX25966363""","""https://doi.org/10.1038/s12276-025-01545-7""","""42 C""","""60 min.""","""SRR30542983""","""Illumina NovaSeq 6000 sequencing: GSM8496166: HSF1_NHS_22RV1 Homo sapiens ChIP-Seq""","""SAMN43495088""","""DBC1 promotes metastatic castration-resistant prostate cancer by enhancing the stability and activity of HSF1 [ChIP-seq]""","""SRA1962544""","""ChIP""","""2025-09-03""","""2820639938;3004840797""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/083/SRR30542983/SRR30542983_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/083/SRR30542983/SRR30542983_2.fastq.gz""","""missing""","""Generic""","""HSF1_NHS_22RV1""","""b134536c3a5864542dcb53aa42891cf7;dc75ad4f6b051ff42a7a3c4d1b771b63""","""SRP530416""",55191425,"""DBC1 promotes metastatic castration-resistant prostate cancer by enhancing the stability and activity of HSF1 [ChIP-seq]""","""GENERATED_FILE;GENERATED_FILE""","""HSF1_NHS_22RV1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM8496166: HSF1_NHS_22RV1 Homo sapiens ChIP-Seq""","""Castration-resistant prostate cancer cells""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/083/SRR30542983/SRR30542983_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/083/SRR30542983/SRR30542983_2.fastq.

In [45]:
defined_pairs_gse276302_chipseq = [
    ("SRX25966366", "SRX25966363"), # hsf1_hs_22rv1 vs. hsf1_nhs_22rv1
    ("SRX25966368", "SRX25966365"), # hsf1_hs_22rv1_d1ko vs. hsf1_nhs_22rv1_d1ko
    ("SRX25966367", "SRX25966364"), # hsf1_hs_22rv1_h1ko vs. hsf1_nhs_22rv1_h1ko
    ("SRX25966372", "SRX25966369"), # input_hsf1_hs_22rv1 vs. input_hsf1_nhs_22rv1
    ("SRX25966374", "SRX25966371"), # input_hsf1_hs_22rv1_d1ko vs. input_hsf1_nhs_22rv1_d1ko
    ("SRX25966373", "SRX25966370"), # input_hsf1_hs_22rv1_h1ko vs. input_hsf1_nhs_22rv1_h1ko
    # ("SRX25966384", "SRX25966381"), # input_k27ac_hs_22rv1 vs. input_k27ac_nhs_22rv1
    # ("SRX25966386", "SRX25966383"), # input_k27ac_hs_22rv1_d1ko vs. input_k27ac_nhs_22rv1_d1ko
    # ("SRX25966385", "SRX25966382"), # input_k27ac_hs_22rv1_h1ko vs. input_k27ac_nhs_22rv1_h1ko

    # # H3K27ac
    # ("SRX25966378", "SRX25966375"), # k27ac_hs_22rv1 vs. k27ac_nhs_22rv1
    # ("SRX25966380", "SRX25966377"), # k27ac_hs_22rv1_d1ko vs. k27ac_nhs_22rv1_d1ko
    # ("SRX25966379", "SRX25966376")  # k27ac_hs_22rv1_h1ko vs. k27ac_nhs_22rv1_h1ko
]

paired_metada_df_gse276302 = create_paired_metadata(
    combined_df=GSE276302_combined,
    defined_sample_pairs=defined_pairs_gse276302_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse276302 is not None:
    paired_metada_df_gse276302.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE276302_chipseq_paired_metadata.tsv",
        separator='\t'
    )
    
    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse276302)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,chip_antibody,cell_line,cell_type,treatment,basecalling,adapter_trimming,align,deduplication,peak_calling,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX25966366""","""SRX25966363""","""hsf1_hs_22rv1 vs hsf1_nhs_22rv1""","""HSF1_HS_22RV1 vs HSF1_NHS_22RV1""","""GSM8496169 vs GSM8496166""","""Public on Sep 02 2025""","""Sep 04 2024""","""Sep 02 2025""","""SRA""","""1""","""22RV1""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL24676""","""Jeong,Hoon,Kim""","""Sungkyunkwan University""","""Irwon-dong Gangnam-gu""","""Seoul""","""135-710""","""South Korea""","""Illumina NovaSeq 6000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8496169 vs GSM8496166""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8496nnn/GSM8496169/suppl/GSM8496169_HSF1_HS_22RV1.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8496nnn/GSM8496166/suppl/GSM8496166_HSF1_NHS_22RV1.bw""","""GSE276302""","""0""","""HSF1 (Enzo, ADI-SPA-901-F)""","""22RV1""","""Castration-resistant prostate cancer cells""","""Heat shock at 42℃ for 60 min (HS) vs Non-heat shock (NHS)""","""bcl2fastq v2.20.0""","""Trimmomatic v0.38""","""Bowtie 1.1.2""","""Picard 1.118 MarkDuplicates""","""MACS2 2.1.1.20160309""","""hg19""","""bigwig""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN43495085 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN43495088""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX25966366 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX25966363""","""https://doi.org/10.1038/s12276-025-01545-7""","""42 C""","""60 min.""","""SRR30542980 vs SRR30542983""","""Illumina NovaSeq 6000 sequencing: GSM8496169: HSF1_HS_22RV1 Homo sapiens ChIP-Seq vs Illumina NovaSeq 6000 sequencing: GSM8496166: HSF1_NHS_22RV1 Homo sapiens ChIP-Seq""","""SAMN43495085 vs SAMN43495088""","""DBC1 promotes metastatic castration-resistant prostate cancer by enhancing the stability and activity of HSF1 [ChIP-seq]""","""SRA1962544""","""ChIP""","""2025-09-03""","""2996536042;3055034222 vs 2820639938;3004840797""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/080/SRR30542980/SRR30542980_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/080/SRR30542980/SRR30542980_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/083/SRR3054…","""missing""","""Generic""","""HSF1_HS_22RV1 vs HSF1_NHS_22RV1""","""89df7e85783be51912122b9acce98

&nbsp;

&nbsp;

&nbsp;

## 22. GSE303857 (ChIP-seq; PolII) (ChIP-Atlas: False)

- [CPSF73 activation and 3′ RNA polymerase II pausing are lost during readthrough transcription after heat shock](https://doi.org/10.1016/j.celrep.2026.117039)

In [46]:
GSE303857_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE303857/GSE303857_PEP/GSE303857_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1016/j.celrep.2026.117039").alias("reference_doi"),
    pl.lit("44 C").alias("stress treatment temperature"), # see "Cell culture section"
    pl.lit("60 min.").alias("stress treatment time") # see "Cell culture section"
).rename(
    {
        "big_key": "experiment_accession"
    }
)

GSE303857 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE303857.metadata.tsv",
    separator='\t'
)

GSE303857_combined = GSE303857_PEP.join(
    GSE303857,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE303857_combined = GSE303857_combined[[s.name for s in GSE303857_combined if not (s.null_count() == GSE303857_combined.height)]]

GSE303857_combined.write_csv(
    "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE303857_chipseq_paired_metadata_sorted.tsv",
    separator='\t'
)

display(GSE303857_combined)

sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,batch,chip_antibody,library_name,assembly,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""input_control_for_chip_in_hek293_cells""","""input control for ChIP in HEK293 cells""","""GSM9137026""","""Public on Jan 30 2026""","""Jul 28 2025""","""Jan 30 2026""","""SRA""",1,"""HEK293""","""Homo sapiens""",9606,"""Cells were heat shocked for 1 hour at 44C""","""genomic DNA""","""GPL24676""","""Kaitlyn,,Walsh""","""kawa3071@colorado.edu""","""Goodrich-Kugel Lab""","""CU Boulder""","""3415 Colorado Ave""","""Boulder""","""CO""",80303,"""USA""","""Illumina NovaSeq 6000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM9137026""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9137nnn/GSM9137026/suppl/GSM9137026_input_TD.ucsc.bedGraph.gz""","""GSE303857, GSE303860""",0,"""SRX29876077""","""HEK293""","""Human embryonic kidney""","""none""","""R6 and R8 chromatin prep combined 1:1""","""none""","""INPUT_HEK293""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN50237551""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX29876077""","""https://doi.org/10.1016/j.celrep.2026.117039""","""44 C""","""60 min.""","""SRR34742217""","""Illumina NovaSeq 6000 sequencing: GSM9137026: input control for ChIP in HEK293 cells Homo sapiens ChIP-Seq""","""SAMN50237551""","""CPSF73 activation and 3’ Pol II pausing are lost during read-through transcription after heat shock [ChIP-seq]""","""SRA2182094""","""ChIP""","""2026-02-22""","""2673334146;2735939949""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR347/017/SRR34742217/SRR34742217_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR347/017/SRR34742217/SRR34742217_2.fastq.gz""","""missing""","""Generic""","""input control for ChIP in HEK293 cells""","""2cae155e028ec26e1cb69b6af8eb6c15;f0533ca878e500969df2a338cc45ac1e""","""SRP604086""",37077020,"""CPSF73 activation and 3' Pol II pausing are lost during read-through transcription after heat shock [ChIP-seq]""","""GENERATED_FILE;GENERATED_FILE""","""input control for ChIP in HEK293 cells""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM9137026: input control for ChIP in HEK293 cells Homo sapiens ChIP-Seq""","""Human embryonic kidney""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR347/017/SRR34742217/SRR

In [47]:
defined_pairs_gse303857_chipseq = [
    ("SRX29876079", "SRX29876078"), # pol_ii_chip-seq__heat_shock_r1 vs. pol_ii_chip-seq__no_heat_shock_r1
    ("SRX29876081", "SRX29876080"), # pol_ii_chip-seq__heat_shock_r2 vs. pol_ii_chip-seq__no_heat_shock_r2
    ("SRX29876091", "SRX29876090"), # pol_ii_chip-seq__heat_shock_r3 vs. pol_ii_chip-seq__no_heat_shock_r3

    ("SRX29876083", "SRX29876082"), # ser2p_pol_ii_chip-seq_heat_shock_r1 vs. ser2p_pol_ii_chip-seq_no_heat_shock_r1
    ("SRX29876085", "SRX29876084"), # ser2p_pol_ii_chip-seq_heat_shock_r2 vs. ser2p_pol_ii_chip-seq_no_heat_shock_r2

    ("SRX29876087", "SRX29876086"), # tyr1p_pol_ii_chip-seq_heat_shock_r1 vs. tyr1p_pol_ii_chip-seq_no_heat_shock_r1
    ("SRX29876089", "SRX29876088")  # tyr1p_pol_ii_chip-seq_heat_shock_r2 vs. tyr1p_pol_ii_chip-seq_no_heat_shock_r2
]

paired_metada_df_gse303857 = create_paired_metadata(
    combined_df=GSE303857_combined,
    defined_sample_pairs=defined_pairs_gse303857_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse303857 is not None:
    paired_metada_df_gse303857.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE303857_chipseq_paired_metadata.tsv",
        separator='\t'
    )
    
    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse303857)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,cell_line,cell_type,treatment,batch,chip_antibody,library_name,assembly,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX29876079""","""SRX29876078""","""pol_ii_chip-seq__heat_shock_r1 vs pol_ii_chip-seq__no_heat_shock_r1""","""Pol II ChIP-seq, heat shock, R1 vs Pol II ChIP-seq, no heat shock, R1""","""GSM9137028 vs GSM9137027""","""Public on Jan 30 2026""","""Jul 28 2025""","""Jan 30 2026""","""SRA""","""1""","""HEK293""","""Homo sapiens""","""9606""","""Cells were heat shocked for 1 hour at 44C""","""genomic DNA""","""GPL24676""","""Kaitlyn,,Walsh""","""kawa3071@colorado.edu""","""Goodrich-Kugel Lab""","""CU Boulder""","""3415 Colorado Ave""","""Boulder""","""CO""","""80303""","""USA""","""Illumina NovaSeq 6000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM9137028 vs GSM9137027""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9137nnn/GSM9137028/suppl/GSM9137028_HS_Rpb1_R6_TD.ucsc.bedGraph.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9137nnn/GSM9137027/suppl/GSM9137027_NHS_Rpb1_R…","""GSE303857, GSE303860""","""0""","""HEK293""","""Human embryonic kidney""","""heat shock vs none""","""R6 chromatin prep""","""anti-Rpb1 N-term Cell Signaling, D8L4Y""","""Rpb1_R1_HS vs Rpb1_R1 NHS""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN50237549 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN50237550""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX29876079 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX29876078""","""https://doi.org/10.1016/j.celrep.2026.117039""","""44 C""","""60 min.""","""SRR34742215 vs SRR34742216""","""Illumina NovaSeq 6000 sequencing: GSM9137028: Pol II ChIP-seq, heat shock, R1 Homo sapiens ChIP-Seq vs Illumina NovaSeq 6000 sequencing: GSM9137027: Pol II ChIP-seq, no heat shock, R1 Homo sapiens ChI…","""SAMN50237549 vs SAMN50237550""","""CPSF73 activation and 3’ Pol II pausing are lost during read-through transcription after heat shock [ChIP-seq]""","""SRA2182094""","""ChIP""","""2026-02-22""","""4896644935;4944618177 vs 4768283924;4851127624""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR347/015/SRR34742215/SRR34742215_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR347/015/SRR34742215/SRR34742215_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR347/016/SRR3474…","""mis

&nbsp;

&nbsp;

&nbsp;

&nbsp;

---

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# Heat-stress related ChIP-seq Data (Histone Modification)

&nbsp;

## 1. GSE89382 (ChIP-seq; H4K5K8K12K16ac) (ChIP-Atlas: True)

- [Transcriptional response to stress is pre-wired by promoter and enhancer architecture](https://doi.org/10.1038/s41467-017-00151-0)
- `Methods; Cell lines and heat shock treatment; ... For treatments, 2 × 107 cells per sample were concentrated in 10 ml culture media and placed in 37 °C incubator (NHS) or a 42 °C water bath (HS) for 30 min. In the water bath, the temperature of the cell suspension reached 42 °C within 4 min.`
- Input sample: SRX2313417

In [48]:
GSE89382_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE89382/GSE89382_PEP/GSE89382_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    pl.lit("https://doi.org/10.1038/s41467-017-00151-0").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("30 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE89382 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE89382.metadata.tsv",
    separator='\t'
)

GSE89382_combined = GSE89382_PEP.join(
    GSE89382,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE89382_combined = GSE89382_combined[[s.name for s in GSE89382_combined if not (s.null_count() == GSE89382_combined.height)]]

# pl.Config.set_tbl_rows(3)
# display(GSE89382_combined)

In [49]:
defined_pairs_gse89382_chipseq = [
    ("SRX2313416", "SRX2313415")  # h4ac_k562_hs30 vs. h4ac_k562_nhs
]

paired_metada_df_gse89382_histone_modification = create_paired_metadata(
    combined_df=GSE89382_combined,
    defined_sample_pairs=defined_pairs_gse89382_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse89382_histone_modification is not None:
    paired_metada_df_gse89382_histone_modification.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE89382_chipseq_histone_modification_paired_metadata.tsv",
        separator='\t'
    )
    
    display(paired_metada_df_gse89382_histone_modification)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,treatment,antibody,antibody_epitope,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession_right,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX2313416""","""SRX2313415""","""h4ac_k562_hs30 vs h4ac_k562_nhs""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR4779641 vs SRR4779640""","""H4ac K562 HS30 vs H4ac K562 NHS""","""GSM2367734 vs GSM2367733""","""Public on May 21 2017""","""Nov 01 2016""","""May 15 2019""","""SRA""","""1""","""K562 cells""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL16791""","""Anniina,,Vihervaara""","""mavih@kth.se""","""+358 50 5463694""","""Vihervaara""","""Gene Technology""","""KTH Royal Institute of Technology""","""Tomtebodavägen 23A""","""Stockholm""","""17165""","""Sweden""","""Illumina HiSeq 2500""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM2367734 vs GSM2367733""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2367nnn/GSM2367734/suppl/GSM2367734_H4ac_ChIP-seq_K562_UpState06-866_R1_36nt_HS30.bigWig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2367nnn/GSM2367733/suppl…","""GSE89382""","""0""","""SRX2313416 vs SRX2313415""","""K562""","""Heat shocked (HS30) vs Non-treated (NHS)""","""Upstate 06-866""","""Recognizes H4 acetylation at lysines 5, 8, 12 and 16.""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN05960685 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN05960686""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX2313416 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX2313415""","""https://doi.org/10.1038/s41467-017-00151-0""","""42 C""","""30 min.""","""SRR4779641 vs SRR4779640""","""Illumina HiSeq 2500 sequencing: GSM2367734: H4ac K562 HS30 Homo sapiens ChIP-Seq vs Illumina HiSeq 2500 sequencing: GSM2367733: H4ac K562 NHS Homo sapiens ChIP-Seq""","""SAMN05960685 vs SAMN05960686""","""Homo sapiens""","""SRA487897""","""ChIP""","""2017-05-23""","""1358854340;1369099831 vs 1496687171;1499112818""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR477/001/SRR4779641/SRR4779641_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR477/001/SRR4779641/SRR4779641_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR477/000/SRR4779640/…","""Generic""","""H4ac K562 HS30 vs H4ac K562 NHS""","""2fb68c3c142d54bdd79a1ee54d3a7c0e;f8f1a9342f19ec42d3e1ab6cc149e5a0 vs 45626a986d6ccb0f325f4b23a1d

&nbsp;

&nbsp;

&nbsp;

## 2. GSE105028 (ChIP-seq; H3K27ac/H3K27me3/H3K4me1/H3K4me3) (ChIP-Atlas: True)

- [Architectural Proteins and Pluripotency Factors Cooperate to Orchestrate the Transcriptional Response of hESCs to Temperature Stress](https://doi.org/10.1016/j.molcel.2018.07.012)
- H3K27ac のデータは、diff analysis が可能

In [50]:
GSE105028_PEP_histone_modification = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE105028/GSE105028_PEP/GSE105028_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ChIP-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens") &
    (
        (pl.col("sample_name").str.contains("h3k27ac")) |
        (pl.col("sample_name").str.contains("h3k27me3")) |
        (pl.col("sample_name").str.contains("h3k4me1")) |
        (pl.col("sample_name").str.contains("h3k4me3")) 
    )
).with_columns(
    pl.lit("https://doi.org/10.1016/j.molcel.2018.07.012").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE105028 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE105028.metadata.tsv",
    separator='\t'
)

GSE105028_combined_histone_modification = GSE105028_PEP_histone_modification.join(
    GSE105028,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE105028_combined_histone_modification = GSE105028_combined_histone_modification[[s.name for s in GSE105028_combined_histone_modification if not (s.null_count() == GSE105028_combined_histone_modification.height)]]

GSE105028_combined_histone_modification.write_csv(
    "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE105028_chipseq_histone_modification_paired_metadata.tsv",
    separator='\t'
)

display(GSE105028_combined_histone_modification)

sample_name,protocol,organism,read_type,data_source,run_accession,experiment_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,condition,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type,fastq_ftp,base_count,library_strategy_right,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str
"""h3k27ac_chip-seq""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6177967""","""SRX3288589""","""H3K27ac ChIP-Seq""","""GSM2816648""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""",1,"""H9 hESCs""","""Homo sapiens""",9606,"""genomic DNA""","""ChIP-Seq under normal conditions, H3K27ac_NT.bw""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""",361102,"""China""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM2816648""","""NONE""","""GSE105028""",0,"""SRX3288589""","""H9 human Embryonic Stem Cell Line""","""under normal conditions""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07790056""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3288589""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""42 C""","""60 min.""","""Illumina HiSeq 2000 sequencing: GSM2816648: H3K27ac ChIP-Seq Homo sapiens ChIP-Seq""","""SAMN07790056""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""SRA619794""","""ChIP""","""2018-08-04""","""695342487;691807077""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/007/SRR6177967/SRR6177967_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/007/SRR6177967/SRR6177967_2.fastq.gz""","""Generic""","""H3K27ac ChIP-Seq""","""418694ebaab36691627d0842ba30c44d;1230634a55dfa66604e2d349d9e0f015""","""SRP120027""",21931006,"""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""GENERATED_FILE;GENERATED_FILE""","""H3K27ac ChIP-Seq""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM2816648: H3K27ac ChIP-Seq Homo sapiens ChIP-Seq""",null,"""ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/007/SRR6177967/SRR6177967_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR617/007/SRR6177967/SRR6177967_2.fastq.gz""",2193100600,"""ChIP-Seq""","""GENOMIC""","""PAIRED""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR617/007/SRR6177967/SRR6177967_1.fas

In [51]:
defined_pairs_gse105028_chipseq_histone_modification = [
    ("SRX3316076", "SRX3316075"), # hs_brg1-ko_h3k27ac vs. brg1-ko_h3k27ac
    ("SRX3288598", "SRX3288597"), # hs_h3k4me3_chip-seq vs. h3k4me3_chip-seq
    ("SRX3288596", "SRX3288595"), # hs_h3k4me1_chip-seq vs. h3k4me1_chip-seq
    ("SRX3288594", "SRX3288593"), # hs_h3k27me3_chip-seq vs. h3k27me3_chip-seq

    # H3K27ac
    ("SRX3288592", "SRX3288591"), # hs_h3k27ac_chip-seq_rep vs. h3k27ac_chip-seq_rep
    ("SRX3288590", "SRX3288589"), # hs_h3k27ac_chip-seq vs. h3k27ac_chip-seq
]

paired_metada_df_gse105028_histone_modification = create_paired_metadata(
    combined_df=GSE105028_combined_histone_modification,
    defined_sample_pairs=defined_pairs_gse105028_chipseq_histone_modification,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse105028_histone_modification is not None:
    paired_metada_df_gse105028_histone_modification.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE105028_chipseq_histone_modification_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metada_df_gse105028_histone_modification)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,condition,genome_build,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type,fastq_ftp,base_count,library_strategy_right,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX3316076""","""SRX3316075""","""hs_brg1-ko_h3k27ac vs brg1-ko_h3k27ac""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6206782 vs SRR6206781""","""HS BRG1-KO_H3K27ac vs BRG1-KO_H3K27ac""","""GSM2829015 vs GSM2829014""","""Public on Aug 03 2018""","""Oct 24 2017""","""May 15 2019""","""SRA""","""1""","""H9 hESCs""","""Homo sapiens""","""9606""","""genomic DNA""","""ChIP-Seq under heat shock conditions vs ChIP-Seq under normal conditions""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""","""361102""","""China""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM2829015 vs GSM2829014""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2829nnn/GSM2829015/suppl/GSM2829015_HS_Brg1-KO_H3K27ac.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2829nnn/GSM2829014/suppl/GSM2829014_Brg1-KO_H3K27ac.bw""","""GSE105028""","""0""","""SRX3316076 vs SRX3316075""","""null""","""null""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07828425 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN07828426""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3316076 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX3316075""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""42 C""","""60 min.""","""Illumina HiSeq 2000 sequencing: GSM2829015: HS BRG1-KO_H3K27ac Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM2829014: BRG1-KO_H3K27ac Homo sapiens ChIP-Seq""","""SAMN07828425 vs SAMN07828426""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""SRA619794""","""ChIP""","""2018-08-04""","""1435487309;1430950991 vs 1165915270;1149296041""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR620/002/SRR6206782/SRR6206782_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR620/002/SRR6206782/SRR6206782_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR620/001/SRR6206781/…","""Generic""","""HS BRG1-KO_H3K27ac vs BRG1-KO_H3K27ac""","""79f32f4556bc64ff2f842f58e90b2ce4;d401a88e986e1da8b840efb2e94c37

&nbsp;

&nbsp;

&nbsp;

## 3. GSE209667 (ChIP-seq; H3K4me3)

- ✅ ChIP-Atlas: Diff Analysis

&nbsp;

- [Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding](https://doi.org/10.1038/s41467-023-43157-7)
- Super series: [GSE209687](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE209687)
- 下記のような記述があることから、42℃ 60 min であると考えられる
- 既に上記で、データフレームを定義済み
- `Results; HSF1 binding is a temperature-independent hallmark of HSR; We began by examining MCF7 breast adenocarcinoma cells during temperature-induced Heat Shock Response (HSR) (Fig. 1, Supplementary Data 1, Supplementary Fig. 1a). A 60-min incubation at 42 °C induced a control HSP gene seen as spreading of the Pol II and nascent RNA signal into the gene body (Fig. 1a).`

In [52]:
defined_pairs_gse209667_chipseq_histone_modification = [
    ("SRX16646109", "SRX16646108"), # mcf7_hs_h3k4me3_rep_1 vs. mcf7_nhs_h3k4me3_rep_1
    ("SRX16646092", "SRX16646110"), # mcf7_hs_h3k4me3_rep_2 vs. mcf7_nhs_h3k4me3_rep_2
]

paired_metada_df_gse209667_histone_modification = create_paired_metadata(
    combined_df=GSE209667_combined,
    defined_sample_pairs=defined_pairs_gse209667_chipseq_histone_modification,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse209667_histone_modification is not None:
    paired_metada_df_gse209667_histone_modification.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE209667_chipseq_histone_modification_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse209667_histone_modification)
else:
    print("No paired metadata to write (create_paired_metadata returned None).") 

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,cell_line,cell_type,chip_antibody,treatment,assembly,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX16646109""","""SRX16646108""","""mcf7_hs_h3k4me3_rep_1 vs mcf7_nhs_h3k4me3_rep_1""","""MCF7 HS H3K4me3 rep 1 vs MCF7 NHS H3K4me3 rep 1""","""GSM6383095 vs GSM6383094""","""Public on Oct 30 2023""","""Jul 25 2022""","""Oct 30 2023""","""SRA""","""1""","""MCF7""","""Homo sapiens""","""9606""","""genomic DNA""","""Raw files were adapter-trimmed with trimmomatic and aligned to hg19 genome using hisat2. For HSF1 ChIP-seq, raw fastq files were randomly subset to 40M read pairs prior to alignment. After removing PC…","""GPL15520""","""Sergei,,Nechaev""","""sergei.nechaev@und.edu""","""Nechaev Lab""","""Biomedical Sciences""","""University of North Dakota""","""Grand Forks""","""North Dakota""","""58202""","""USA""","""Illumina MiSeq""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM6383095 vs GSM6383094""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383095/suppl/GSM6383095_MCF7_HS_H3K4me3_ChIPseq_rep1.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383094/suppl/GSM6383094_MCF7_NHS_…","""GSE209667, GSE209687""","""0""","""MCF7""","""mammary gland, adenocarcinoma""","""H3K4me3 (Abcam #ab8580)""","""Heat shock vs Untreated""","""hg19""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN29938370 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN29938371""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX16646109 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX16646108""","""https://doi.org/10.1038/s41467-023-43157-7""","""42 C""","""60 min.""","""SRR20622353 vs SRR20622354""","""Illumina MiSeq sequencing: GSM6383095: MCF7 HS H3K4me3 rep 1 Homo sapiens ChIP-Seq vs Illumina MiSeq sequencing: GSM6383094: MCF7 NHS H3K4me3 rep 1 Homo sapiens ChIP-Seq""","""SAMN29938370 vs SAMN29938371""","""Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding""","""SRA1461984""","""ChIP""","""2023-11-01""","""280148615;280325693 vs 357657509;357815720""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR206/053/SRR20622353/SRR20622353_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR206/053/SRR20622353/SRR20622353_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR206/054/SRR2062…"

&nbsp;

&nbsp;

&nbsp;

## 4. GSE276302 (ChIP-seq; H3K27ac) (ChIP-Atlas: False)

- [HSF1–DBC1 axis drives prostate cancer progression by activating a metastatic transcriptional program](https://doi.org/10.1038/s12276-025-01545-7)
- `Detection of phosphorylated HSF1 and kinase assays; 22RV1 and SM1 cells were treated with HS at 42 °C or GSK-626616 (DYRK inhibitor, Tocris Bioscience) and lysed in FLAG lysis buffer24 containing phosphatase and protease inhibitor cocktails (Roche Diagnostics). `

In [53]:
defined_pairs_gse276302_chipseq_histone_modification = [
    # H3K27ac
    ("SRX25966378", "SRX25966375"), # k27ac_hs_22rv1 vs. k27ac_nhs_22rv1
    ("SRX25966380", "SRX25966377"), # k27ac_hs_22rv1_d1ko vs. k27ac_nhs_22rv1_d1ko
    ("SRX25966379", "SRX25966376")  # k27ac_hs_22rv1_h1ko vs. k27ac_nhs_22rv1_h1ko
]

paired_metada_df_gse276302_histone_modification = create_paired_metadata(
    combined_df=GSE276302_combined,
    defined_sample_pairs=defined_pairs_gse276302_chipseq_histone_modification,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_gse276302_histone_modification is not None:
    paired_metada_df_gse276302_histone_modification.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/GSE276302_chipseq_histone_modification_paired_metadata.tsv",
        separator='\t'
    )
    
    pl.Config.set_tbl_rows(2)
    display(paired_metada_df_gse276302_histone_modification)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,chip_antibody,cell_line,cell_type,treatment,basecalling,adapter_trimming,align,deduplication,peak_calling,assembly,supplementary_files_format_and_content,biosample,sra,reference_doi,stress treatment temperature,stress treatment time,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX25966378""","""SRX25966375""","""k27ac_hs_22rv1 vs k27ac_nhs_22rv1""","""K27ac_HS_22RV1 vs K27ac_NHS_22RV1""","""GSM8496181 vs GSM8496178""","""Public on Sep 02 2025""","""Sep 04 2024""","""Sep 02 2025""","""SRA""","""1""","""22RV1""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL24676""","""Jeong,Hoon,Kim""","""Sungkyunkwan University""","""Irwon-dong Gangnam-gu""","""Seoul""","""135-710""","""South Korea""","""Illumina NovaSeq 6000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM8496181 vs GSM8496178""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8496nnn/GSM8496181/suppl/GSM8496181_K27ac_HS_22RV1.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8496nnn/GSM8496178/suppl/GSM8496178_K27ac_NHS_22RV1.bw""","""GSE276302""","""0""","""H3K27ac (Abcam, ab4729)""","""22RV1""","""Castration-resistant prostate cancer cells""","""Heat shock at 42℃ for 60 min (HS) vs Non-heat shock (NHS)""","""bcl2fastq v2.20.0""","""Trimmomatic v0.38""","""Bowtie 1.1.2""","""Picard 1.118 MarkDuplicates""","""MACS2 2.1.1.20160309""","""hg19""","""bigwig""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN43495073 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN43495076""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX25966378 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX25966375""","""https://doi.org/10.1038/s12276-025-01545-7""","""42 C""","""60 min.""","""SRR30542968 vs SRR30542971""","""Illumina NovaSeq 6000 sequencing: GSM8496181: K27ac_HS_22RV1 Homo sapiens ChIP-Seq vs Illumina NovaSeq 6000 sequencing: GSM8496178: K27ac_NHS_22RV1 Homo sapiens ChIP-Seq""","""SAMN43495073 vs SAMN43495076""","""DBC1 promotes metastatic castration-resistant prostate cancer by enhancing the stability and activity of HSF1 [ChIP-seq]""","""SRA1962544""","""ChIP""","""2025-09-03""","""2580967841;2638989142 vs 2871919145;2921090141""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/068/SRR30542968/SRR30542968_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/068/SRR30542968/SRR30542968_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR305/071/SRR3054…","""missing""","""Generic""","""K27ac_HS_22RV1 vs K27ac_NHS_22RV1""","""579f0808893ca2fe9fb061

&nbsp;

&nbsp;

&nbsp;

## Strategy 2: Double Check using [GEOfinder](https://bioapps.byu.edu/geofinder/) 

- [GEOfinder](https://bioapps.byu.edu/geofinder/) で、収集の漏れがないか簡易的に調査
- 収集されていないものはないと判断

&nbsp;

&nbsp;

## Strategy 3: `BioProject` で調査

- 以下のクエリで `BioProject`のページを調査 (2026−05−03)
- 1つだけ発見

```text
(((((((((("heat stress") OR "heat shock") OR "high temperature stress") OR "hyperthermia") OR "heat stroke")) AND "homo sapiens"[Organism]) AND "bioproject sra"[Filter]) AND "epigenomics"[Project Data Type])) NOT "bioproject gds"[Filter]
```

&nbsp;

&nbsp;

## 1. PRJNA369311 (SRP098615) (ChIP-seq; HIRA) (ChIP-Atlas: False)

- [Histone H3.3 chaperone HIRA renders stress-responsive genes poised for prospective lethal stresses in acquired tolerance](https://doi.org/10.1111/gtc.13140)
- `4.2 Heat treatment; Cells were seeded at a density of 1.1 × 104 or 1.7 × 104 cells/cm2 in 35 mm dishes for immortalized cells and primary cells, respectively, and incubated for 24 h at 37°C before heat shock treatment. For priming stress treatment, culture media was replaced with fresh media pre-heated at 40°C, and cells were cultured in an incubator at 40°C (±0.5°C) and 5% CO2 for 2 h. For lethal stress treatment, culture media was replaced with fresh media pre-heated at 47°C, and cells were cultured in an incubator at 47°C (±0.5°C) and 5% CO2 for 35 min. After heat treatment, the media was replaced by fresh media (37°C), and the cells were incubated in an incubator at 37°C (±0.5°C) and 5% CO2 for 48 h. Cell viability was quantified using a Cell Counting Kit-8 (Dojindo) and SpectraMax 340 Microplate Reader (Molecular Devices).`

In [54]:
PRJNA369311 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/PRJNA369311.metadata.tsv",
    separator='\t'
).filter(
    (pl.col("library_strategy") == "ChIP-Seq") &
    (pl.col("tax_id") == 9606)
).with_columns(
    pl.lit("https://doi.org/10.1111/gtc.13140").alias("reference_doi"),
    pl.lit("40 C").alias("stress treatment temperature"),
    pl.lit("120 min.").alias("stress treatment time")
)

PRJNA369311_filtered = PRJNA369311[[s.name for s in PRJNA369311 if not (s.null_count() == PRJNA369311.height)]]


# pl.Config.set_tbl_rows(6)
# display(PRJNA369311_filtered)

In [55]:
defined_pairs_prjna369311_chipseq = [
    ("SRX2558059", "SRX2529349"), # 2015_007A_40C_HIRA vs. 2015_007A_37C_HIRA
    ("SRX2558060", "SRX2558057"), # 2015_007A_40C_IgG vs. 2015_007A_37C_IgG
    ("SRX2558061", "SRX2558058")  # 2015_007A_40C_input vs. 2015_007A_37C_input
]

paired_metada_df_prjna369311_chipseq = create_paired_metadata(
    combined_df=PRJNA369311_filtered,
    defined_sample_pairs=defined_pairs_prjna369311_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_prjna369311_chipseq is not None:
    paired_metada_df_prjna369311_chipseq.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/PRJNA369311_chipseq_paired_metadata.tsv",
        separator='\t'
    )
    
    display(paired_metada_df_prjna369311_chipseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,sex,fastq_galaxy,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title,instrument_model,description,cell_type,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,experiment_alias,center_name,cell_line,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,tissue_type,reference_doi,stress treatment temperature,stress treatment time
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX2558059""","""SRX2529349""","""SRR5252359 vs SRR5252355""","""Illumina HiSeq 2500 sequencing: priming_HIRA_ChIP vs Illumina HiSeq 2500 sequencing: no_stress_HIRA_ChIP""","""SAMN06284127""","""Homo sapiens strain:WI38""","""SRA533844""","""unspecified""","""2018-02-01""","""1852889708 vs 1984908520""","""ILLUMINA""","""female""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR525/009/SRR5252359/SRR5252359.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR525/005/SRR5252355/SRR5252355.fastq.gz""","""Human""","""WI-38_Human sample from Homo sapiens""","""4f326dd97b4d36c4d9fb2c0302d3b193 vs f585d7c559c5501430fb16dbb6275764""","""SRP098615""","""40817199 vs 43851261""","""HIRA_WI-38_ChIP-seq""","""GENERATED_FILE""","""WI-38_Human sample from Homo sapiens""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: priming_HIRA_ChIP vs Illumina HiSeq 2500 sequencing: no_stress_HIRA_ChIP""","""fibroblast""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR525/009/SRR5252359/SRR5252359.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR525/005/SRR5252355/SRR5252355.fastq.gz""","""2693935134 vs 2894183226""","""ChIP-Seq""","""GENOMIC""","""SINGLE""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/SRR525/009/SRR5252359/SRR5252359.fastq.gz vs fasp.sra.ebi.ac.uk:/vol1/fastq/SRR525/005/SRR5252355/SRR5252355.fastq.gz""","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;7776;117570;117571;8287;1338369;32523;32524;40674;32525;9347;1437010;314146;9443;376913;314293;9526;314295;9604;207598;9605;9606""","""WI-38_Human_ caucasian fibroblast-like fetal lung cell""","""2015_007A_40C_HIRA vs 2015_007A_37C_HIRA""","""priming_HIRA_ChIP vs no_stress_HIRA_ChIP""","""The University of Tokyo, IMCB""","""WI-38""","""env_tax:terrestrial;env_tax""","""2018-02-01""","""Homo sapiens""","""9606""","""PRJNA369311""","""SRR5252359 vs SRR5252355""","""SRS1950313""","""2018-02-01""","""PRJNA369311""","""lung""","""https://doi.org/10.1111/gtc.13140""","""40 C""","""120 min."""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX2558061""","""SRX2558058""","""SRR5252361 vs SRR5252358""","""Illumina HiSeq 2500 sequencing: priming_input vs Illumina HiSeq 2500 sequencing: no_stress_input""","""SAMN06284127""","""Homo sapiens strain:WI38""","""SRA533844""","""unspecified""","""2018-02-01""","""2014788482 vs 2035598415""","""ILLUMINA""","""female""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR525/001/SRR5252361/SRR5252361.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR525/008/SRR5252358/SRR5252358.fastq.gz""","""Human""","""WI-38_Human sample from Homo sapiens""","""43f92b741a5f55ddd42928afc5cabd0d vs c94c4d255e899742b9fd3cb62f556b74""","""SRP098615""","""44256247 vs 44878364""","""HIRA_WI-38_ChIP-seq""","""GENERATED_FILE""","""WI-38_Human sample from Homo sapiens""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: priming_input vs Illumina HiSeq 2500 sequencing: no_stress_input""","""fibroblast""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR525/001/SRR5252361/SRR5252361.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR525/008/SRR5252358/SRR525235

&nbsp;

&nbsp;

&nbsp;

## Strategy 4: `BioProject` で調査 (2)

- 以下のクエリで `BioProject`のページを調査 (2026−05−03)
- `other"[Project Data Type]` を指定して調査

```text
((((((("heat stress"[All Fields] OR "heat shock"[All Fields]) OR "high temperature stress"[All Fields]) OR "hyperthermia"[All Fields]) OR "heat stroke"[All Fields]) AND "homo sapiens"[Organism]) AND "bioproject sra"[Filter]) AND "other"[Project Data Type]) NOT "bioproject gds"[Filter]
```

&nbsp;

## 1. PRJEB5930 (ChIP-seq; SUMO2)

- [ENA browser: PRJEB5930](https://www.ebi.ac.uk/ena/browser/view/PRJEB5930)
- [Proteotoxic stress reprograms the chromatin landscape of SUMO modification](https://doi.org/10.1126/scisignal.aaa2213)
- `Results;ChIP-seq reveals substantial increases in the binding of SUMO-2 to active DNA-regulatory elements in response to HS; To characterize changes in the association of SUMO-2 with chromatin in response to HS in an unbiased manner, we used chromatin immunoprecipitation (ChIP) coupled to high-throughput sequencing (ChIP-seq) to determine the genome-wide profiles of SUMO-2 in U2OS (human osteosarcoma) cells that were either unstressed or were subjected to HS by shifting the incubation temperature of the cells to 43°C for 30 min (HS cells),`

In [56]:
PRJEB5930 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/PRJEB5930.metadata.tsv",
    separator='\t'
).filter(
    (pl.col("library_strategy") == "ChIP-Seq") &
    (pl.col("tax_id") == 9606)
).with_columns(
    pl.lit("https://doi.org/10.1126/scisignal.aaa2213").alias("reference_doi"),
    pl.lit("43 C").alias("stress treatment temperature"),
    pl.lit("30 min.").alias("stress treatment time")
)

PRJEB5930_filtered = PRJEB5930[[s.name for s in PRJEB5930 if not (s.null_count() == PRJEB5930.height)]]

# display(PRJEB5930_filtered)

In [57]:
defined_pairs_prjeb5930_chipseq = [
    ("ERX432314", "ERX432310"), # heat_shock_SUMO2_ChIP_protein G Dynabeads beads vs. untreated_SUMO2_ChIP_protein G Dynabeads beads
    ("ERX432312", "ERX432309"), # heat_shock_SUMO2_ChIP_protein G sepharose 4B beads vs. untreated_SUMO2_ChIP_protein G sepharose 4B beads
    ("ERX432311", "ERX432313")  # heat_shock_input vs. untreated_input
]

paired_metada_df_prjeb5930_chipseq = create_paired_metadata(
    combined_df=PRJEB5930_filtered,
    defined_sample_pairs=defined_pairs_prjeb5930_chipseq,
    sample_id_column_name="experiment_accession"
)

if paired_metada_df_prjeb5930_chipseq is not None:
    paired_metada_df_prjeb5930_chipseq.write_csv(
        "../../Data/metadata/metadata_chipseq/HS_related_paired_metadata/PRJEB5930_chipseq_paired_metadata.tsv",
        separator='\t'
    )
    
    display(paired_metada_df_prjeb5930_chipseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,submitted_galaxy,submitted_ftp,fastq_bytes,instrument_platform,submitted_aspera,sex,submitted_md5,checklist,fastq_galaxy,submitted_bytes,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title,instrument_model,description,cell_type,fastq_ftp,broker_name,base_count,library_strategy,library_source,library_layout,datahub,status,fastq_aspera,tax_lineage,sample_alias,run_alias,submitted_file_role,submitted_read_type,library_construction_protocol,experiment_alias,center_name,cell_line,library_name,tag,first_created,scientific_name,tax_id,study_accession,submitted_format,accession,secondary_sample_accession,first_public,study_alias,reference_doi,stress treatment temperature,stress treatment time
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERX432314""","""ERX432310""","""ERR466073 vs ERR466078""","""Illumina Genome Analyzer IIx sequencing: heat shock SUMO-2 ChIP-seq""","""SAMEA2420694 vs SAMEA2420690""","""heat shock SUMO-2 ChIP-seq""","""ERA294640""","""ChIP""","""2018-11-16""","""ftp.sra.ebi.ac.uk/vol1/run/ERR466/ERR466073/HSChip_2.fq.gz vs ftp.sra.ebi.ac.uk/vol1/run/ERR466/ERR466078/UTChip_2.fq.gz""","""ftp.sra.ebi.ac.uk/vol1/run/ERR466/ERR466073/HSChip_2.fq.gz vs ftp.sra.ebi.ac.uk/vol1/run/ERR466/ERR466078/UTChip_2.fq.gz""","""867924044 vs 1050208663""","""ILLUMINA""","""fasp.sra.ebi.ac.uk:/vol1/run/ERR466/ERR466073/HSChip_2.fq.gz vs fasp.sra.ebi.ac.uk:/vol1/run/ERR466/ERR466078/UTChip_2.fq.gz""","""female""","""93e7aeb8b88d3935988d0987485114e8 vs e0c1d906a9aa237dca14b0305820d0fa""","""ERC000011""","""ftp.sra.ebi.ac.uk/vol1/fastq/ERR466/ERR466073/ERR466073.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/ERR466/ERR466078/ERR466078.fastq.gz""","""897109036 vs 1080946836""","""Protocols: Human osteosarcoma (U2OS) cells were obtained from CRUK and grown at 37C according to standard cell culture protocols. U2OS cells were grown at 37C and either left untreated or incubated at…","""9fcfa22fbb334416aa3bf247105b07d5 vs 72e270d00543166328c3196aeb2f875e""","""ERP005368""","""23386224 vs 28459857""","""heat shock SUMO-2 ChIP-seq""","""GENERATED_FILE""","""growing_U2OS_cells_6 vs growing_U2OS_cells_5""","""Illumina Genome Analyzer IIx""","""Illumina Genome Analyzer IIx sequencing: heat shock SUMO-2 ChIP-seq""","""human osteosarcoma cell""","""ftp.sra.ebi.ac.uk/vol1/fastq/ERR466/ERR466073/ERR466073.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/ERR466/ERR466078/ERR466078.fastq.gz""","""ArrayExpress""","""841904064 vs 1024554852""","""ChIP-Seq""","""GENOMIC""","""SINGLE""","""dcc_arrayexpress""","""public""","""fasp.sra.ebi.ac.uk:/vol1/fastq/ERR466/ERR466073/ERR466073.fastq.gz vs fasp.sra.ebi.ac.uk:/vol1/fastq/ERR466/ERR466078/ERR466078.fastq.gz""","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;7776;117570;117571;8287;1338369;32523;32524;40674;32525;9347;1437010;314146;9443;376913;314293;9526;314295;9604;207598;9605;9606""","""E-MTAB-2368:growing_U2OS_cells_6 vs E-MTAB-2368:growing_U2OS_cells_5""","""E-MTAB-2368:HSChip_2.fq.gz vs E-MTAB-2368:UTChip_2.fq.gz""","""SUBMISSION_FILE""","""SINGLE""","""Human osteosarcoma (U2OS) cells were obtained from CRUK and grown at 37C according to standard cell culture protocols. U2OS cells were grown at 37C and either left untreated or incubated at 43C for 30…","""E-MTAB-2368:SO_1308_1308_43-2 vs E-MTAB-2368:SO_1308_1308_37-2""","""European Bioinformatics Institute;Centre for Gene Regulation and Expression;University of Dundee;Dundee DD1 5EH;United Kingdom""","""U2OS""","""heat_shock_SUMO2_ChIP_protein G Dynabeads beads vs untreated_SUMO2_ChIP_protein G Dynabeads beads""","""env_tax:terrestrial;env_tax""","""2014-03

&nbsp;

&nbsp;

## Concat Dataframe

- ここにある全てのデータフレームを `concat` する

In [58]:
df_chipseq_all: list[pl.DataFrame | None] = [
    paired_metadata_df_gse140053_chipseq,
    paired_metada_df_gse145525,
    paired_metada_df_gse38901,
    paired_metada_df_gse43579,
    paired_metada_df_gse62309,
    paired_metada_df_gse57398,
    paired_metada_df_gse66448,
    paired_metada_df_gse60984,
    paired_metada_df_gse105028,
    paired_metada_df_gse112379,
    paired_metada_df_gse121182,
    paired_metada_df_gse137558,
    paired_metada_df_gse152144,
    paired_metada_df_gse143591,
    paired_metada_df_gse194364,
    paired_metada_df_gse166456,
    paired_metada_df_gse209667,
    paired_metada_df_gse270671,
    paired_metada_df_gse253873,
    paired_metada_df_gse259305,
    paired_metada_df_gse276302,
    paired_metada_df_gse303857,
    paired_metada_df_prjna369311_chipseq,
    paired_metada_df_prjeb5930_chipseq
]

dfs_chipseq:  list[pl.DataFrame] = [df for df in df_chipseq_all if df is not None]

In [59]:
chipseq_concat = pl.concat(dfs_chipseq, how="diagonal")

chipseq_concat.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/chipseq_metadata_all_{execute_date}.tsv",
    separator='\t'
)

In [60]:
dfs_chipseq_histone_modification_all: list[pl.DataFrame | None] = [
    paired_metada_df_gse89382_histone_modification,
    paired_metada_df_gse105028_histone_modification,
    paired_metada_df_gse209667_histone_modification,
    paired_metada_df_gse276302_histone_modification
]

dfs_chipseq_histone_modification: list[pl.DataFrame] = [df for df in dfs_chipseq_histone_modification_all if df is not None]

chipseq_histone_modification_concat = pl.concat(dfs_chipseq_histone_modification, how="diagonal")

chipseq_histone_modification_concat.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/chipseq_histone_modification_metadata_all_{execute_date}.tsv",
    separator='\t'
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### ChIP-seq ではないが、熱ストレス条件下のデータを活用している エピゲノムデータ

- 使用できるかわからないものの、念のためこちらに記載する

&nbsp;

## GSE192370 (CUT&TAG)

In [61]:
# GSE192370_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE192370/GSE192370_PEP/GSE192370_PEP_raw.csv",
#     separator=','
# )

# display(GSE192370_PEP)

&nbsp;

&nbsp;

## 2. GSE326102 (CUT&Run) 

In [62]:
# GSE326102_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE326102/GSE326102_PEP/GSE326102_PEP_raw.csv",
#     separator=','
# )

# display(GSE326102_PEP)